> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# Agent factual-conflict · v5.1 tool-sequence repair

** Full standalone version: in a new version  Colab notebook/runtime  runs from top to bottom. All codes, 192  Official missions and  6  Software / Capability check tasks are embedded. **
 No need for old.  notebook、 Old variable or  `.py`  Uploads. You can reuse the loaded model that is compatible with the current session;old  v4  The answer will not be imported.  prompt。

 Preserve  **Llama-3.1-8B BASE**、 fixed  commit `d04e592bb4f6aa9cfee91e2e20afa771667e1d4b`、bf16、
TransformerLens 2.17.0  and actual implementation  `check_stock → reserve_item → create_order`。 Do not automatically quantify or switch models. 

 This edition is fixed.  v5 smoke  Its straight down. Its made up.  reservation  the process of presenting the full range of three independent teaching tasks 10 valid next-step examples covering stock checks, out-of-stock fallback, reservation, and complete ordering. 
 Models are still free to generate complete.  JSON； The program will not supplement the city, add fields or ease the tool validation. ** Whether or not actual improvements are to be made below  GPU smoke  Authentication. **

##  Data and Tasks 

|  project  |  Frozen contents  |
|---|---|
|  Formally independent missions  | 192 = 3  attribute  × 4  template  × 2  Inventory status  × 8  repeating;each attribute 64 |
|  attribute  |  Delivery city, ordering account, and preferred item all affect actual tool outcomes  |
|  Example  |  Fixed New  seed 83022；12  Cities, 3  Properties, 4  Templates to follow  v4  Design, source ID/ Parameter Regeneration  |
| Context | aligned/conflict； One at a time.  verified_profile  Add one.  retrieved_note， only  note.value  Change  |
|  order  | profile_first / note_first， Press the same source paired  |
| Baseline | untreated / authority_first / authority_only |
| Writer |  From each  arm  Actual information and  Actor  Tracks generate complete memory and check values and sources  |

 New instances were frozen before generation and use the same previously studied ordering-task design; they are not unseen-template tests; 
 This experiment is not directly equivalent to the outside world.  agent benchmark。baseline  If the source label is reliable, filtering reduces input information. 

##  Execution order 

1.  Select Support  bf16   CUDA GPU（ Original experimental use  A100）。 Run installation checks; if dependencies were installed, restart once as instructed and rerun from the beginning. 
2.  Mount  Drive， Perform definition and data checks, preview fixed tasks; then login  Hugging Face  and loads the model. Account numbers require model access. 
3.  Run interface and length check, 6  Independent.  smoke  Tasks; executability checks followed by formal assignments and export. 

 The right environment will skip.  pip。 Compatible loaded models preferred to reuse, otherwise they would be used  Hugging Face  The general weight of the power cache. 
v5  Use new  Drive  Directory. Old.  v4  Data not modified, old generation is not used in new experiments; same  v5  Directory supports automatic renewal. 

 A full formal assessment.  2304  Logical cross-reference of line, to resize after  **960   Actor episode  and corresponding  Writer**。
 each  Actor episode  Include multiple model generation. Default for each batch 24 source， Repeat formal evaluation blocks until 192/192； Replace with None All the remaining items can be completed. 
 Every generation and every complete  source  save checkpoints and resume under the same protocol. Changes to code, configuration, or data require a new run directory. 

 Add Error Reservation, First / Any error in the listing, city Statistics such as missing; successful booking requires a real tool to be observed successfully. 
 Conflict SKU Booking is only right. preferred_item Yes, full assessment denominator is 64； The remaining non-applicable tasks are not added as zero errors. 
 Accepted incorrect final orders remain the primary metric; intermediate actions are secondary. Retries do not increase the independent sample size. 

 Three conflict-free control groups are routinely inspected before formal assessment: each 6 smoke Zero protocol error, first order. city、 At least. 5 One mission was successful. 
 Failed saves / Export smoke trajectories and stop; checks require no conflict errors and do not select formal tasks. 

** This file does not include new  GPU  Experimental results. **  Local validation uses actual tools with mocked generation to test implementation and resume rules. 


## 01 ·  Install Inspection 
 A matching environment passes directly. First-time installation stops execution; restart  runtime  Runs from here again to skip installation automatically. 


In [ ]:
import sys, subprocess, importlib.metadata as v5_metadata
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(' Fixed-dependent combination needs  Python 3.10–3.13。')
V5_REQUIRED_VERSIONS = {
    'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0',
    'numpy': '2.1.3' if sys.version_info[:2] == (3, 13) else '1.26.4',
    'pandas': '2.2.3', 'plotly': '6.1.2', 'scipy': '1.15.3',
    'scikit-learn': '1.6.1', 'datasets': '3.6.0', 'pyarrow': '20.0.0',
}
V5_INSTALL_PROBLEMS = []
for v5_name, v5_version in V5_REQUIRED_VERSIONS.items():
    try:
        v5_installed = v5_metadata.version(v5_name)
    except v5_metadata.PackageNotFoundError:
        v5_installed = None
    if v5_installed != v5_version:
        V5_INSTALL_PROBLEMS.append((v5_name, v5_installed, v5_version))
if V5_INSTALL_PROBLEMS:
    print(' The original experimental fixed version will be installed: ', V5_INSTALL_PROBLEMS)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    *[f'{name}=={version}' for name, version in V5_REQUIRED_VERSIONS.items()], 'tqdm'], check=True)
    raise SystemExit(' Installation complete. Please restart  Colab runtime， Runs from this block; versions are not repeated after matching. ')
print(' Fixed reliance matches, skips installation. ')


## 02 ·  Dependence, Drive  With Run Configuration 
 Keep default settings on the first run. Reuse existing same-name results only with exactly the same code / Configure recovery. 


In [ ]:
#  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
import sys
import importlib.metadata as metadata
import importlib.util

print("Python:", sys.version.split()[0])
print(" Interpreter :", sys.executable)
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

expected_versions = {
    "transformer-lens": "2.17.0", "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
    "pandas": "2.2.3",
    "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
    "datasets": "3.6.0", "pyarrow": "20.0.0",
}
problems = []
for package, expected in expected_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
    if installed != expected:
        problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

for package in ("torch", "tqdm"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        problems.append(f"{package}:  not installed ")

try:
    torch_version = metadata.version("torch")
    torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
    if torch_major_minor < (2, 6):
        problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
except metadata.PackageNotFoundError:
    pass  #  The record is missing. 
except ValueError:
    problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

if importlib.util.find_spec("transformer_lens") is None:
    problems.append(" current  Python  Cannot find  transformer_lens  module ")
if problems:
    raise RuntimeError(
        " Reliance check failed: \n- " + "\n- ".join(problems)
        + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
          " Go on from this block. "
    )
print(" Reliance on a version check pass; continues to run the third code block. ")


In [ ]:
#  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
import gc, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, utils
from huggingface_hub import notebook_login, get_token, model_info
import importlib.metadata as metadata
from IPython.display import display

V5_PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
V5_PACKAGE_VERSIONS = {name: metadata.version(name) for name in V5_PACKAGE_NAMES}
print(V5_PACKAGE_VERSIONS)
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print(" Reliance on import success. ")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
V5_EXPERIMENT_ROOT = Path('/content/drive/MyDrive/AAAI_Workshop/agent_entrainment')
V5_RUN_NAME = 'agent_entrainment_conflict_v5p1_protocol_actions_full'
V5_RUN_ROOT = V5_EXPERIMENT_ROOT / V5_RUN_NAME
V5_RUN_ROOT.mkdir(parents=True, exist_ok=True)
V5_BATCH_SOURCES = 24  # None =  All remaining; 24 =  One balanced batch each time, repeats the formal evaluation cell resume 
V5_REUSE_LOADED_MODEL = True
V5_REUSE_PREVIOUS_RESULTS = False
V5_PRIOR_RESULT_DIRS = []
V5_LIVE_ONLY = False
V5_CFG = {
    'experiment_version': 'conflict_v5p1_protocol_actions_full_v1',
    'model_name': 'meta-llama/Llama-3.1-8B',
    'model_revision': 'd04e592bb4f6aa9cfee91e2e20afa771667e1d4b',
    'dtype': 'bfloat16', 'context_window': 4096, 'max_prompt_tokens': 3072,
    'hook_kind': 'z', 'use_cache': True, 'actor_max_new_tokens': 128,
    'writer_max_new_tokens': 384, 'max_actions': 6, 'data_seed': 83022,
}
print(' New version of full result directory: ', V5_RUN_ROOT)


## 03 ·  Original model executor and real tool 
 Embedded original generator, strict  JSON  Parser and tool definition, independent initialization complete  runtime； Definitions only; model weights are not downloaded. 


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
import linecache
V5_CORE_SOURCE = r'''import copy, gc, hashlib, json, math, os, random, re, time
from pathlib import Path


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result

def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks

class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}

def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result

def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None

class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}

def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
'''
V5_RUNTIME_SOURCE = r'''"""Independent, pinned v4 runtime with checked live-model reuse.

No function in this module loads weights until v4_load_runtime is called.
The original backend/tools are supplied verbatim in the same isolated namespace.
"""
import ast as v4_ast
import copy as v4_copy
import hashlib as v4_hashlib
import importlib.metadata as v4_metadata
import json as v4_json
from pathlib import Path as V4Path
import sys as v4_sys
import types as v4_types


V4_MODEL_NAME = "meta-llama/Llama-3.1-8B"
V4_MODEL_REVISION = "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
V4_PACKAGE_NAMES = (
    "torch", "transformer-lens", "transformers", "huggingface-hub", "numpy",
    "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly",
)
V4_FIXED_CONFIG = {
    "model_name": V4_MODEL_NAME, "model_revision": V4_MODEL_REVISION,
    "dtype": "bfloat16", "context_window": 4096, "max_prompt_tokens": 3072,
    "actor_max_new_tokens": 128, "writer_max_new_tokens": 384,
    "max_actions": 6, "use_cache": True, "hook_kind": "z",
}


def v4_backend_contract(cfg):
    """Only settings that can change the original generation implementation."""
    keys = ("context_window", "max_prompt_tokens", "actor_max_new_tokens",
            "writer_max_new_tokens", "use_cache", "hook_kind", "dtype", "model_name")
    if not isinstance(cfg, dict) or any(key not in cfg for key in keys):
        raise ValueError("Incomplete backend generation contract")
    return {key: cfg[key] for key in keys}


def v4_code_state(code):
    """Executable-code identity independent of notebook filename/line numbers."""
    def v4_constant(value):
        if isinstance(value, v4_types.CodeType):
            return {"code": v4_code_state(value)}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, tuple):
            return {"tuple": [v4_constant(item) for item in value]}
        if isinstance(value, frozenset):
            return {"frozenset": sorted((v4_constant(item) for item in value), key=repr)}
        if value is None or isinstance(value, (bool, int, float, str)):
            return value
        return {"type": type(value).__name__, "repr": repr(value)}
    return {
        "bytecode": code.co_code.hex(), "constants": [v4_constant(x) for x in code.co_consts],
        "names": code.co_names, "variables": code.co_varnames,
        "free": code.co_freevars, "cells": code.co_cellvars,
        "args": code.co_argcount, "posonly": code.co_posonlyargcount,
        "kwonly": code.co_kwonlyargcount, "flags": code.co_flags & 0xffff,
        "exception_table": getattr(code, "co_exceptiontable", b"").hex(),
    }


def v4_definition_state(value):
    if isinstance(value, type):
        methods = {}
        for name, member in vars(value).items():
            member = member.__func__ if isinstance(member, (staticmethod, classmethod)) else member
            if isinstance(member, v4_types.FunctionType):
                methods[name] = v4_definition_state(member)
        return {"class": value.__name__, "bases": [base.__name__ for base in value.__bases__],
                "methods": methods}
    if isinstance(value, v4_types.MethodType):
        value = value.__func__
    if isinstance(value, v4_types.FunctionType):
        return {"code": v4_code_state(value.__code__),
                "defaults": v4_copy.deepcopy(value.__defaults__),
                "kwdefaults": v4_copy.deepcopy(value.__kwdefaults__)}
    return v4_copy.deepcopy(value)


def v4_source_exports(source):
    names = []
    for node in v4_ast.parse(source).body:
        if isinstance(node, (v4_ast.FunctionDef, v4_ast.ClassDef)):
            names.append(node.name)
        elif isinstance(node, v4_ast.Assign):
            names.extend(target.id for target in node.targets if isinstance(target, v4_ast.Name))
    return names


def v4_check_sources(core_source, runtime_source):
    """Check live definitions against the bundled sources, without inference."""
    if not isinstance(core_source, str) or not core_source.strip() or not isinstance(runtime_source, str) or not runtime_source.strip():
        raise ValueError("Both exact runtime source strings are required")
    expected = dict(globals())
    names = []
    for label, source in (("core", core_source), ("runtime", runtime_source)):
        exec(compile(source, "<v4-expected-" + label + ">", "exec", dont_inherit=True), expected)
        names.extend(v4_source_exports(source))
    if len(names) != len(set(names)):
        raise RuntimeError("Bundled core/runtime definitions overlap")
    states = {name: v4_definition_state(expected[name]) for name in names}
    for name in names:
        if name not in globals() or stable_hash(v4_definition_state(globals()[name])) != stable_hash(states[name]):
            raise RuntimeError("Live definition does not match bundled source: " + name)
    return states


def v4_check_packages(package_versions):
    if not isinstance(package_versions, dict) or set(package_versions) != set(V4_PACKAGE_NAMES):
        raise ValueError("Use the original complete PACKAGE_VERSIONS schema")
    actual = {name: v4_metadata.version(name) for name in V4_PACKAGE_NAMES}
    if actual != package_versions or str(torch.__version__) != actual["torch"]:
        raise RuntimeError("Installed/imported packages differ from the supplied package identity")
    for name, required in (("transformer-lens", "2.17.0"), ("transformers", "4.57.1"),
                           ("huggingface-hub", "0.36.0")):
        if actual[name] != required:
            raise RuntimeError("Original model stack required: " + name + "==" + required)
    return actual


def v4_plain_state(value):
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, dict):
        return {str(key): v4_plain_state(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [v4_plain_state(item) for item in value]
    return {"type": type(value).__module__ + "." + type(value).__qualname__, "repr": str(value)}


def v4_tokenizer_state(tokenizer):
    fields = ("name_or_path", "vocab_size", "model_max_length", "padding_side", "truncation_side",
              "bos_token_id", "eos_token_id", "pad_token_id", "unk_token_id", "all_special_ids",
              "special_tokens_map", "added_tokens_encoder", "add_bos_token", "add_eos_token",
              "clean_up_tokenization_spaces")
    return {"class": type(tokenizer).__module__ + "." + type(tokenizer).__qualname__,
            **{key: v4_plain_state(getattr(tokenizer, key, None)) for key in fields}}


def v4_no_active_hooks(model):
    hook_points = getattr(model, "hook_dict", None)
    if not isinstance(hook_points, dict):
        raise RuntimeError("Model lacks the original TransformerLens hook registry")
    for name, point in hook_points.items():
        if getattr(point, "fwd_hooks", ()) or getattr(point, "bwd_hooks", ()):
            raise RuntimeError("Active TransformerLens hooks at " + str(name))
    if not callable(getattr(model, "modules", None)):
        raise RuntimeError("Model does not expose its module hook state")
    for module in model.modules():
        for field in ("_forward_hooks", "_forward_pre_hooks", "_backward_hooks", "_backward_pre_hooks"):
            if getattr(module, field, None):
                raise RuntimeError("Active module hooks: " + field)


def v4_parameter_state(model):
    result = []
    for parameter in model.parameters():
        result.append({"object_id": id(parameter), "version": getattr(parameter, "_version", None),
                       "shape": list(getattr(parameter, "shape", ())),
                       "dtype": str(parameter.dtype), "device": str(parameter.device),
                       "requires_grad": bool(parameter.requires_grad),
                       "data_ptr": int(parameter.data_ptr()) if callable(getattr(parameter, "data_ptr", None)) else None})
    if not result:
        raise RuntimeError("Loaded model has no parameters")
    return result


def v4_check_model(model, cfg):
    if isinstance(HookedTransformer, type) and type(model) is not HookedTransformer:
        raise RuntimeError("The loaded model is not the original HookedTransformer class")
    v4_no_active_hooks(model)
    if bool(getattr(model, "training", True)):
        raise RuntimeError("Model must already be in evaluation mode")
    if any(bool(getattr(model, key, False)) for key in ("is_loaded_in_4bit", "is_loaded_in_8bit")):
        raise RuntimeError("Quantized models cannot be reused")
    if int(model.cfg.n_ctx) != cfg["context_window"]:
        raise RuntimeError("Model context differs from the frozen backend contract")
    for field, expected in (("n_layers", 32), ("n_heads", 32), ("d_model", 4096), ("d_head", 128)):
        if getattr(model.cfg, field, None) != expected:
            raise RuntimeError("Model architecture differs at " + field)
    if getattr(model.cfg, "use_attn_result", False):
        raise RuntimeError("Model attention-result setting differs from the original z backend")
    parameters = v4_parameter_state(model)
    if any(item["dtype"] != str(torch.bfloat16) or item["requires_grad"]
           or item["device"].split(":")[0] != "cuda" for item in parameters):
        raise RuntimeError("Model parameters must all be frozen, CUDA, and bfloat16")
    if len({item["device"] for item in parameters}) != 1:
        raise RuntimeError("The original single-device model contract is required")
    tokenizer = model.tokenizer
    if not callable(getattr(tokenizer, "encode", None)) or not callable(getattr(tokenizer, "decode", None)):
        raise RuntimeError("Tokenizer encode/decode interface is missing")
    if tokenizer.eos_token_id is None or tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer eos/padding setup is incomplete")
    module_code, modules = {}, []
    for module in model.modules():
        forward = getattr(module, "forward", None)
        function = forward.__func__ if isinstance(forward, v4_types.MethodType) else forward
        function_id = id(function)
        if function_id not in module_code:
            module_code[function_id] = (stable_hash(v4_code_state(function.__code__))
                                       if hasattr(function, "__code__") else None)
        modules.append({"object_id": id(module), "class": type(module).__module__ + "." + type(module).__qualname__,
                        "forward_object_id": function_id, "forward_code": module_code[function_id]})
    return {"parameters": parameters, "config": v4_plain_state(vars(model.cfg)),
            "tokenizer": v4_tokenizer_state(tokenizer), "modules": modules}


def v4_check_backend(backend, cfg, model_identity, expected_identity):
    if stable_hash(model_identity) != stable_hash(expected_identity):
        raise RuntimeError("Candidate model/revision/dtype/packages/Python identity differs")
    if type(backend).__name__ != "TLBackend" or stable_hash(v4_definition_state(type(backend))) != stable_hash(v4_definition_state(TLBackend)):
        raise RuntimeError("Candidate backend class differs from the copied original TLBackend")
    for name, method in vars(TLBackend).items():
        if isinstance(method, v4_types.FunctionType):
            actual = getattr(backend, name, None)
            if stable_hash(v4_definition_state(actual)) != stable_hash(v4_definition_state(method)):
                raise RuntimeError("Candidate backend method differs: " + name)
            function = actual.__func__ if isinstance(actual, v4_types.MethodType) else actual
            for dependency in ("normalize_heads", "ablation_hooks", "parse_json_output", "_json_no_duplicates"):
                reference = function.__globals__.get(dependency)
                if reference is None or stable_hash(v4_definition_state(reference)) != stable_hash(v4_definition_state(globals()[dependency])):
                    raise RuntimeError("Candidate backend dependency differs: " + dependency)
    if v4_backend_contract(backend.cfg) != v4_backend_contract(cfg) or backend.hook_kind != cfg["hook_kind"]:
        raise RuntimeError("Candidate generation settings differ")
    if backend.tokenizer is not backend.model.tokenizer:
        raise RuntimeError("Candidate backend/model tokenizers differ")
    state = v4_check_model(backend.model, cfg)
    if str(backend.device) != state["parameters"][0]["device"]:
        raise RuntimeError("Candidate backend device differs from its weights")
    return state


def v4_load_runtime(cfg, runtime_dir, package_versions, core_source, runtime_source, candidates=()):
    """Load/reuse pinned weights and return an independent parent namespace.

    The same original TLBackend wraps a compatible live model with a new config
    dictionary. No old backend configuration, run directory, or result is changed.
    Live checks protect code, identities, package versions, model/tokenizer config,
    parameter objects/versions/storage and hooks. They do not hash all 8B weight
    values; reuse trusts the supplied reviewed pinned model-run identity.
    """
    if not isinstance(cfg, dict):
        raise TypeError("cfg must be a dictionary")
    frozen_cfg = v4_copy.deepcopy(cfg)
    for key, expected in V4_FIXED_CONFIG.items():
        if key not in frozen_cfg or type(frozen_cfg[key]) is not type(expected) or frozen_cfg[key] != expected:
            raise ValueError("This notebook preserves the original fixed setting: " + key)
    if type(frozen_cfg.get("data_seed")) is not int or not 0 <= frozen_cfg["data_seed"] < 2**32:
        raise ValueError("data_seed must be an integer in [0, 2**32)")
    if not isinstance(frozen_cfg.get("experiment_version"), str) or not frozen_cfg["experiment_version"].strip():
        raise ValueError("A standalone experiment_version is required")
    packages = v4_check_packages(v4_copy.deepcopy(package_versions))
    source_states = v4_check_sources(core_source, runtime_source)
    identity = {"name": frozen_cfg["model_name"], "revision": frozen_cfg["model_revision"],
                "dtype": frozen_cfg["dtype"], "packages": packages,
                "python": v4_sys.version.split()[0]}
    implementation = stable_hash({"core_source": core_source, "runtime_source": runtime_source})
    fingerprint = stable_hash({"config": frozen_cfg, "model": identity, "implementation": implementation})
    directory = V4Path(runtime_dir).expanduser().resolve()
    manifest = {
        "schema": "conflict_v4_standalone_runtime_v1", "config": frozen_cfg,
        "model": identity, "implementation": implementation, "fingerprint": fingerprint,
        "generation_contract": v4_backend_contract(frozen_cfg),
        "sources": {"core": core_source, "runtime": runtime_source},
        "source_sha256": {"core": v4_hashlib.sha256(core_source.encode("utf-8")).hexdigest(),
                          "runtime": v4_hashlib.sha256(runtime_source.encode("utf-8")).hexdigest()},
    }
    manifest_path = directory / "manifest.json"
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise RuntimeError("Locked runtime identity differs. Choose a fresh v4 directory before loading weights.")
    # This guard precedes login, downloads, rewrapping, or any inference.
    if not torch.cuda.is_available():
        raise RuntimeError("The unchanged 8B runtime requires a CUDA GPU")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The unchanged runtime requires bfloat16 support")
    attempts, wrapped, mode, selected_label = [], None, None, None
    for index, candidate in enumerate(candidates):
        label = str(candidate.get("label", "candidate-" + str(index))) if isinstance(candidate, dict) else "candidate-" + str(index)
        try:
            if not isinstance(candidate, dict) or "backend" not in candidate or "model_identity" not in candidate:
                raise ValueError("Candidate must contain backend and model_identity")
            old_backend = candidate["backend"]
            v4_check_backend(old_backend, frozen_cfg, candidate["model_identity"], identity)
            old_cfg = v4_copy.deepcopy(old_backend.cfg)
            wrapped = TLBackend(old_backend.model, frozen_cfg)
            if old_backend.cfg != old_cfg:
                raise RuntimeError("Wrapping unexpectedly modified the previous backend config")
            v4_check_backend(wrapped, frozen_cfg, identity, identity)
            mode, selected_label = "reuse_live_model", label
            attempts.append({"label": label, "accepted": True})
            break
        except Exception as error:
            wrapped = None
            attempts.append({"label": label, "accepted": False,
                             "reason": type(error).__name__ + ": " + str(error)})
    login_called = False
    if wrapped is None:
        if not get_token():
            notebook_login()
            login_called = True
        seed_everything(frozen_cfg["data_seed"])
        model = HookedTransformer.from_pretrained(
            frozen_cfg["model_name"], revision=frozen_cfg["model_revision"],
            device="cuda", dtype=torch.bfloat16, n_ctx=frozen_cfg["context_window"],
            fold_ln=False, center_writing_weights=False, center_unembed=False,
        )
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        if model.tokenizer.pad_token is None:
            model.tokenizer.pad_token = model.tokenizer.eos_token
        wrapped = TLBackend(model, frozen_cfg)
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        mode = "load_pinned_model"
    model = wrapped.model
    model_state = v4_check_model(model, frozen_cfg)
    tokenizer = model.tokenizer
    source_state_hash = stable_hash(source_states)
    manifest_snapshot = v4_copy.deepcopy(manifest)
    # Recheck before committing in case a concurrent cell changed the lock.
    if manifest_path.exists() and v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot:
        raise RuntimeError("Runtime manifest changed while initializing the model")
    atomic_json(manifest_path, manifest_snapshot)
    metadata = {
        "mode": mode, "reused_candidate": selected_label, "candidate_attempts": attempts,
        "login_called": login_called, "device": str(wrapped.device),
        "weight_download": "not requested" if mode == "reuse_live_model" else "HuggingFace normal local cache; download only if required",
        "model_identity_policy": "pinned loader or compatible live model with reviewed run identity",
        "live_weight_guard": "parameter identity/version/storage checks; no full weight-content digest",
        "original_backend_config_changed": False,
    }
    parent = {
        "backend": wrapped, "model": model, "RUN_DIR": directory,
        "CFG": v4_copy.deepcopy(frozen_cfg), "MODEL_IDENTITY": v4_copy.deepcopy(identity),
        "IMPLEMENTATION_HASH": implementation, "RUN_FINGERPRINT": fingerprint,
        "PACKAGE_VERSIONS": v4_copy.deepcopy(packages),
        "runtime_manifest": v4_copy.deepcopy(manifest_snapshot),
        "loader_reuse": v4_copy.deepcopy(metadata), "runtime_metadata": v4_copy.deepcopy(metadata),
    }

    def v4_verify_live_run():
        if parent.get("backend") is not wrapped or parent.get("model") is not model:
            raise RuntimeError("Standalone runtime backend/model was replaced")
        if V4Path(parent["RUN_DIR"]).expanduser().resolve() != directory:
            raise RuntimeError("Standalone runtime directory changed")
        if (stable_hash(parent["CFG"]) != stable_hash(frozen_cfg)
                or stable_hash(parent["MODEL_IDENTITY"]) != stable_hash(identity)
                or parent["IMPLEMENTATION_HASH"] != implementation
                or parent["RUN_FINGERPRINT"] != fingerprint):
            raise RuntimeError("Standalone runtime identity/config was modified")
        if wrapped.cfg != frozen_cfg:
            raise RuntimeError("Live backend configuration changed")
        if model.tokenizer is not tokenizer or wrapped.tokenizer is not tokenizer:
            raise RuntimeError("Live tokenizer object changed")
        live_states = {name: v4_definition_state(globals().get(name)) for name in source_states}
        if stable_hash(live_states) != source_state_hash:
            raise RuntimeError("Live backend/tools/runtime implementation changed")
        if v4_sys.version.split()[0] != identity["python"] or v4_check_packages(packages) != packages:
            raise RuntimeError("Live package/Python identity changed")
        v4_check_backend(wrapped, frozen_cfg, identity, identity)
        if stable_hash(v4_check_model(model, frozen_cfg)) != stable_hash(model_state):
            raise RuntimeError("Live model parameters, configuration, or tokenizer settings changed")
        if (not manifest_path.exists()
                or v4_json.loads(manifest_path.read_text(encoding="utf-8")) != manifest_snapshot):
            raise RuntimeError("Standalone runtime manifest changed on disk")
        if stable_hash(parent["runtime_manifest"]) != stable_hash(manifest_snapshot):
            raise RuntimeError("In-memory runtime manifest changed")
        return True

    parent["verify_live_run"] = v4_verify_live_run
    v4_verify_live_run()
    reuse_path = directory / "loader-reuse.json"
    previous = v4_json.loads(reuse_path.read_text(encoding="utf-8")) if reuse_path.exists() else {"run_fingerprint": fingerprint, "loads": []}
    if previous.get("run_fingerprint") != fingerprint or not isinstance(previous.get("loads"), list):
        raise RuntimeError("Loader reuse provenance belongs to another runtime")
    previous["loads"].append(metadata)
    atomic_json(reuse_path, previous)
    return parent
'''
V5_RUNTIME_NAMESPACE = {'__name__': 'v5_runtime', 'np': np, 'torch': torch,
                        'HookedTransformer': HookedTransformer, 'get_token': get_token, 'notebook_login': notebook_login}
for v5_source, v5_filename in ((V5_CORE_SOURCE, '<v5-core>'), (V5_RUNTIME_SOURCE, '<v5-runtime>')):
    linecache.cache[v5_filename] = (len(v5_source), None, v5_source.splitlines(True), v5_filename)
    exec(compile(v5_source, v5_filename, 'exec'), V5_RUNTIME_NAMESPACE)


## 04 · Actor、Writer  And action statistics 
Actor Show complete process phases and retain city You have to fill in the hint. Writer、 Retain original definitions of actual tools and the generator; new statistics distinguish attempts from tool acceptance. 


In [ ]:
V5_GX_SOURCE = r'''
"""V5 complete action demonstrations and intermediate tool-action outcomes.

This append-only module needs the original OrderEnvironment, parse_json_output
and canonical_json. It never changes their definitions or the loaded backend.
Gold fields remain evaluator data: the Actor sees only its focal source records
and the other, fixed task fields. All reported outcomes come from real tool
transitions after free generation of complete actions.
"""
import copy as gx_copy
import random as gx_random
import time as gx_time


GX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")
GX_CITIES = (
    "Lisbon", "Prague", "Helsinki", "Warsaw", "Cairo", "Nairobi",
    "Mumbai", "Osaka", "New York", "Los Angeles", "Cape Town", "Buenos Aires",
)
GX_TEMPLATES = (
    ("gx_dispatch", "Handle request {request_id}: create one order using its resolved fields. "
     "Use its preferred item if stocked; otherwise use the listed fallback."),
    ("gx_shipment", "For request {request_id}, arrange exactly one shipment. Choose the requested "
     "preferred SKU when available, else the permitted fallback."),
    ("gx_inventory", "Execute request {request_id} through the inventory and order tools. "
     "Preserve its resolved recipient and destination, and prefer its requested SKU "
     "over the fallback whenever stock permits."),
    ("gx_submission", "Place a single order for request {request_id}. Derive the missing field "
     "from the supplied source records, use the given fixed fields, and substitute "
     "the fallback only when the preferred SKU has no stock."),
)
GX_DEMO_CITIES = ("Valletta", "Reykjavik", "Suva", "La Paz")
GX_SMOKE_TEMPLATE = (
    "gx_smoke",
    "Exercise the tool interface for request {request_id}: resolve its field, check inventory, "
    "reserve and submit one order using the allowed fallback if necessary.",
)
GX_ORDER_FIELDS = {
    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",
}


def gx_build_cases(stage="evaluation", seed=83022):
    """Return 192 evaluation sources or six independent usability smoke sources.

    A six-source block contains every attribute x stock combination. Four
    consecutive blocks also cover all four templates, so each 24-source chunk
    is balanced over all three factors. City pools cycle in shuffled order per
    attribute, covering every evaluation city without token eligibility filters.
    Opaque identifiers are independent random draws, never value/role suffixes.
    """
    if stage not in ("evaluation", "smoke"):
        raise ValueError("stage must be evaluation or smoke")
    if type(seed) is not int:
        raise ValueError("seed must be an integer")
    rng = gx_random.Random(f"gx-generalization-v1:{stage}:{seed}")
    used_ids = set()

    def gx_opaque(prefix):
        while True:
            value = prefix + "-" + f"{rng.getrandbits(64):016x}"
            if value not in used_ids:
                used_ids.add(value)
                return value

    cities = GX_CITIES if stage == "evaluation" else GX_DEMO_CITIES
    city_queues = {attribute: [] for attribute in GX_ATTRIBUTES}

    def gx_next_city(attribute):
        if not city_queues[attribute]:
            city_queues[attribute] = list(cities)
            rng.shuffle(city_queues[attribute])
        return city_queues[attribute].pop()

    templates = GX_TEMPLATES if stage == "evaluation" else (GX_SMOKE_TEMPLATE,)
    replicates = range(8) if stage == "evaluation" else range(1)
    cases = []
    for replicate in replicates:
        for template_id, template_text in templates:
            for attribute in GX_ATTRIBUTES:
                for stock_available in (True, False):
                    source_id = gx_opaque("gx-" + stage)
                    request_id = gx_opaque("req")
                    customer, alternative_customer = gx_opaque("acct"), gx_opaque("acct")
                    preferred, fallback, alternative_item = (
                        gx_opaque("sku"), gx_opaque("sku"), gx_opaque("sku"))
                    city = gx_next_city(attribute)
                    other_city = rng.choice([value for value in cities if value != city])
                    gold = {"delivery_city": city, "order_customer": customer,
                            "preferred_item": preferred}
                    competing = {"delivery_city": other_city,
                                 "order_customer": alternative_customer,
                                 "preferred_item": alternative_item}
                    case = {
                        "source_id": source_id, "stage": stage, "attribute": attribute,
                        "template_id": template_id, "template_text": template_text,
                        "replicate": replicate, "request_id": request_id,
                        "gold_fields": gold, "conflict_value": competing[attribute],
                        "fallback_item": fallback, "alternative_item": alternative_item,
                        "profile_event_id": gx_opaque("evt"), "note_event_id": gx_opaque("evt"),
                        "stock_available": stock_available,
                        "initial_env": {
                            "stock": {preferred: int(stock_available), fallback: 2,
                                      alternative_item: 2},
                            "reservations": {}, "orders": [],
                            "next_reservation": 1, "next_order": 1,
                        },
                    }
                    gx_validate_case(case)
                    cases.append(case)
    return cases


def gx_validate_case(case):
    """Validate evaluator metadata without exposing it to model generation."""
    if case.get("attribute") not in GX_ATTRIBUTES:
        raise ValueError("Unknown focal attribute")
    gold = case.get("gold_fields")
    if not isinstance(gold, dict) or set(gold) != set(GX_ATTRIBUTES):
        raise ValueError("gold_fields must contain the three task fields")
    if any(not isinstance(value, str) or not value.strip() for value in gold.values()):
        raise ValueError("Gold task fields must be nonempty strings")
    for key in ("source_id", "request_id", "conflict_value", "fallback_item",
                "profile_event_id", "note_event_id", "template_id", "template_text"):
        if not isinstance(case.get(key), str) or not case[key].strip():
            raise ValueError("Missing/nonstring case field: " + key)
    if case["conflict_value"] == gold[case["attribute"]]:
        raise ValueError("Conflict value must differ from the verified focal value")
    if case["request_id"] == gold["order_customer"]:
        raise ValueError("Request identity must not be the answer customer")
    if case["profile_event_id"] == case["note_event_id"]:
        raise ValueError("The two sources require distinct event IDs")
    if case["fallback_item"] == gold["preferred_item"]:
        raise ValueError("Fallback and preferred item must differ")
    stock = case["initial_env"]["stock"]
    if gold["preferred_item"] not in stock or stock.get(case["fallback_item"], 0) <= 0:
        raise ValueError("The preferred SKU must exist and the distinct fallback must be stocked")
    if type(case.get("stock_available")) is not bool or case["stock_available"] != (stock[gold["preferred_item"]] > 0):
        raise ValueError("stock_available does not match the initial environment")
    if case["attribute"] == "preferred_item":
        if case["conflict_value"] == case["fallback_item"] or stock.get(case["conflict_value"], 0) <= 0:
            raise ValueError("Conflicting preferred SKU must be a distinct stocked alternative")
    return case


def gx_memory(case, condition, position, arm):
    """Make paired records, then apply source-kind-only structural baselines."""
    gx_validate_case(case)
    if condition not in ("aligned", "conflict"):
        raise ValueError("Unknown context condition")
    if position not in ("profile_first", "note_first"):
        raise ValueError("Unknown input record position")
    if arm not in ("untreated", "authority_first", "authority_only"):
        raise ValueError("Unknown structural baseline")
    profile = {"kind": "verified_profile", "entity": case["request_id"],
               "attribute": case["attribute"],
               "value": case["gold_fields"][case["attribute"]],
               "source_event": case["profile_event_id"]}
    note = {"kind": "retrieved_note", "entity": case["request_id"],
            "attribute": case["attribute"],
            "value": profile["value"] if condition == "aligned" else case["conflict_value"],
            "source_event": case["note_event_id"]}
    records = [profile, note] if position == "profile_first" else [note, profile]
    if arm == "authority_first":
        records = ([record for record in records if record["kind"] == "verified_profile"]
                   + [record for record in records if record["kind"] != "verified_profile"])
    elif arm == "authority_only":
        records = [record for record in records if record["kind"] == "verified_profile"]
    return gx_copy.deepcopy(records)


def gx_task_payload(case):
    """Only the other two fixed fields and fallback accompany the request ID."""
    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],
            "fixed_fields": {attribute: case["gold_fields"][attribute]
                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},
            "fallback_item": case["fallback_item"]}


def gx_actor_examples():
    """Independent tool examples cover all field mappings and real fallback.

    These short teaching cases use only demo cities and identifiers. They never
    depend on evaluation cases, and do not furnish actions for evaluated tasks.
    """
    examples = []
    for index, attribute in enumerate(GX_ATTRIBUTES):
        request = ("req-demo-71d3", "req-demo-a926", "req-demo-4f80")[index]
        customer = ("acct-demo-8c31", "acct-demo-90ea", "acct-demo-2d74")[index]
        preferred = ("sku-demo-31ab", "sku-demo-e072", "sku-demo-64dc")[index]
        fallback = ("sku-demo-8f62", "sku-demo-72de", "sku-demo-0a13")[index]
        city = GX_DEMO_CITIES[index]
        fields = {"delivery_city": city, "order_customer": customer, "preferred_item": preferred}
        available = index != 2
        env = OrderEnvironment({"stock": {preferred: int(available), fallback: 2},
                                "next_reservation": (7, 19, 41)[index]})
        events = []

        def gx_demo_step(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        observation = gx_demo_step({"tool": "check_stock", "item": preferred})
        chosen = preferred if observation["available"] > 0 else fallback
        if chosen != preferred:
            gx_demo_step({"tool": "check_stock", "item": fallback})
        reservation = gx_demo_step({"tool": "reserve_item", "item": chosen})["reservation"]
        gx_demo_step({"tool": "create_order", "customer": customer, "item": chosen,
                      "city": city, "reservation": reservation})
        examples.append({
            "task": {"request_id": request, "resolve_attribute": attribute,
                     "fixed_fields": {key: value for key, value in fields.items() if key != attribute},
                     "fallback_item": fallback},
            "memory": [{"kind": "verified_profile", "entity": request, "attribute": attribute,
                        "value": fields[attribute], "source_event": "evt-demo-" + str(index)}],
            "events": events,
        })
    return examples


def gx_actor_example_text():
    """Teach every next-call phase, including the initially empty history.

    The v5 smoke model copied the final-call pattern before any reservation.
    Render every legal prefix of the same three independent teaching cases,
    not only the final create_order prefix. Every observation is produced by
    the unchanged tools. All ten examples are static and identical across arms;
    no demonstration is chosen using an evaluated task or its hidden truth.
    """
    continuations = []
    for example in gx_actor_examples():
        for step, event in enumerate(example["events"]):
            continuations.append(
                "INDEPENDENT EXAMPLE:\nTASK_JSON:\n" + canonical_json(example["task"])
                + "\nMEMORY_JSON:\n" + canonical_json(example["memory"])
                + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(example["events"][:step])
                + "\nNEXT_TOOL_CALL:\n" + canonical_json(event["action"])
                + "\nEND_EXAMPLE")
    return "\n".join(continuations)


def gx_actor_prompt(case, memory, events):
    """Present the effective arm's memory verbatim; do not insert focal gold."""
    visible = [{"action": event.get("action"), "observation": event.get("observation")}
               for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute one order using the tools.\n"
        "A request_id identifies a request, not a customer account. Resolve the indicated attribute "
        "for that request from its records, then combine it with the task's fixed_fields. "
        "delivery_city maps to order.city; order_customer maps to order.customer; preferred_item "
        "is the SKU to check and reserve when available. The fallback_item is allowed only if "
        "the resolved preferred SKU is out of stock.\n"
        "Records are oldest to newest. Use the latest verified_profile for each request and attribute. "
        "A retrieved_note is unverified and cannot change a verified profile, even if it is later. "
        "Actions and tool observations do not update the request's fields.\n"
        "Check the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen "
        "available SKU, then create exactly one order. Copy the reservation ID only from its "
        "successful reserve_item observation; a stock count is not a reservation.\n"
        "Output exactly one complete JSON object for the next tool call, without explanation. "
        "Every create_order requires all four arguments: customer, item, city, reservation. "
        "Use city as the tool key even when the task field is named delivery_city.\n"
        'Tools: {"tool":"check_stock","item":"SKU"}; '
        '{"tool":"reserve_item","item":"SKU"}; '
        '{"tool":"create_order","customer":"ACCOUNT","item":"SKU","city":"CITY","reservation":"RESERVATION"}.\n'
        "Independent next-call examples (never copy their values or IDs):\n"
        + gx_actor_example_text()
        + "\nEND_EXAMPLES. Execute this independent request:\n"
        + case["template_text"].format(request_id=case["request_id"])
        + "\nOnly this request's EXECUTED_TOOL_HISTORY_JSON records completed actions. "
        "An empty history means nothing has been checked or reserved: start with check_stock "
        "for the resolved preferred SKU. After a positive stock result use reserve_item. "
        "If the preferred SKU is unavailable, check the fallback before reserving it. "
        "Use create_order only after reserve_item succeeded in this request's history. "
        "Never invent a reservation ID or take one from an example."
        + "\nGenerate only the next call. For create_order include customer, item, city, "
        "and the reservation from a successful reserve_item observation. Resolve city from "
        "this request's delivery_city, including when it is supplied in fixed_fields."
        + "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case))
        + "\nMEMORY_JSON:\n" + canonical_json(memory)
        + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(visible)
        + "\nNEXT_TOOL_CALL:\n"
    )


def gx_expected_order(case, initial_state=None):
    """Evaluator-only final fields, including stock-contingent SKU truth."""
    state = case["initial_env"] if initial_state is None else initial_state
    fields = case["gold_fields"]
    preferred = fields["preferred_item"]
    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]
    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}


def gx_value_missing(value):
    """Absent/null/blank values are not explicit choices of a wrong value.

    Nonstring values are schema/type failures, not choices of a string-valued
    attribute. They are excluded from wrong-choice rates by gx_wrong_value;
    the schema-complete indicators still expose their invalidity.
    """
    return value is None or (isinstance(value, str) and not value.strip())


def gx_wrong_value(value, expected):
    """Count only an explicitly supplied, nonblank string that is incorrect."""
    return isinstance(value, str) and bool(value.strip()) and value != expected


def gx_order_schema_complete(action):
    """A complete order attempt has exactly five keys with nonblank strings.

    This is a diagnostic; the original executable schema remains unchanged.
    It deliberately treats a whitespace-only value as missing even if the
    executable tool's nonempty-string check would accept that value.
    """
    return (isinstance(action, dict)
            and set(action) == {"tool", "customer", "item", "city", "reservation"}
            and action.get("tool") == "create_order"
            and all(isinstance(value, str) and bool(value.strip()) for value in action.values()))


def gx_score_actor(case, events, orders, initial_state):
    """Score accepted fields and the executed prerequisite subsequence.

    A wrong focal field is counted on any accepted order, independently of the
    correctness of the other fields. Missing orders are failures and have their
    own indicator; they never masquerade as correct field decisions. Tool
    recovery is allowed: strict success requires a valid final check/reserve/
    order subsequence, while all earlier tool errors remain reported separately.

    Intermediate rates are source-level booleans, not counts of repeated calls.
    The first order attempt is the first parsed create_order action, including
    a rejected one; any-attempt metrics cover all such actions. Missing or
    nonstring focal values do not count as explicit wrong choices. Reservation
    acceptance requires that reserve_item's own observation has ok=True.
    Expected items include the legal initial-stock fallback. Conflict-SKU
    reservation metrics apply only to preferred_item sources (None otherwise).
    """
    expected = gx_expected_order(case, initial_state)
    focal = GX_ORDER_FIELDS[case["attribute"]]
    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())
    preferred = case["gold_fields"]["preferred_item"]
    chosen = expected["item"]
    checked, grounded = {}, set()
    checked_preferred = checked_chosen = reserved_chosen = path_valid = False
    tool_errors = []
    actions = []
    order_attempts, reservation_attempts, accepted_reservations = [], [], []
    for step, event in enumerate(events):
        action = event.get("action")
        observation = event.get("observation")
        if isinstance(action, dict):
            actions.append(action)
            if action.get("tool") == "create_order":
                order_attempts.append(action)
            elif action.get("tool") == "reserve_item":
                reservation_attempts.append(action)
                if isinstance(observation, dict) and observation.get("ok") is True:
                    accepted_reservations.append(action)
        if not isinstance(action, dict) or not isinstance(observation, dict):
            continue
        if not observation.get("ok", False):
            tool_errors.append({"step": step, "error": observation.get("error")})
            continue
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock":
            checked[item] = observation["available"]
            checked_preferred |= item == preferred
            checked_chosen |= item == chosen and observation["available"] > 0
        elif tool == "reserve_item" and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded.add(observation["reservation"])
        elif tool == "create_order" and item == chosen:
            path_valid |= action.get("reservation") in grounded
    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}
    format_failure = any(event.get("parse_error") is not None for event in events) or any(
        error["error"] in protocol_errors for error in tool_errors)
    first_order = order_attempts[0] if order_attempts else None
    order_fields = ("customer", "item", "city", "reservation")
    first_missing_attribute = first_order is not None and gx_value_missing(first_order.get(focal))
    first_missing_city = first_order is not None and gx_value_missing(first_order.get("city"))
    first_missing_required = first_order is not None and any(
        gx_value_missing(first_order.get(field)) for field in order_fields)
    conflict_item_applicable = case["attribute"] == "preferred_item"
    return {
        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),
        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),
        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),
        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),
        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()
                                                if key != focal) for order in orders),
        "no_accepted_order": not orders, "accepted_order_count": len(orders),
        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),
        "tool_error_count": len(tool_errors), "action_count": len(actions),
        "order_attempted": first_order is not None,
        "order_accepted": bool(orders), "order_attempt_count": len(order_attempts),
        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],
        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],
        "any_order_conflict_value": any(action.get(focal) == case["conflict_value"] for action in order_attempts),
        "first_order_wrong_attribute": first_order is not None and gx_wrong_value(first_order.get(focal), expected[focal]),
        "any_order_wrong_attribute": any(gx_wrong_value(action.get(focal), expected[focal]) for action in order_attempts),
        "first_order_missing_attribute": bool(first_missing_attribute),
        "any_order_missing_attribute": any(gx_value_missing(action.get(focal)) for action in order_attempts),
        "first_order_missing_city": bool(first_missing_city),
        "any_order_missing_city": any(gx_value_missing(action.get("city")) for action in order_attempts),
        "first_order_missing_required_fields": bool(first_missing_required),
        "any_order_missing_required_fields": any(any(gx_value_missing(action.get(field)) for field in order_fields)
                                                  for action in order_attempts),
        "first_order_schema_complete": first_order is not None and gx_order_schema_complete(first_order),
        "any_order_schema_complete": any(gx_order_schema_complete(action) for action in order_attempts),
        "reservation_attempted": bool(reservation_attempts),
        "reservation_accepted": bool(accepted_reservations),
        "reservation_attempt_count": len(reservation_attempts),
        "accepted_reservation_count": len(accepted_reservations),
        "attempted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                 for action in reservation_attempts),
        "accepted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                for action in accepted_reservations),
        "attempted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in reservation_attempts)
            if conflict_item_applicable else None),
        "accepted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in accepted_reservations)
            if conflict_item_applicable else None),
        "prerequisite_path_valid": bool(path_valid),
        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),
        "reserved_chosen": bool(reserved_chosen),
        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,
    }


def gx_run_actor(backend, case, memory, cfg):
    """Generate whole actions and execute the original tools without repairs."""
    gx_validate_case(case)
    cfg = dict(cfg)
    max_actions = cfg.get("max_actions", 6)
    if type(max_actions) is not int or max_actions < 1:
        raise ValueError("max_actions must be a positive integer")
    initial = gx_copy.deepcopy(case["initial_env"])
    presented = gx_copy.deepcopy(memory)
    env = OrderEnvironment(initial)
    events = []
    token_count = prompt_tokens = 0
    model_seconds = 0.0
    terminal = "action_budget"
    started = gx_time.perf_counter()
    for step in range(max_actions):
        prompt = gx_actor_prompt(case, presented, events)
        output = backend.generate(prompt, role="actor", heads=(),
                                  max_new_tokens=cfg.get("actor_max_new_tokens", 128),
                                  use_cache=cfg.get("use_cache", True))
        raw = output.get("text", "")
        action, parse_error = parse_json_output(raw, dict)
        event = {"step": step, "prompt": prompt, "raw_output": raw,
                 "action": action, "observation": None, "parse_error": parse_error,
                 "stop_reason": output.get("stop_reason")}
        token_count += int(output.get("token_count", 0))
        prompt_tokens += int(output.get("prompt_tokens", 0))
        model_seconds += float(output.get("seconds", 0.0))
        if parse_error is not None:
            events.append(event)
            terminal = "format_failure"
            break
        observation = env.step(action)
        event["observation"] = observation
        events.append(event)
        if observation.get("error") in {"action_must_be_object", "unknown_tool", "invalid_arguments"}:
            terminal = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            terminal = "order_created"
            break
    orders = gx_copy.deepcopy(env.state["orders"][len(initial.get("orders", [])):])
    metrics = gx_score_actor(case, events, orders, initial)
    metrics["budget_exhausted"] = terminal == "action_budget"
    return {
        "source_id": case["source_id"], "stage": case["stage"],
        "attribute": case["attribute"], "request_id": case["request_id"],
        "template_id": case["template_id"], "stock_available": case["stock_available"],
        "memory": presented, "events": events, "orders": orders,
        "actions": [gx_copy.deepcopy(event["action"]) for event in events if event.get("action") is not None],
        "initial_state": initial, "final_state": env.snapshot(),
        "terminal": terminal, "metrics": metrics, **metrics,
        "model_calls": len(events), "token_count": token_count, "prompt_tokens": prompt_tokens,
        "model_seconds": model_seconds, "seconds": gx_time.perf_counter() - started,
        "evaluation_layer": "end_to_end", "heads": [],
    }

# Append-only Writer evaluation for the three-attribute generalization study.
# All extension definitions use gx_/GX_ names and leave the loaded v3 model,
# functions, configuration, datasets, interventions and run directory untouched.
import copy as gx_copy
import math as gx_math


GX_WRITER_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")


def gx_history(case, memory, actor_result):
    """Freeze the arm's supplied records and its actually generated Actor events.

    No profile is reconstructed from case.gold_fields. In particular, filtering
    or reordering performed by an input baseline is preserved. The sequence
    index records presentation order; it is not a fabricated event timestamp.
    Wrong orders and fallback selections remain observed actions, never profile
    updates. Actor prompt text is not duplicated into the Writer history.
    """
    if not isinstance(memory, list):
        raise TypeError("gx_history requires the effective presented memory list")
    if not isinstance(actor_result, dict) or not isinstance(actor_result.get("events"), list):
        raise TypeError("gx_history requires an actual Actor result with events")
    if actor_result.get("source_id") != case["source_id"]:
        raise ValueError("Actor trajectory belongs to another source")
    if "memory" in actor_result and actor_result["memory"] != memory:
        raise ValueError("Actor trajectory used a different arm's effective memory")
    for field in ("attribute", "request_id"):
        if field in actor_result and actor_result[field] != case[field]:
            raise ValueError(f"Actor trajectory has a different {field}")
    history = []
    for record in memory:
        if not isinstance(record, dict):
            raise TypeError("Effective memory records must be dictionaries")
        event = gx_copy.deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    for index, event in enumerate(actor_result["events"]):
        if not isinstance(event, dict):
            raise TypeError("Actor events must be dictionaries")
        origin = f'{case["source_id"]}:observed:{index}'
        if event.get("action") is not None:
            history.append({
                "kind": "observed_action", "sequence": len(history),
                "source_event": origin + ":action",
                "action": gx_copy.deepcopy(event["action"]),
            })
        elif event.get("raw_output") is not None or event.get("parse_error") is not None:
            history.append({
                "kind": "observed_output", "sequence": len(history),
                "source_event": origin + ":output",
                "text": gx_copy.deepcopy(event.get("raw_output", "")),
                "parse_error": gx_copy.deepcopy(event.get("parse_error")),
            })
        if event.get("observation") is not None:
            history.append({
                "kind": "tool_observation", "sequence": len(history),
                "source_event": origin + ":observation",
                "observation": gx_copy.deepcopy(event["observation"]),
            })
    return history


def gx_writer_reference(history):
    """Latest authorized profile per entity/attribute, never notes or executions."""
    if not isinstance(history, list):
        raise TypeError("Writer reference history must be a list")
    profiles = []
    for index, event in enumerate(history):
        if not isinstance(event, dict):
            raise TypeError("Writer history events must be dictionaries")
        if event.get("kind") != "verified_profile":
            continue
        fields = ("entity", "attribute", "value", "source_event")
        if any(not isinstance(event.get(key), str) or not event[key].strip() for key in fields):
            raise ValueError("Malformed verified profile in Writer history")
        if event["attribute"] not in GX_WRITER_ATTRIBUTES:
            raise ValueError("Unsupported verified profile attribute")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float))
                or not gx_math.isfinite(sequence)):
            raise ValueError("Verified profile sequence must be a finite number")
        profiles.append((sequence, index, event))
    reference = {}
    for _, _, event in sorted(profiles, key=lambda item: (item[0], item[1])):
        reference[event["entity"], event["attribute"]] = {
            "kind": "preference", **{key: event[key] for key in ("entity", "attribute", "value", "source_event")}
        }
    return reference


def gx_writer_schema_error(memory):
    if not isinstance(memory, list):
        return "memory must be a JSON list"
    required = {"kind", "entity", "attribute", "value", "source_event"}
    for index, record in enumerate(memory):
        if not isinstance(record, dict) or set(record) != required:
            return f"record {index}: keys must be exactly {sorted(required)}"
        if record["kind"] != "preference":
            return f"record {index}: kind must be preference"
        if record["attribute"] not in GX_WRITER_ATTRIBUTES:
            return f"record {index}: unsupported attribute"
        if any(not isinstance(record[key], str) or not record[key].strip()
               for key in ("entity", "attribute", "value", "source_event")):
            return f"record {index}: record fields must be nonempty strings"
    return None


def gx_writer_presented_history(history):
    """Preserve the arm's actual presentation order, excluding internal indices."""
    if not isinstance(history, list):
        raise TypeError("Writer history must be a list")
    presented = gx_copy.deepcopy(history)
    previous = None
    for index, event in enumerate(presented):
        if not isinstance(event, dict):
            raise TypeError("Writer history events must be dictionaries")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float))
                or not gx_math.isfinite(sequence)):
            raise ValueError("Writer history sequence must be a finite number")
        if previous is not None and sequence < previous:
            raise ValueError("Writer history presentation order conflicts with sequence")
        previous = sequence
        event.pop("sequence", None)
    return presented


def gx_writer_examples():
    """Independent local demonstrations; no evaluation cities/accounts/SKUs."""
    def gx_profile(request, attribute, value, source):
        return {"kind": "verified_profile", "entity": request,
                "attribute": attribute, "value": value, "source_event": source}

    city = gx_profile("demo-gx-request-u", "delivery_city", "DemoHarbor", "demo-gx-profile-u")
    customer = gx_profile("demo-gx-request-v", "order_customer", "demo-account-v7", "demo-gx-profile-v")
    item = gx_profile("demo-gx-request-w", "preferred_item", "demo-sku-w3", "demo-gx-profile-w")
    return [
        {"name": "Retain a verified city despite an unverified note",
         "request_id": city["entity"], "attribute": city["attribute"],
         "events": [city, {**city, "kind": "retrieved_note", "value": "DemoMeadow",
                            "source_event": "demo-gx-note-u"}],
         "memory": [{**city, "kind": "preference"}]},
        {"name": "A generated order is not an authorized customer update",
         "request_id": customer["entity"], "attribute": customer["attribute"],
         "events": [{**customer, "kind": "retrieved_note", "value": "demo-account-v9",
                     "source_event": "demo-gx-note-v"}, customer,
                    {"kind": "observed_action", "source_event": "demo-gx-action-v",
                     "action": {"tool": "create_order", "customer": "demo-account-v9",
                                "city": "DemoHarbor", "item": "demo-sku-v2", "reservation": "R0041"}}],
         "memory": [{**customer, "kind": "preference"}]},
        {"name": "Using a fallback item does not change the preferred item",
         "request_id": item["entity"], "attribute": item["attribute"],
         "events": [item,
                    {"kind": "observed_action", "source_event": "demo-gx-action-w0",
                     "action": {"tool": "check_stock", "item": "demo-sku-w3"}},
                    {"kind": "tool_observation", "source_event": "demo-gx-observation-w0",
                     "observation": {"ok": True, "item": "demo-sku-w3", "available": 0}},
                    {"kind": "observed_action", "source_event": "demo-gx-action-w1",
                     "action": {"tool": "create_order", "customer": "demo-account-w5",
                                "city": "DemoMeadow", "item": "demo-sku-w8", "reservation": "R0051"}}],
         "memory": [{**item, "kind": "preference"}]},
    ]


def gx_writer_prompt(case, history):
    """Whole JSON generation for the request's focal attribute, without a gold prefix."""
    if case["attribute"] not in GX_WRITER_ATTRIBUTES:
        raise ValueError("Unsupported focal Writer attribute")
    examples = "\n\n".join(
        "EXAMPLE " + str(index + 1) + ": " + example["name"]
        + "\nExample request: " + canonical_json(example["request_id"])
        + "\nExample events:\n" + canonical_json(example["events"])
        + "\nExample complete output:\n" + canonical_json(example["memory"])
        for index, example in enumerate(gx_writer_examples())
    )
    return (
        "You are the memory writer for an order-processing assistant.\n"
        "Return ONLY a single complete JSON list. Each record must have exactly "
        'the fields {"kind":"preference","entity":"request id",'
        '"attribute":"attribute name","value":"value","source_event":"profile event id"}.\n'
        "Supported attributes are delivery_city, order_customer, and preferred_item. "
        "The entity is the request identifier; it is not the order's customer account.\n"
        "Only verified_profile records authorize preference facts. For each entity "
        "and attribute, preserve the latest supplied verified_profile's exact value "
        "and source_event. A retrieved_note cannot update a profile, even if it "
        "appears later, names the same request, or is repeated in an action. "
        "Actions and tool observations describe execution, not authorized updates. "
        "In particular, a fallback item selected because of stock does not replace "
        "the verified preferred_item. Do not infer preference facts from successful "
        "orders, tool observations, unrelated metadata, or previous generated outputs.\n"
        "Keep one record per authorized entity and attribute, retaining every supplied "
        "verified profile. Put the current request's focal record first, then other "
        "authorized records. Do not invent additional facts, omit a fact because a "
        "note disagrees, or copy demonstration identifiers. No explanations, extra "
        "fields, or Markdown fences. Stop after the closing ] of the single list.\n\n"
        + examples + "\nEND_GX_WRITER_EXAMPLES.\n\nCURRENT_TASK\nREQUEST_JSON:\n"
        + canonical_json({"request_id": case["request_id"], "attribute": case["attribute"]})
        + "\nEVENTS_JSON:\n" + canonical_json(gx_writer_presented_history(history))
        + "\nMEMORY_JSON:\n"
    )


def gx_score_memory(case, history, memory, valid):
    """Score generated values and provenance against supplied authorized evidence.

    The preferred_item gold remains the profile's preference even if the actual
    Actor correctly chose an in-stock fallback. Invalid memories remain failures
    in every planned observation; wrong value, omission and provenance are kept
    separate. Whole-memory correctness rejects extra unsupported records.
    """
    reference = gx_writer_reference(history)
    valid = bool(valid) and gx_writer_schema_error(memory) is None
    records = memory if valid else []
    by_key = {}
    for record in records:
        by_key.setdefault((record["entity"], record["attribute"]), []).append(record)
    focal_key = (case["request_id"], case["attribute"])
    focal_reference = reference.get(focal_key)
    focal_records = by_key.get(focal_key, [])
    has_reference = bool(reference)
    value_correct = (valid and len(focal_records) == 1
                     and focal_records[0]["value"] == focal_reference["value"]
                     if focal_reference is not None else None)
    source_correct = (valid and len(focal_records) == 1
                      and focal_records[0]["source_event"] == focal_reference["source_event"]
                      if focal_reference is not None else None)
    conflict_error = (any(record["value"] == case["conflict_value"] for record in focal_records)
                      and case["conflict_value"] != focal_reference["value"]
                      if focal_reference is not None else None)
    missing = sum(key not in by_key for key in reference)
    unsupported = (sum(key not in reference or record["value"] != reference[key]["value"]
                       for key, entries in by_key.items() for record in entries)
                   if has_reference else None)
    source_mismatch = (sum(key in reference and record["source_event"] != reference[key]["source_event"]
                           for key, entries in by_key.items() for record in entries)
                       if has_reference else None)
    correct_count = (sum(len(by_key.get(key, [])) == 1
                         and by_key[key][0]["value"] == profile["value"]
                         for key, profile in reference.items()) if has_reference else None)
    joint_count = (sum(len(by_key.get(key, [])) == 1
                       and by_key[key][0]["value"] == profile["value"]
                       and by_key[key][0]["source_event"] == profile["source_event"]
                       for key, profile in reference.items()) if has_reference else None)
    return {
        "target_correct": value_correct, "target_source_correct": source_correct,
        "target_joint_correct": (bool(value_correct and source_correct)
                                 if focal_reference is not None else None),
        "target_conflict_error": conflict_error,
        "whole_memory_correct": (valid and len(records) == len(reference)
                                 and joint_count == len(reference) if has_reference else None),
        "format_failure": not valid,
        "omission": bool(missing) if has_reference else None,
        "target_omission": not focal_records if focal_reference is not None else None,
        "duplicate_count": sum(max(0, len(entries) - 1) for entries in by_key.values()),
        "missing_count": missing if has_reference else None,
        "unsupported_count": unsupported, "source_mismatch_count": source_mismatch,
        "required_count": len(reference), "record_count": len(records),
        "correct_count": correct_count, "joint_correct_count": joint_count,
        "reference_available": focal_reference is not None,
        "reference_policy": "gx_latest_supplied_verified_profile_v1",
    }


def gx_run_writer(backend, case, history, cfg):
    """Run the original backend with empty heads and retain complete raw output."""
    evidence = gx_copy.deepcopy(history)
    before = stable_hash(evidence)
    prompt = gx_writer_prompt(case, evidence)
    generation = backend.generate(
        prompt, role="writer", heads=(),
        max_new_tokens=cfg.get("writer_max_new_tokens", 384),
        use_cache=cfg.get("use_cache", True),
    )
    raw = generation.get("text", "")
    memory, parse_error = parse_json_output(raw, expected_type=list)
    error = parse_error or gx_writer_schema_error(memory)
    valid = error is None
    memory = memory if valid else None
    if stable_hash(history) != before or stable_hash(evidence) != before:
        raise RuntimeError("GX Writer modified its actual input history")
    return {
        "memory": memory, "valid": valid, "raw": raw, "error": error,
        "metrics": gx_score_memory(case, evidence, memory, valid),
        "history_hash": before, "prompt_hash": stable_hash(prompt),
        "token_count": generation.get("token_count", 0),
        "prompt_tokens": generation.get("prompt_tokens", 0),
        "seconds": generation.get("seconds", 0.0),
        "stop_reason": generation.get("stop_reason"),
        "model_calls": 1, "heads": [],
        "memory_origin": "gx_model_generated_from_effective_input_and_actual_actor_trace",
    }

"""V5 protocol-complete actions, intermediate metrics, and smoke-gated evaluation."""
import copy
import csv
import datetime
import itertools
import json
import math
import marshal
from pathlib import Path
import random
import zipfile


GX_ARMS = ("untreated", "authority_first", "authority_only")
GX_CONDITIONS = ("aligned", "conflict")
GX_POSITIONS = ("profile_first", "note_first")
GX_PRIMARY_METRICS = ("actor_accepted_wrong_attribute", "actor_task_success",
                      "writer_target_joint_correct", "writer_target_conflict_error")
GX_ACTION_METRICS = (
    "actor_reservation_attempted", "actor_reservation_accepted",
    "actor_attempted_wrong_reservation_item", "actor_accepted_wrong_reservation_item",
    "actor_attempted_conflict_reservation_item", "actor_accepted_conflict_reservation_item",
    "actor_order_attempted", "actor_order_accepted",
    "actor_first_order_wrong_attribute", "actor_any_order_wrong_attribute",
    "actor_first_order_conflict_value", "actor_any_order_conflict_value",
    "actor_first_order_missing_attribute", "actor_any_order_missing_attribute",
    "actor_first_order_missing_city", "actor_any_order_missing_city",
    "actor_first_order_missing_required_fields", "actor_any_order_missing_required_fields",
    "actor_first_order_schema_complete", "actor_any_order_schema_complete",
)
GX_ACTION_PAIRED_METRICS = (
    "actor_attempted_wrong_reservation_item", "actor_attempted_conflict_reservation_item",
    "actor_accepted_wrong_reservation_item", "actor_accepted_conflict_reservation_item",
    "actor_first_order_wrong_attribute", "actor_first_order_conflict_value",
    "actor_any_order_wrong_attribute", "actor_any_order_conflict_value",
    "actor_any_order_missing_city",
)
GX_METRIC_APPLICABILITY = {
    "actor_attempted_conflict_reservation_item": ("preferred_item",),
    "actor_accepted_conflict_reservation_item": ("preferred_item",),
}
GX_READINESS_RULES = {
    "version": "v5_control_usability_v1",
    "required_smoke_sources": 6,
    "controls": [["untreated", "aligned", "profile_first"],
                 ["untreated", "aligned", "note_first"],
                 ["authority_only", "aligned", "profile_first"]],
    "maximum_protocol_failures_per_control": 0,
    "required_first_complete_city_per_control": 6,
    "minimum_task_successes_per_control": 5,
    "selection_policy": "Fixed engineering guardrails; no conflict-error-rate criterion; never select evaluation cases by outcomes",
}


def gx_code_signature():
    """Detect redefining live extension functions/constants after the run is locked."""
    definitions = {}
    for name, value in list(globals().items()):
        if not name.startswith(("gx_", "GX")):
            continue
        if hasattr(value, "__code__"):
            # Version 0 avoids reference/interning flags that can change after
            # a code object is first executed despite identical executable code.
            definitions[name] = marshal.dumps(value.__code__, 0).hex()
        elif isinstance(value, type):
            definitions[name] = {key: marshal.dumps(method.__code__, 0).hex()
                                 for key, method in vars(value).items() if hasattr(method, "__code__")}
        elif isinstance(value, (str, bool, int, float, list, tuple, dict)):
            definitions[name] = value
    return stable_hash(definitions)


def gx_wilson(successes, total):
    if not total:
        return [None, None]
    z = 1.959963984540054
    p = successes / total
    divisor = 1 + z * z / total
    center = (p + z * z / (2 * total)) / divisor
    radius = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / divisor
    return [max(0.0, center - radius), min(1.0, center + radius)]


def gx_pair_stats(before, after, seed=83022, reps=2000):
    """One paired difference per source; exact binary discordance test."""
    if set(before) != set(after) or not before:
        raise ValueError("Paired source coverage differs or is empty")
    sources = sorted(before)
    if any(before[s] not in (0, 1, False, True) or after[s] not in (0, 1, False, True)
           for s in sources):
        raise ValueError("Paired metric must be binary")
    differences = [int(after[s]) - int(before[s]) for s in sources]
    n = len(sources)
    plus, minus = differences.count(1), differences.count(-1)
    discordant = plus + minus
    exact = (min(1.0, 2 * sum(math.comb(discordant, k)
                             for k in range(min(plus, minus) + 1)) / 2 ** discordant)
             if discordant else 1.0)
    degenerate = len(set(differences)) == 1
    if degenerate:
        ci = [sum(differences) / n] * 2
    else:
        rng = random.Random(seed)
        draws = sorted(sum(rng.choices(differences, k=n)) / n for _ in range(reps))
        ci = [draws[int(.025 * (reps - 1))], draws[int(.975 * (reps - 1))]]
    return {"n_sources": n, "before_count": sum(before.values()),
            "after_count": sum(after.values()), "difference_after_minus_before": sum(differences) / n,
            "bootstrap_ci_low": ci[0], "bootstrap_ci_high": ci[1],
            "bootstrap_degenerate": degenerate,
            "ci_note": ("Degenerate empirical bootstrap; not evidence of population equivalence."
                        if degenerate else "Source-paired percentile bootstrap; descriptive, unadjusted."),
            "discordant_0_to_1": plus, "discordant_1_to_0": minus,
            "exact_discordance_p_unadjusted": exact}


def gx_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = sorted(set().union(*(row.keys() for row in rows))) if rows else ["no_completed_rows"]
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: canonical_json(value) if isinstance(value, (dict, list, tuple)) else value
                             for key, value in row.items()})
    temporary.replace(path)


def gx_read_checked(path, signature):
    record = json.loads(Path(path).read_text(encoding="utf-8"))
    if record.get("signature") != signature:
        raise RuntimeError(f"Stale extension checkpoint: {path}")
    if stable_hash(record.get("result")) != record.get("result_hash"):
        raise RuntimeError(f"Corrupt extension checkpoint: {path}")
    return record["result"]


def gx_write_checked(path, signature, result):
    atomic_json(path, {"signature": signature, "result_hash": stable_hash(result), "result": result})


class GXCachedBackend:
    """Content-addressed greedy generations; display-arm names never enter prompts."""
    def __init__(self, backend, directory, identity_hash):
        self.backend = backend
        self.directory = Path(directory)
        self.identity_hash = identity_hash
        self.new_calls = 0
        self.cache_hits = 0

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        if heads:
            raise ValueError("This extension evaluates unablated baselines only")
        payload = {"identity": self.identity_hash, "prompt": prompt, "role": role,
                   "heads": [], "max_new_tokens": max_new_tokens, "use_cache": use_cache}
        signature = stable_hash(payload)
        path = self.directory / (signature + ".json")
        if path.exists():
            self.cache_hits += 1
            return copy.deepcopy(gx_read_checked(path, signature))
        result = self.backend.generate(prompt, role=role, heads=(),
                                       max_new_tokens=max_new_tokens, use_cache=use_cache)
        gx_write_checked(path, signature, result)
        self.new_calls += 1
        return copy.deepcopy(result)


class GXExperiment:
    """Keep the parent's notebook state and all old checkpoints untouched."""
    def __init__(self, namespace, source_bundle, root_dir=None, cfg=None):
        required = ("backend", "RUN_DIR", "RUN_FINGERPRINT", "IMPLEMENTATION_HASH",
                    "MODEL_IDENTITY", "CFG", "verify_live_run")
        missing = [key for key in required if key not in namespace]
        if missing:
            raise RuntimeError(" Please stay where you are.  v3 runtime  Finish Model / Functions / Initialization of identity; missing: " + ", ".join(missing))
        namespace["verify_live_run"]()
        self.parent = namespace
        self.backend = namespace["backend"]
        if not hasattr(self.backend, "cfg") or stable_hash(self.backend.cfg) != stable_hash(namespace["CFG"]):
            raise RuntimeError("backend.cfg  And Hara  CFG  mismatch; restore the original model configuration before resuming the original experiment. ")
        self.original_dir = Path(namespace["RUN_DIR"])
        self.root = (Path(root_dir) if root_dir is not None else
                     self.original_dir.parent / (self.original_dir.name + "_protocol_actions_v5p1"))
        if self.root.resolve() == self.original_dir.resolve() or self.root.resolve().is_relative_to(self.original_dir.resolve()):
            raise ValueError(" Extension directory must be located in the original  RUN_DIR  Outside ")
        self.cfg = {"actor_max_new_tokens": namespace["CFG"].get("actor_max_new_tokens", 128),
                    "writer_max_new_tokens": namespace["CFG"].get("writer_max_new_tokens", 384),
                    "max_actions": 6, "use_cache": namespace["CFG"].get("use_cache", True),
                    "bootstrap_reps": 2000}
        if cfg:
            unknown = set(cfg) - set(self.cfg)
            if unknown:
                raise ValueError(f"Unknown extension configuration: {unknown}")
            self.cfg.update(cfg)
        for key in ("actor_max_new_tokens", "writer_max_new_tokens", "max_actions", "bootstrap_reps"):
            if type(self.cfg[key]) is not int or self.cfg[key] < (100 if key == "bootstrap_reps" else 1):
                raise ValueError(f"Invalid {key}")
        if type(self.cfg["use_cache"]) is not bool:
            raise ValueError("use_cache must be boolean")
        self.cases = {stage: gx_build_cases(stage=stage, seed=83022) for stage in ("smoke", "evaluation")}
        self.protocol = {
            "version": "conflict_v5p1_protocol_actions_v1", "seed": 83022,
            "source_counts": {k: len(v) for k, v in self.cases.items()},
            "attributes": list(GX_ATTRIBUTES), "cities": list(GX_CITIES),
            "conditions": list(GX_CONDITIONS), "positions": list(GX_POSITIONS), "arms": list(GX_ARMS),
            "unit": "source_id; never a condition/position/baseline row",
            "model_intervention": "none", "layer": "end_to_end",
            "baseline_policy": "kind-based preprocessing applied to Actor and its Writer history",
            "writer_truth": "latest verified focal preference, including preferred SKU when fallback is ordered",
            "primary_contrast": "conflict minus aligned, untreated, note_first, accepted_wrong_attribute",
            "secondary_contrasts": "source order and simple baselines; stratified attributes; unadjusted",
            "generalization_scope": "new seed83022 instances; same reviewed vocabulary/templates and local tool environment",
            "interpretation": "joint extension, no separate causal attribution to vocabulary vs template changes",
            "source_tags_assumed_reliable": True, "old_test_used": False,
            "head_search_used": False, "error_based_selection": False,
            "frozen_before_generation": True, "config": copy.deepcopy(self.cfg),
            "actor_protocol": "v5.1 all ten legal next-call prefixes and explicit empty-history rule; no output repair",
            "intermediate_outcomes": "secondary endpoints frozen before this run, motivated by v4 post-hoc diagnosis",
            "action_metrics": list(GX_ACTION_METRICS),
            "action_paired_metrics": list(GX_ACTION_PAIRED_METRICS),
            "metric_applicability": {key: list(value) for key, value in GX_METRIC_APPLICABILITY.items()},
            "readiness_rules": copy.deepcopy(GX_READINESS_RULES),
        }
        self.identity = {
            "protocol": copy.deepcopy(self.protocol), "extension_code_hash": stable_hash(source_bundle),
            "live_extension_signature": gx_code_signature(),
            "original_run_fingerprint": namespace["RUN_FINGERPRINT"],
            "original_implementation_hash": namespace["IMPLEMENTATION_HASH"],
            "original_config": copy.deepcopy(namespace["CFG"]),
            "model": copy.deepcopy(namespace["MODEL_IDENTITY"]),
            "backend_config": copy.deepcopy(getattr(self.backend, "cfg", {})),
            "manifest_hashes": {k: stable_hash(v) for k, v in self.cases.items()},
        }
        self.identity_hash = stable_hash(self.identity)
        self.root.mkdir(parents=True, exist_ok=True)
        identity_file = self.root / "identity.json"
        if identity_file.exists():
            stored = json.loads(identity_file.read_text(encoding="utf-8"))
            if stored != self.identity:
                raise RuntimeError(" Extended version / Agreement / The model has changed; please use a new extension directory and not mix the old results. ")
        else:
            atomic_json(identity_file, self.identity)
        for stage, cases in self.cases.items():
            path = self.root / (stage + "-manifest.json")
            if path.exists() and json.loads(path.read_text(encoding="utf-8")) != cases:
                raise RuntimeError(f"Frozen manifest changed: {path}")
            if not path.exists():
                atomic_json(path, cases)
        self.cached_backend = GXCachedBackend(self.backend, self.root / "items" / "generations", self.identity_hash)

    def _ensure_live(self, check_disk=True):
        self.parent["verify_live_run"]()
        if stable_hash(self.identity) != self.identity_hash:
            raise RuntimeError("Frozen extension identity was modified")
        if gx_code_signature() != self.identity["live_extension_signature"]:
            raise RuntimeError("Live extension functions/constants changed after protocol lock")
        if self.parent["backend"] is not self.backend:
            raise RuntimeError("Loaded backend was replaced; recreate extension after verifying the original runtime")
        checks = {"original_run_fingerprint": self.parent["RUN_FINGERPRINT"],
                  "original_implementation_hash": self.parent["IMPLEMENTATION_HASH"],
                  "original_config": self.parent["CFG"], "model": self.parent["MODEL_IDENTITY"],
                  "backend_config": getattr(self.backend, "cfg", {})}
        if any(stable_hash(value) != stable_hash(self.identity[key]) for key, value in checks.items()):
            raise RuntimeError("Original model/runtime/config identity changed")
        if Path(self.parent["RUN_DIR"]) != self.original_dir:
            raise RuntimeError("Original RUN_DIR changed; do not overlap extension with a directory-switching cell")
        if stable_hash(self.cfg) != stable_hash(self.identity["protocol"]["config"]):
            raise RuntimeError("Extension config changed after manifest lock")
        for stage, cases in self.cases.items():
            if stable_hash(cases) != self.identity["manifest_hashes"][stage]:
                raise RuntimeError("In-memory frozen manifest changed")
        if check_disk:
            if json.loads((self.root / "identity.json").read_text(encoding="utf-8")) != self.identity:
                raise RuntimeError("Frozen identity on disk changed")
            for stage, cases in self.cases.items():
                if json.loads((self.root / (stage + "-manifest.json")).read_text(encoding="utf-8")) != cases:
                    raise RuntimeError("Frozen manifest on disk changed")

    def _path(self, case):
        return self.root / "sources" / case["stage"] / (case["source_id"] + ".json")

    def _signature(self, case):
        return stable_hash({"identity": self.identity_hash, "case": case})

    def _read_source(self, case):
        result = gx_read_checked(self._path(case), self._signature(case))
        expected = set(itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS))
        keys = [(r["arm"], r["condition"], r["position"]) for r in result["rows"]]
        if len(keys) != len(expected) or set(keys) != expected:
            raise RuntimeError("Completed source has missing or duplicate paired rows")
        if any(r["source_id"] != case["source_id"] for r in result["rows"]):
            raise RuntimeError("Source checkpoint provenance mismatch")
        return result

    def _run_source(self, case):
        rows, episodes = [], {}
        # Order is fixed before observing outcomes. Identical inputs share inference.
        for arm, condition, position in itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS):
            memory = gx_memory(case, condition, position, arm)
            episode_id = stable_hash({"case": case, "effective_memory": memory})
            if episode_id not in episodes:
                actor = gx_run_actor(self.cached_backend, copy.deepcopy(case), copy.deepcopy(memory), dict(self.cfg))
                history = gx_history(case, memory, actor)
                writer = gx_run_writer(self.cached_backend, copy.deepcopy(case), history, dict(self.cfg))
                episodes[episode_id] = {"memory": memory, "actor": actor, "writer": writer,
                                        "history_hash": stable_hash(history)}
            episode = episodes[episode_id]
            row = {"source_id": case["source_id"], "stage": case["stage"],
                   "attribute": case["attribute"], "template_id": case["template_id"],
                   "stock_available": case["stock_available"],
                   "city": case["gold_fields"]["delivery_city"],
                   "multiword_city": " " in case["gold_fields"]["delivery_city"],
                   "arm": arm, "condition": condition, "position": position,
                   "episode_id": episode_id,
                   **{"actor_" + k: v for k, v in episode["actor"]["metrics"].items()},
                   **{"writer_" + k: v for k, v in episode["writer"]["metrics"].items()}}
            rows.append(row)
        if len(episodes) != 5:
            raise RuntimeError(f"Expected five distinct effective inputs; found {len(episodes)}")
        result = {"source_id": case["source_id"], "rows": rows, "episodes": episodes}
        gx_write_checked(self._path(case), self._signature(case), result)
        return result

    def inspect(self):
        self._ensure_live()
        counts = {}
        for stage, cases in self.cases.items():
            counts[stage] = {"planned_sources": len(cases),
                             "completed_sources": sum(self._path(c).exists() for c in cases),
                             "logical_rows": len(cases) * 12,
                             "unique_actor_episodes": len(cases) * 5,
                             "unique_writer_episodes": len(cases) * 5}
        print(" Continue to reuse current models and tools; expanded directory: ", self.root)
        print(json.dumps(counts, ensure_ascii=False, indent=2))
        print("960  Complete  Actor episode  Not equal to  960  Subgeneration: Models are required for each step of tool deployment, and there are others  Writer。")
        print(" Each batch 24 Independent. source； complete evaluation as 192。smoke Do not pick the wrong question, do not trigger the wrong rate threshold. ")
        return counts

    def audit_token_budget(self):
        """Tokenizer-only legal-path length check; no model call or scored episode.

        Constructed traces here are length fixtures, never written to evaluation
        sources or passed to run_writer. Actual evaluation always uses generated
        actions. The backend still checks every actual prompt without truncation.
        """
        self._ensure_live()
        if not callable(getattr(self.backend, "encode", None)):
            print("No tokenizer encode interface; actual backend remains responsible for context limits.")
            return {"status": "unavailable"}
        configured = int(getattr(self.backend, "cfg", {}).get("max_prompt_tokens", 3072))
        model_context = getattr(getattr(getattr(self.backend, "model", None), "cfg", None), "n_ctx", None)
        limit = min(configured, int(model_context)) if model_context else configured
        maxima = {"actor": 0, "writer": 0}
        for case in self.cases["smoke"] + self.cases["evaluation"]:
            memory = gx_memory(case, "conflict", "note_first", "untreated")
            env = OrderEnvironment(case["initial_env"])
            events = []

            def execute_for_length(action):
                observation = env.step(action)
                events.append({"action": action, "observation": observation})
                return observation

            preferred = case["gold_fields"]["preferred_item"]
            preferred_observation = execute_for_length({"tool": "check_stock", "item": preferred})
            item = preferred
            if preferred_observation["available"] == 0:
                item = case["fallback_item"]
                execute_for_length({"tool": "check_stock", "item": item})
            reservation = execute_for_length({"tool": "reserve_item", "item": item})["reservation"]
            for prefix_length in range(len(events) + 1):
                maxima["actor"] = max(maxima["actor"], len(self.backend.encode(
                    gx_actor_prompt(case, memory, events[:prefix_length]))))
            execute_for_length({"tool": "create_order", "customer": case["gold_fields"]["order_customer"],
                                "city": case["gold_fields"]["delivery_city"], "item": item,
                                "reservation": reservation})
            fixture = {"source_id": case["source_id"], "events": events, "memory": memory}
            maxima["writer"] = max(maxima["writer"], len(self.backend.encode(
                gx_writer_prompt(case, gx_history(case, memory, fixture)))))
        audit = {"status": "tokenizer_only", "max_legal_path_prompt_tokens": maxima,
                 "prompt_limit": limit, "model_context": int(model_context) if model_context else None,
                 "note": "Constructed legal-path fixtures, no inference or evaluation outcome; recovery traces may be longer."}
        atomic_json(self.root / "token-budget-audit.json", audit)
        for role, tokens in maxima.items():
            if tokens > limit or (model_context and tokens + self.cfg[role + "_max_new_tokens"] > model_context):
                raise RuntimeError(f"{role} planned prompt exceeds original context budget: {audit}; shorten extension prompts in a new version.")
        print(" Only code check (no model generation): ", maxima, "； Original prompt Upper limit: ", limit)
        return audit

    def smoke_readiness(self, require=False):
        """Recompute controls from checked checkpoints, never trust a saved pass flag."""
        self._ensure_live()
        complete, rows = self._collect("smoke")
        rules = GX_READINESS_RULES
        required = rules["required_smoke_sources"]
        checks = []
        for arm, condition, position in rules["controls"]:
            chosen = [row for row in rows if
                      (row["arm"], row["condition"], row["position"]) == (arm, condition, position)]
            protocol_failures = sum(bool(row["actor_format_failure"]) for row in chosen)
            complete_city = sum(bool(row["actor_order_attempted"]) and
                                not bool(row["actor_first_order_missing_city"]) for row in chosen)
            task_successes = sum(bool(row["actor_task_success"]) for row in chosen)
            passed = (len(chosen) == required and
                      protocol_failures <= rules["maximum_protocol_failures_per_control"] and
                      complete_city >= rules["required_first_complete_city_per_control"] and
                      task_successes >= rules["minimum_task_successes_per_control"])
            checks.append({"arm": arm, "condition": condition, "position": position,
                           "n_sources": len(chosen), "protocol_failures": protocol_failures,
                           "first_order_with_city": complete_city, "task_successes": task_successes,
                           "passed": passed})
        result = {"identity_hash": self.identity_hash, "rules": copy.deepcopy(rules),
                  "completed_smoke_sources": len(complete), "controls": checks,
                  "passed": len(complete) == required and all(check["passed"] for check in checks),
                  "conflict_errors_used_for_gate": False,
                  "source_result_hashes": {item["source_id"]: stable_hash(item) for item in complete}}
        atomic_json(self.root / "readiness.json", result)
        if require and not result["passed"]:
            archive = self.export(stage="smoke", download=False)
            raise RuntimeError("Smoke  Executability check not adopted; formal assessment not initiated / Go on. "
                               " Please look.  readiness.json  and  smoke  The trajectories, not by changing the conflict error rate, are checked. "
                               " The diagnosis has been saved: " + str(archive))
        return result

    def run(self, stage="evaluation", max_sources=24):
        self._ensure_live()
        if stage not in self.cases:
            raise ValueError("stage must be smoke or evaluation")
        if max_sources is not None and (type(max_sources) is not int or max_sources < 1):
            raise ValueError("max_sources must be positive integer or None (all remaining)")
        if stage == "evaluation":
            self.smoke_readiness(require=True)
        cases = self.cases[stage]
        pending = []
        for case in cases:
            if self._path(case).exists():
                self._read_source(case)
            else:
                pending.append(case)
        todo = pending if max_sources is None else pending[:max_sources]
        completed = len(cases) - len(pending)
        start_calls = self.cached_backend.new_calls
        for case in todo:
            self._ensure_live(check_disk=False)
            self._run_source(case)
            completed += 1
            print(f"[{stage}] {completed}/{len(cases)} sources complete | {case['source_id']} | "
                  f"new model calls this batch={self.cached_backend.new_calls - start_calls}", flush=True)
        report = self.report(stage)
        print(f" Add to this instalment  {len(todo)} sources； Completed  {report['completed_sources']}/{report['planned_sources']}。")
        print(" Not available source Save. Repeat this book. cell Hell run. max_sources=None Will complete all remaining tasks. ")
        if stage == "smoke":
            readiness = self.smoke_readiness(require=True)
            print("Smoke  Executability checks passed; these checks do not require any conflict-induced error. ", readiness["controls"])
        return report

    def _collect(self, stage):
        if stage not in self.cases:
            raise ValueError("Unknown stage")
        complete = []
        for case in self.cases[stage]:
            if self._path(case).exists():
                complete.append(self._read_source(case))
        return complete, [row for result in complete for row in result["rows"]]

    def report(self, stage="evaluation"):
        self._ensure_live()
        complete, rows = self._collect(stage)
        metrics = GX_PRIMARY_METRICS + GX_ACTION_METRICS + ("actor_strict_task_success", "actor_accepted_conflict_value",
                   "actor_format_failure", "actor_tool_error", "actor_no_accepted_order",
                   "writer_target_correct", "writer_target_source_correct", "writer_format_failure",
                   "writer_whole_memory_correct", "writer_omission")
        summaries = []
        strata = [("all", None)] + [("attribute", a) for a in GX_ATTRIBUTES]
        strata += [("stock_available", v) for v in (0, 1)]
        strata += [("template_id", v) for v in sorted({c["template_id"] for c in self.cases[stage]})]
        strata += [("city", v) for v in sorted({c["gold_fields"]["delivery_city"] for c in self.cases[stage]})]
        strata += [("multiword_city", v) for v in (False, True)]
        for stratum, value in strata:
            selected = rows if stratum == "all" else [r for r in rows if r[stratum] == value]
            for arm, condition, position in itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS):
                group = [r for r in selected if (r["arm"], r["condition"], r["position"]) == (arm, condition, position)]
                if not group:
                    continue
                if len({r["source_id"] for r in group}) != len(group):
                    raise RuntimeError("Summary double-counts a source")
                for metric in metrics:
                    if metric not in group[0]:
                        continue
                    eligible = [r for r in group if metric not in GX_METRIC_APPLICABILITY
                                or r["attribute"] in GX_METRIC_APPLICABILITY[metric]]
                    if not eligible:
                        continue
                    if any(r.get(metric) not in (True, False, 0, 1) for r in eligible):
                        raise ValueError(f"Missing/nonbinary primary metric {metric}")
                    count, n = sum(int(r[metric]) for r in eligible), len(eligible)
                    low, high = gx_wilson(count, n)
                    summaries.append({"stratum": stratum, "stratum_value": value,
                                      "arm": arm, "condition": condition, "position": position,
                                      "metric": metric, "count": count, "n_sources": n,
                                      "eligible_attributes": list(GX_METRIC_APPLICABILITY.get(metric, GX_ATTRIBUTES)),
                                      "outcome_scope": "intermediate secondary" if metric in GX_ACTION_METRICS else "original endpoint",
                                      "rate": count / n, "wilson_low": low, "wilson_high": high})
        contrasts = []
        definitions = []
        for arm, position in itertools.product(GX_ARMS, GX_POSITIONS):
            definitions.append(("conflict_minus_aligned", (arm, "aligned", position), (arm, "conflict", position)))
        for arm, condition, position in itertools.product(GX_ARMS[1:], GX_CONDITIONS, GX_POSITIONS):
            definitions.append(("baseline_minus_untreated", ("untreated", condition, position), (arm, condition, position)))
        for condition in GX_CONDITIONS:
            definitions.append(("note_first_minus_profile_first", ("untreated", condition, "profile_first"),
                                ("untreated", condition, "note_first")))
        for attribute in ("all",) + tuple(GX_ATTRIBUTES):
            selected = rows if attribute == "all" else [r for r in rows if r["attribute"] == attribute]
            for label, before_key, after_key in definitions:
                before_rows = [r for r in selected if (r["arm"], r["condition"], r["position"]) == before_key]
                after_rows = [r for r in selected if (r["arm"], r["condition"], r["position"]) == after_key]
                if not before_rows and not after_rows:
                    continue
                for metric in GX_PRIMARY_METRICS + GX_ACTION_PAIRED_METRICS:
                    eligible_before = [r for r in before_rows if metric not in GX_METRIC_APPLICABILITY
                                       or r["attribute"] in GX_METRIC_APPLICABILITY[metric]]
                    eligible_after = [r for r in after_rows if metric not in GX_METRIC_APPLICABILITY
                                      or r["attribute"] in GX_METRIC_APPLICABILITY[metric]]
                    if not eligible_before and not eligible_after:
                        continue
                    before = {r["source_id"]: r[metric] for r in eligible_before}
                    after = {r["source_id"]: r[metric] for r in eligible_after}
                    if len(before) != len(eligible_before) or len(after) != len(eligible_after):
                        raise RuntimeError("Duplicate source in paired contrast")
                    stats = gx_pair_stats(before, after, reps=self.cfg["bootstrap_reps"])
                    contrasts.append({"attribute": attribute, "contrast": label,
                                      "before": list(before_key), "after": list(after_key),
                                      "metric": metric,
                                      "eligible_attributes": list(GX_METRIC_APPLICABILITY.get(metric, GX_ATTRIBUTES)),
                                      "analysis_role": "primary" if (
                                          attribute == "all" and metric == "actor_accepted_wrong_attribute"
                                          and label == "conflict_minus_aligned"
                                          and before_key == ("untreated", "aligned", "note_first")) else "secondary",
                                      **stats})
        failures = []
        for result in complete:
            for episode_id, episode in result["episodes"].items():
                actor, writer = episode["actor"]["metrics"], episode["writer"]["metrics"]
                if (not actor["task_success"] or not writer["target_joint_correct"]
                        or actor["attempted_wrong_reservation_item"] or actor["accepted_wrong_reservation_item"]
                        or actor["any_order_wrong_attribute"]
                        or actor["any_order_missing_city"]):
                    failures.append({"source_id": result["source_id"], "episode_id": episode_id,
                                     "logical_arms": [{k: r[k] for k in ("arm", "condition", "position")}
                                                      for r in result["rows"] if r["episode_id"] == episode_id],
                                     **episode})
        report = {"stage": stage, "identity_hash": self.identity_hash,
                  "planned_sources": len(self.cases[stage]), "completed_sources": len(complete),
                  "complete": len(complete) == len(self.cases[stage]), "logical_rows": len(rows),
                  "unique_actor_episodes": sum(len(r["episodes"]) for r in complete),
                  "unique_writer_episodes": sum(len(r["episodes"]) for r in complete),
                  "generation_records_whole_extension_including_smoke_and_partial":
                      sum(1 for _ in (self.root / "items" / "generations").glob("*.json")),
                  "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                  "analysis_status": "complete fixed manifest" if len(complete) == len(self.cases[stage]) else "PARTIAL; not full evaluation",
                  "inference_note": "Finite designed task suite; intervals reflect source resampling, not all real-world agents. Secondary p values unadjusted.",
                  "action_metric_note": "Attempts may fail tool validation; accepted reservations require observation.ok. Missing values are not chosen wrong values. Correct fallback is scored against initial stock. Each rate counts a source once, never individual repeated attempts.",
                  "summary": summaries, "paired_effects": contrasts}
        destination = self.root / "reports" / stage
        atomic_json(destination / "report.json", report)
        atomic_json(destination / "failures.json", failures)
        gx_csv(destination / "rows.csv", rows)
        gx_csv(destination / "summary.csv", summaries)
        gx_csv(destination / "paired-effects.csv", contrasts)
        # Small on-screen summary; all attribute/template/stock/city strata stay in CSV.
        print(f"{stage}: {len(complete)}/{len(self.cases[stage])} sources; "
              f"{len(rows)} logical rows; {report['unique_actor_episodes']} distinct Actor episodes")
        for row in summaries:
            visible_metrics = GX_PRIMARY_METRICS[:3] + (
                "actor_accepted_wrong_reservation_item", "actor_accepted_conflict_reservation_item",
                "actor_first_order_wrong_attribute", "actor_first_order_conflict_value",
                "actor_any_order_missing_city", "actor_format_failure")
            if row["stratum"] == "all" and row["condition"] == "conflict" and row["position"] == "note_first" and row["metric"] in visible_metrics:
                print(f"  {row['arm']:16s} {row['metric']:35s} {row['count']}/{row['n_sources']} ({row['rate']:.1%})")
        return report

    def export(self, stage="evaluation", download=False):
        self.report(stage)
        destination = self.root.parent / (self.root.name + "_" + stage + ".zip")
        files = [self.root / "identity.json", self.root / (stage + "-manifest.json")]
        if (self.root / "token-budget-audit.json").exists():
            files.append(self.root / "token-budget-audit.json")
        if (self.root / "readiness.json").exists():
            files.append(self.root / "readiness.json")
        files += sorted((self.root / "reports" / stage).glob("*"))
        files += sorted((self.root / "sources" / stage).glob("*.json"))
        with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in files:
                if path.is_file():
                    archive.write(path, path.relative_to(self.root).as_posix())
        print(" Results package: ", destination)
        if download:
            from google.colab import files as gx_colab_files
            gx_colab_files.download(str(destination))
        return str(destination)
'''
V5_GX_NAMESPACE = {'__name__': 'gx_protocol_actions_v5p1'}
for v5_helper in ('OrderEnvironment', 'parse_json_output', 'canonical_json', 'stable_hash', 'atomic_json'):
    V5_GX_NAMESPACE[v5_helper] = V5_RUNTIME_NAMESPACE[v5_helper]
exec(compile(V5_GX_SOURCE, '<gx-protocol-actions-v5p1>', 'exec'), V5_GX_NAMESPACE)
V5_GX_NAMESPACE['v4_backend_contract'] = V5_RUNTIME_NAMESPACE['v4_backend_contract']


## 05 ·  Prepared data lists, previews and  CPU  Inspection 
 Data embedded; checking all before any model is loaded or the answer is generated  source、 Matches with tool scores.  case  Not the whole model. 


In [ ]:
V5_PREPARED_DATA = json.loads(r'''{"evaluation":[{"alternative_item":"sku-f3ea392cb2f54aba","attribute":"delivery_city","conflict_value":"Osaka","fallback_item":"sku-a16fb5b732826fcd","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-7aa16cf7ad61bd48","preferred_item":"sku-b8ded62621833a9f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-a16fb5b732826fcd":2,"sku-b8ded62621833a9f":1,"sku-f3ea392cb2f54aba":2}},"note_event_id":"evt-b44abfaf30d14893","profile_event_id":"evt-4e10d159a167dfbb","replicate":0,"request_id":"req-294d8ac1642321fd","source_id":"gx-evaluation-1f235ade3f8cf942","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-b644a5c94910d6e3","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-57ab69d5faa68838","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-8369dc57feff4c3c","preferred_item":"sku-3246adb579295518"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3246adb579295518":0,"sku-57ab69d5faa68838":2,"sku-b644a5c94910d6e3":2}},"note_event_id":"evt-00053ed9e2988167","profile_event_id":"evt-2c2a799b9d16755d","replicate":0,"request_id":"req-bb38ac155fee0f18","source_id":"gx-evaluation-72efc80a54248a3a","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-a6b45993e7dbe336","attribute":"order_customer","conflict_value":"acct-98c2252cc336002e","fallback_item":"sku-119929b6d3d1e73f","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-fe6fa43ff6561b19","preferred_item":"sku-7b94b1e83378c86f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-119929b6d3d1e73f":2,"sku-7b94b1e83378c86f":1,"sku-a6b45993e7dbe336":2}},"note_event_id":"evt-3d5dd10934b51d2e","profile_event_id":"evt-f3f957f3b85bf97c","replicate":0,"request_id":"req-c864a8ad2884971b","source_id":"gx-evaluation-7bd836b98e8684f6","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-9251427690b0949f","attribute":"order_customer","conflict_value":"acct-6915655e45d861d3","fallback_item":"sku-69f82627863ebfa2","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-efa83f3d7c131f65","preferred_item":"sku-215b6ae147593e98"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-215b6ae147593e98":0,"sku-69f82627863ebfa2":2,"sku-9251427690b0949f":2}},"note_event_id":"evt-526e93beadf70893","profile_event_id":"evt-0d432dda0c4ed8cd","replicate":0,"request_id":"req-2475abcf895197c0","source_id":"gx-evaluation-408f6f5ec0bbfd19","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-ca4d58bdf572be5e","attribute":"preferred_item","conflict_value":"sku-ca4d58bdf572be5e","fallback_item":"sku-7c41e640f16a2b42","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-bf1a3200490170d4","preferred_item":"sku-d1d03efe1fea4945"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7c41e640f16a2b42":2,"sku-ca4d58bdf572be5e":2,"sku-d1d03efe1fea4945":1}},"note_event_id":"evt-76cfcbf6f62866c4","profile_event_id":"evt-efbce285db42acb0","replicate":0,"request_id":"req-41a70bd06df7225b","source_id":"gx-evaluation-2b19bd076bd82ee9","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-35bdd8e2ec7878c2","attribute":"preferred_item","conflict_value":"sku-35bdd8e2ec7878c2","fallback_item":"sku-402880fa100734e6","gold_fields":{"delivery_city":"New York","order_customer":"acct-7ed842d2a1a25c15","preferred_item":"sku-d9d4ada9cb1ddec7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-35bdd8e2ec7878c2":2,"sku-402880fa100734e6":2,"sku-d9d4ada9cb1ddec7":0}},"note_event_id":"evt-c37f9d69e6fdd928","profile_event_id":"evt-fee8938d63ea9b60","replicate":0,"request_id":"req-0eddc47cb3114518","source_id":"gx-evaluation-e9ed02eeae11a186","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-529df92c9c378884","attribute":"delivery_city","conflict_value":"Buenos Aires","fallback_item":"sku-97db85be10e5b691","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-97ed8a53cf185162","preferred_item":"sku-27409bc316f9b73e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-27409bc316f9b73e":1,"sku-529df92c9c378884":2,"sku-97db85be10e5b691":2}},"note_event_id":"evt-039f82fa60090d2d","profile_event_id":"evt-78ed83f5113b06a0","replicate":0,"request_id":"req-6287e68c0ea38268","source_id":"gx-evaluation-20e7d435f8248a22","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-96147de37e8bde67","attribute":"delivery_city","conflict_value":"Lisbon","fallback_item":"sku-be0d1a44a1dd25c5","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-9e88451c56fc7620","preferred_item":"sku-1e7d87224e0c948d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1e7d87224e0c948d":0,"sku-96147de37e8bde67":2,"sku-be0d1a44a1dd25c5":2}},"note_event_id":"evt-650772d74ad2718a","profile_event_id":"evt-6743258b4699346b","replicate":0,"request_id":"req-72ed8a7bbe466bd9","source_id":"gx-evaluation-a75edd3d5e270cb3","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-87501afe3cd62014","attribute":"order_customer","conflict_value":"acct-09cb8c5693ecc703","fallback_item":"sku-192915bc615b6dd3","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-e1efcb46d9ba5c7d","preferred_item":"sku-7ef93514f0cee325"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-192915bc615b6dd3":2,"sku-7ef93514f0cee325":1,"sku-87501afe3cd62014":2}},"note_event_id":"evt-eda7f6febf7abc4a","profile_event_id":"evt-9059fc94ac575a98","replicate":0,"request_id":"req-bde29d8143369a35","source_id":"gx-evaluation-387fe276f5a56601","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-ccddbf9e1219cc79","attribute":"order_customer","conflict_value":"acct-108801871b669cd0","fallback_item":"sku-bdc00246622e1af6","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-35b5838218981234","preferred_item":"sku-e10094307ad58bdf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-bdc00246622e1af6":2,"sku-ccddbf9e1219cc79":2,"sku-e10094307ad58bdf":0}},"note_event_id":"evt-72b0bec9bddb778c","profile_event_id":"evt-728afebd93886966","replicate":0,"request_id":"req-7d2c0473b4f0ad30","source_id":"gx-evaluation-48ebcb22675db83c","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-5bbe850ab85c8d10","attribute":"preferred_item","conflict_value":"sku-5bbe850ab85c8d10","fallback_item":"sku-03d7328ba9861a25","gold_fields":{"delivery_city":"Prague","order_customer":"acct-be503593b7a81024","preferred_item":"sku-f9c4c4d0a761f0a7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-03d7328ba9861a25":2,"sku-5bbe850ab85c8d10":2,"sku-f9c4c4d0a761f0a7":1}},"note_event_id":"evt-b92efd3f3d09c850","profile_event_id":"evt-46a885c173c11a46","replicate":0,"request_id":"req-686ac0a3fcf770c9","source_id":"gx-evaluation-98db728b973ffdd1","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-806eae18448f14ff","attribute":"preferred_item","conflict_value":"sku-806eae18448f14ff","fallback_item":"sku-befa14f982f17bca","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-9b74271763bd9eca","preferred_item":"sku-5746f2c22c30a301"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-5746f2c22c30a301":0,"sku-806eae18448f14ff":2,"sku-befa14f982f17bca":2}},"note_event_id":"evt-d966046e58b8dd38","profile_event_id":"evt-2c20dc1c187c48f9","replicate":0,"request_id":"req-3ca037bc0a2326d1","source_id":"gx-evaluation-da6ba9d87327272d","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-9310d68cde621d1e","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-da26db80586e7062","gold_fields":{"delivery_city":"New York","order_customer":"acct-f1fbb78d6f003769","preferred_item":"sku-6a9881165a10dcf3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6a9881165a10dcf3":1,"sku-9310d68cde621d1e":2,"sku-da26db80586e7062":2}},"note_event_id":"evt-d1eff6dca76a6645","profile_event_id":"evt-367ce53c59460915","replicate":0,"request_id":"req-b24a53211d6f51c3","source_id":"gx-evaluation-6215025b9d85c7d7","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-fc282418ff6b1aa7","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-4b63c43c43464ff8","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-54c9ea79d266f38d","preferred_item":"sku-3af0fb6d7a46bd32"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3af0fb6d7a46bd32":0,"sku-4b63c43c43464ff8":2,"sku-fc282418ff6b1aa7":2}},"note_event_id":"evt-a1c27189508a5923","profile_event_id":"evt-1795894c38754e52","replicate":0,"request_id":"req-d888c4fc7ce1858f","source_id":"gx-evaluation-ec3dc8a6c793c029","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-dc67c19bbcf53b55","attribute":"order_customer","conflict_value":"acct-07af665c1d8681b5","fallback_item":"sku-73ebf0456fde10f6","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-3c0ca6b5b4b79e04","preferred_item":"sku-a7d3844b3b7fc237"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-73ebf0456fde10f6":2,"sku-a7d3844b3b7fc237":1,"sku-dc67c19bbcf53b55":2}},"note_event_id":"evt-33fa3e090f3bba2e","profile_event_id":"evt-c978f8a607800d65","replicate":0,"request_id":"req-3deb59da76fc34fb","source_id":"gx-evaluation-659c78ab4b4cc7f6","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-b31b03d70ed6b991","attribute":"order_customer","conflict_value":"acct-7d3ac2be07825ba4","fallback_item":"sku-215ccd81f62faeba","gold_fields":{"delivery_city":"New York","order_customer":"acct-4983fffbaf09fd61","preferred_item":"sku-7790a25cb227f20d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-215ccd81f62faeba":2,"sku-7790a25cb227f20d":0,"sku-b31b03d70ed6b991":2}},"note_event_id":"evt-12da3072a8dec4fd","profile_event_id":"evt-03dd296b7a1c7663","replicate":0,"request_id":"req-db5155b4b37c5a41","source_id":"gx-evaluation-2b71e9f3e3dc10d0","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-bc165ffe90912d40","attribute":"preferred_item","conflict_value":"sku-bc165ffe90912d40","fallback_item":"sku-1b607435b1d0dc61","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-97b76bd4b247bbc6","preferred_item":"sku-292468e00543dbe3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1b607435b1d0dc61":2,"sku-292468e00543dbe3":1,"sku-bc165ffe90912d40":2}},"note_event_id":"evt-aca5a50011312a33","profile_event_id":"evt-54177b1f6b5b5284","replicate":0,"request_id":"req-16cda756f9ac4d7b","source_id":"gx-evaluation-4162f25da7561691","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-6e3fd1aea0fb351d","attribute":"preferred_item","conflict_value":"sku-6e3fd1aea0fb351d","fallback_item":"sku-62088fced4d4c9c6","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-0d536a8349d05bb1","preferred_item":"sku-645c177b2a8be2fe"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-62088fced4d4c9c6":2,"sku-645c177b2a8be2fe":0,"sku-6e3fd1aea0fb351d":2}},"note_event_id":"evt-03537f4646e53db4","profile_event_id":"evt-e0dd565e14e1e9a1","replicate":0,"request_id":"req-a05b6dc53470a2e4","source_id":"gx-evaluation-98286381fadc6aa3","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-f9c2ce2f683abf3d","attribute":"delivery_city","conflict_value":"Osaka","fallback_item":"sku-c49f61cdf662aa89","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-aaeff0cf7245c64b","preferred_item":"sku-f452a7a9430409bf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-c49f61cdf662aa89":2,"sku-f452a7a9430409bf":1,"sku-f9c2ce2f683abf3d":2}},"note_event_id":"evt-090b40a7f808a42e","profile_event_id":"evt-fde507f241def636","replicate":0,"request_id":"req-69a4eec9f28e014b","source_id":"gx-evaluation-b237dd75bf4d29e0","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-2a9475d4e7676393","attribute":"delivery_city","conflict_value":"Helsinki","fallback_item":"sku-60206b2438d35875","gold_fields":{"delivery_city":"Prague","order_customer":"acct-7eeaa386b1633cdc","preferred_item":"sku-0c5771ea42dd6576"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0c5771ea42dd6576":0,"sku-2a9475d4e7676393":2,"sku-60206b2438d35875":2}},"note_event_id":"evt-efa0694b8acc2ee3","profile_event_id":"evt-4db9bc515bbc8bf2","replicate":0,"request_id":"req-f83830d076dce2cb","source_id":"gx-evaluation-7ba2c9d7184959b0","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-ac72898776105f1e","attribute":"order_customer","conflict_value":"acct-3abc800cabd68678","fallback_item":"sku-a12a5a835c453860","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-e96c817b744470ef","preferred_item":"sku-0ebb109421c2a44b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0ebb109421c2a44b":1,"sku-a12a5a835c453860":2,"sku-ac72898776105f1e":2}},"note_event_id":"evt-445b305edf15a1b3","profile_event_id":"evt-26aad020a3c8d07a","replicate":0,"request_id":"req-78018343db9e60bc","source_id":"gx-evaluation-d1643a088b30cc60","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-402ce9a8c4e224d8","attribute":"order_customer","conflict_value":"acct-52a73ba901f869e9","fallback_item":"sku-a3d0ac6d69247fe0","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-cb36710e5cb6322a","preferred_item":"sku-fdbed78db60bc58d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-402ce9a8c4e224d8":2,"sku-a3d0ac6d69247fe0":2,"sku-fdbed78db60bc58d":0}},"note_event_id":"evt-7076c8a05ce46bd5","profile_event_id":"evt-0f178ea82133d630","replicate":0,"request_id":"req-3babe7a5a2973833","source_id":"gx-evaluation-1ecba942f0532030","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-12504500d3aac9ba","attribute":"preferred_item","conflict_value":"sku-12504500d3aac9ba","fallback_item":"sku-5d2321879ac56e15","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-f46c2dc6ddcc7616","preferred_item":"sku-234dbf2e2bade65d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-12504500d3aac9ba":2,"sku-234dbf2e2bade65d":1,"sku-5d2321879ac56e15":2}},"note_event_id":"evt-ddc7d35e8b4c3166","profile_event_id":"evt-0be6e3d91294dfaf","replicate":0,"request_id":"req-a210e0150d71eec6","source_id":"gx-evaluation-085928267401161b","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-c9cb45d1ca5138b9","attribute":"preferred_item","conflict_value":"sku-c9cb45d1ca5138b9","fallback_item":"sku-b1eb2363ad684990","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-7b47e6f397f86d15","preferred_item":"sku-277ec9345c3f1211"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-277ec9345c3f1211":0,"sku-b1eb2363ad684990":2,"sku-c9cb45d1ca5138b9":2}},"note_event_id":"evt-69a5d3e35a8e7b09","profile_event_id":"evt-098be2ccba45e4e0","replicate":0,"request_id":"req-9d574b49644d8772","source_id":"gx-evaluation-01162005ba147895","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-c63bebb0d6499df1","attribute":"delivery_city","conflict_value":"New York","fallback_item":"sku-1e1ff91d807f4912","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-44a08525484cf52f","preferred_item":"sku-9ca681286d966834"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1e1ff91d807f4912":2,"sku-9ca681286d966834":1,"sku-c63bebb0d6499df1":2}},"note_event_id":"evt-302a2300d3300116","profile_event_id":"evt-2c1fd1cd070ceb97","replicate":1,"request_id":"req-7d236ba41ebd89b3","source_id":"gx-evaluation-b78329d6cbc7bc6b","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-93e08f89be99b3f4","attribute":"delivery_city","conflict_value":"Osaka","fallback_item":"sku-adab571ef80c4432","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-e7a5a5e2f350380b","preferred_item":"sku-91fd0de4454c6a3b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-91fd0de4454c6a3b":0,"sku-93e08f89be99b3f4":2,"sku-adab571ef80c4432":2}},"note_event_id":"evt-15e905d7eb2399a7","profile_event_id":"evt-534ebc6c11f9bd16","replicate":1,"request_id":"req-9581191071beff1b","source_id":"gx-evaluation-b2d166d1637c8ef2","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-8fb306da624dc944","attribute":"order_customer","conflict_value":"acct-13928ac13a06e34a","fallback_item":"sku-0b6f4ebe96ad44fc","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-12983d19db4aa9f0","preferred_item":"sku-0735d99eb41c7945"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0735d99eb41c7945":1,"sku-0b6f4ebe96ad44fc":2,"sku-8fb306da624dc944":2}},"note_event_id":"evt-472318d27e8ade1b","profile_event_id":"evt-49c91a0cf16028f7","replicate":1,"request_id":"req-3dbab948a90e27b6","source_id":"gx-evaluation-1550c8f0123c247a","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-f7f4eb5b41be1355","attribute":"order_customer","conflict_value":"acct-37e13a0c9a33a70b","fallback_item":"sku-979be013b33ff461","gold_fields":{"delivery_city":"Prague","order_customer":"acct-c8b72987dd391fc2","preferred_item":"sku-d8d5940dc6862300"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-979be013b33ff461":2,"sku-d8d5940dc6862300":0,"sku-f7f4eb5b41be1355":2}},"note_event_id":"evt-78e0018019a6b110","profile_event_id":"evt-22d5ebe776c22278","replicate":1,"request_id":"req-720b0ab13723a100","source_id":"gx-evaluation-c74724ad8b381874","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-473a8a9fbdf1ed8e","attribute":"preferred_item","conflict_value":"sku-473a8a9fbdf1ed8e","fallback_item":"sku-63b0a77d73ac62f9","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-5371dcdfc37d63c6","preferred_item":"sku-75511ddb4455d2aa"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-473a8a9fbdf1ed8e":2,"sku-63b0a77d73ac62f9":2,"sku-75511ddb4455d2aa":1}},"note_event_id":"evt-5e8c26d07d859db4","profile_event_id":"evt-39a1e0dd886fb0a6","replicate":1,"request_id":"req-e1cd03607965567f","source_id":"gx-evaluation-c180acae9767a53a","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-12530c6dd2de8a79","attribute":"preferred_item","conflict_value":"sku-12530c6dd2de8a79","fallback_item":"sku-a16e248d40b6610b","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-4256b5c05c5f033a","preferred_item":"sku-16aeb2312d1b45d8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-12530c6dd2de8a79":2,"sku-16aeb2312d1b45d8":0,"sku-a16e248d40b6610b":2}},"note_event_id":"evt-5cb495fcc273140e","profile_event_id":"evt-8cfc3a94d578e6f5","replicate":1,"request_id":"req-3b7102496a0dabc5","source_id":"gx-evaluation-1657ae171d8d1da4","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-d9745dc4e743b245","attribute":"delivery_city","conflict_value":"Mumbai","fallback_item":"sku-e67f92f56f49ce82","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-02333bb2f4562331","preferred_item":"sku-f6eb0ab587688e2d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-d9745dc4e743b245":2,"sku-e67f92f56f49ce82":2,"sku-f6eb0ab587688e2d":1}},"note_event_id":"evt-f686a2c4505495c6","profile_event_id":"evt-48b662291ec98ce0","replicate":1,"request_id":"req-9a65699130d4de9f","source_id":"gx-evaluation-49ffa7c26e585f7d","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-ee38ee4d6bac1864","attribute":"delivery_city","conflict_value":"Cairo","fallback_item":"sku-a8154dc67eab234a","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-c2685f22e17c9118","preferred_item":"sku-fb6bfccf0a9197d2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-a8154dc67eab234a":2,"sku-ee38ee4d6bac1864":2,"sku-fb6bfccf0a9197d2":0}},"note_event_id":"evt-7c580f5a0e9db7b0","profile_event_id":"evt-e4a8996101897b35","replicate":1,"request_id":"req-f04f62a4c20d10ef","source_id":"gx-evaluation-33072c464bdb61be","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-b95c07ecd4ab1c4b","attribute":"order_customer","conflict_value":"acct-2909391e026342b3","fallback_item":"sku-26e77d274e8da645","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-821123d62bdf17c0","preferred_item":"sku-7739d380fba31bd2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-26e77d274e8da645":2,"sku-7739d380fba31bd2":1,"sku-b95c07ecd4ab1c4b":2}},"note_event_id":"evt-58df966494b93a83","profile_event_id":"evt-28164884241d8eec","replicate":1,"request_id":"req-5601b3a96c716073","source_id":"gx-evaluation-eb9c44ea71ff43d0","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-db39c46e3c8f6619","attribute":"order_customer","conflict_value":"acct-57eb311a7ebc7557","fallback_item":"sku-ad0039c9666e990f","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-b3574c8c56f12b71","preferred_item":"sku-4f4fba384391e202"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4f4fba384391e202":0,"sku-ad0039c9666e990f":2,"sku-db39c46e3c8f6619":2}},"note_event_id":"evt-e5bdd31d3a513279","profile_event_id":"evt-0b78cbc0c433baea","replicate":1,"request_id":"req-d46d3250acc09f1e","source_id":"gx-evaluation-d107e7a9939a2939","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-a554f4b48bd39242","attribute":"preferred_item","conflict_value":"sku-a554f4b48bd39242","fallback_item":"sku-d731d95c4b4295a6","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-f69a1cc294cb00af","preferred_item":"sku-a65b9b4e980557db"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-a554f4b48bd39242":2,"sku-a65b9b4e980557db":1,"sku-d731d95c4b4295a6":2}},"note_event_id":"evt-63c9ca02e7fa9962","profile_event_id":"evt-699314f4a3327b8b","replicate":1,"request_id":"req-722b4e9e471842ba","source_id":"gx-evaluation-65610fd7c009c4fd","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-a3ba1ae359630b9a","attribute":"preferred_item","conflict_value":"sku-a3ba1ae359630b9a","fallback_item":"sku-0198c564b8255462","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-9795559fbd520c9d","preferred_item":"sku-684020ed4a27e15e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0198c564b8255462":2,"sku-684020ed4a27e15e":0,"sku-a3ba1ae359630b9a":2}},"note_event_id":"evt-21e40c7fb3ee2f72","profile_event_id":"evt-ac61c211728b0528","replicate":1,"request_id":"req-0e9cd7995207c48c","source_id":"gx-evaluation-712b04dc416ce49a","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-288aa8597335058d","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-8d8afecfe92fe6a4","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-9233f0fe5d825438","preferred_item":"sku-6023ae82e7f11de8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-288aa8597335058d":2,"sku-6023ae82e7f11de8":1,"sku-8d8afecfe92fe6a4":2}},"note_event_id":"evt-272e117ff92f16f3","profile_event_id":"evt-129453ac28f3814f","replicate":1,"request_id":"req-1b9203ed1410f486","source_id":"gx-evaluation-0dafe150c4434cd5","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-b72ca21a8cbc924d","attribute":"delivery_city","conflict_value":"New York","fallback_item":"sku-35fcea690a1943a8","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-6bdd14f271422dbb","preferred_item":"sku-a9f6dca0a5ec7c84"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-35fcea690a1943a8":2,"sku-a9f6dca0a5ec7c84":0,"sku-b72ca21a8cbc924d":2}},"note_event_id":"evt-648249b4acd84f92","profile_event_id":"evt-63a026bef1425762","replicate":1,"request_id":"req-8052413734fb4769","source_id":"gx-evaluation-e36f6f764bf6ad1d","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-c26528cbfbf3d74a","attribute":"order_customer","conflict_value":"acct-3c30d43bda75a02f","fallback_item":"sku-f61226e956f4d24f","gold_fields":{"delivery_city":"New York","order_customer":"acct-ea554072e59bc5a1","preferred_item":"sku-aa8de954563851ab"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-aa8de954563851ab":1,"sku-c26528cbfbf3d74a":2,"sku-f61226e956f4d24f":2}},"note_event_id":"evt-8f8815ea745cb07f","profile_event_id":"evt-d9ba0423824b9b32","replicate":1,"request_id":"req-185e6c6e84492375","source_id":"gx-evaluation-0cc69efe6e8538c3","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-86bf4a511d6290d0","attribute":"order_customer","conflict_value":"acct-a31368d344ac4c8e","fallback_item":"sku-2d43f2e2dd0b7006","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-c508226584952cfc","preferred_item":"sku-a428e021ada0970c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2d43f2e2dd0b7006":2,"sku-86bf4a511d6290d0":2,"sku-a428e021ada0970c":0}},"note_event_id":"evt-7067d974b653e887","profile_event_id":"evt-cce528456f90cc5b","replicate":1,"request_id":"req-3126451851e26eaf","source_id":"gx-evaluation-d32da947b1ae9e61","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-736019608a364036","attribute":"preferred_item","conflict_value":"sku-736019608a364036","fallback_item":"sku-c3b3ce69653ff613","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-0632fc083a703a11","preferred_item":"sku-37665d37d40dcf3e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-37665d37d40dcf3e":1,"sku-736019608a364036":2,"sku-c3b3ce69653ff613":2}},"note_event_id":"evt-c1c65c6faeff7809","profile_event_id":"evt-d00032ceae41f077","replicate":1,"request_id":"req-971149b7b8925dea","source_id":"gx-evaluation-f12253d9316bfc53","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-22f805b70e29c556","attribute":"preferred_item","conflict_value":"sku-22f805b70e29c556","fallback_item":"sku-ccc522d04ab5d4b3","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-46ef0eebbf3302db","preferred_item":"sku-244665b028d3c2dc"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-22f805b70e29c556":2,"sku-244665b028d3c2dc":0,"sku-ccc522d04ab5d4b3":2}},"note_event_id":"evt-0f865011ea30e942","profile_event_id":"evt-2e61e97e74784766","replicate":1,"request_id":"req-ddadda4c5c3525e6","source_id":"gx-evaluation-5eac1a96ce8f63d3","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-7e682e9a8b172288","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-8428ee3670a15ac7","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-f49d688345bed443","preferred_item":"sku-f9d8281f617f26a4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7e682e9a8b172288":2,"sku-8428ee3670a15ac7":2,"sku-f9d8281f617f26a4":1}},"note_event_id":"evt-6f0de98845a32dd5","profile_event_id":"evt-6b201bab2d11fbd8","replicate":1,"request_id":"req-fd293de9dc8d96a8","source_id":"gx-evaluation-e0de7f61b9386aa7","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-8dcda7d7480cd067","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-87ca16e879c39ff4","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-4f30af4e5c51d3e9","preferred_item":"sku-45939988bbcf69de"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-45939988bbcf69de":0,"sku-87ca16e879c39ff4":2,"sku-8dcda7d7480cd067":2}},"note_event_id":"evt-465abc2507636bc4","profile_event_id":"evt-13adc404a90974f3","replicate":1,"request_id":"req-beb749cf364e1c00","source_id":"gx-evaluation-d62dc53e2e0df642","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-3f4152856f0f8932","attribute":"order_customer","conflict_value":"acct-a90749e9c8a9ca22","fallback_item":"sku-4fcff8fc7a2049af","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-32719fdf712537db","preferred_item":"sku-c399effd4db0a96d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3f4152856f0f8932":2,"sku-4fcff8fc7a2049af":2,"sku-c399effd4db0a96d":1}},"note_event_id":"evt-a0f5394c8eac23c9","profile_event_id":"evt-c60b4f4c2a784afe","replicate":1,"request_id":"req-cac19aaf7e4a704c","source_id":"gx-evaluation-1d63de958fd87d71","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-1f254ecba8a2b927","attribute":"order_customer","conflict_value":"acct-b2545aa4c73cb1d2","fallback_item":"sku-a4f3deb1d9111262","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-ed918bf4845a700d","preferred_item":"sku-31b70cfebb1721f3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1f254ecba8a2b927":2,"sku-31b70cfebb1721f3":0,"sku-a4f3deb1d9111262":2}},"note_event_id":"evt-c148d9827de74929","profile_event_id":"evt-2563017755125e64","replicate":1,"request_id":"req-3dfbe997ad9bb9cc","source_id":"gx-evaluation-859d6f7e7e7cd5db","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-35943b236958e24b","attribute":"preferred_item","conflict_value":"sku-35943b236958e24b","fallback_item":"sku-dfc4f1a92654f7a5","gold_fields":{"delivery_city":"New York","order_customer":"acct-d81b96fbb44a979a","preferred_item":"sku-647d4c278eff3fcf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-35943b236958e24b":2,"sku-647d4c278eff3fcf":1,"sku-dfc4f1a92654f7a5":2}},"note_event_id":"evt-05b251255278ed01","profile_event_id":"evt-20fc5525c4da455f","replicate":1,"request_id":"req-aa26cd45d713e470","source_id":"gx-evaluation-a7adb83112f191b6","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-fd6e388cffde1fff","attribute":"preferred_item","conflict_value":"sku-fd6e388cffde1fff","fallback_item":"sku-8a6c9df9750a985b","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-deafa8ffb7f0395f","preferred_item":"sku-41ea0b6939980ab7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-41ea0b6939980ab7":0,"sku-8a6c9df9750a985b":2,"sku-fd6e388cffde1fff":2}},"note_event_id":"evt-022043cbb94a8efc","profile_event_id":"evt-f623558820e5b176","replicate":1,"request_id":"req-c97e610deb7745c8","source_id":"gx-evaluation-4f966dd51c4b9a2c","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-17b5589183bc2aa5","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-f3c76d75f2a2b69a","gold_fields":{"delivery_city":"New York","order_customer":"acct-7e036222b86d1140","preferred_item":"sku-636c43acc9cab2e5"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-17b5589183bc2aa5":2,"sku-636c43acc9cab2e5":1,"sku-f3c76d75f2a2b69a":2}},"note_event_id":"evt-fea88dc520d0186f","profile_event_id":"evt-28c6a60058b94f22","replicate":2,"request_id":"req-6a16c98a24f0235b","source_id":"gx-evaluation-30138a155b945d2b","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-3e1ccd5019e46464","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-b90bfc1e7a01e451","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-a825a2d7a5e06dc3","preferred_item":"sku-ba55e54902bf588f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3e1ccd5019e46464":2,"sku-b90bfc1e7a01e451":2,"sku-ba55e54902bf588f":0}},"note_event_id":"evt-d7e054f63ec0b748","profile_event_id":"evt-6860b6635131118d","replicate":2,"request_id":"req-a69261fad46b3708","source_id":"gx-evaluation-5368c06b0c8af5c9","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-b4db78f9f0c82ca8","attribute":"order_customer","conflict_value":"acct-e92b0da867cec492","fallback_item":"sku-4679fdc17de48e86","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-87fbe25d15b82193","preferred_item":"sku-93b7e1839ed112dd"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4679fdc17de48e86":2,"sku-93b7e1839ed112dd":1,"sku-b4db78f9f0c82ca8":2}},"note_event_id":"evt-b48d08ed4b21d56c","profile_event_id":"evt-0fd0b7ecc08e5a7e","replicate":2,"request_id":"req-518563813773b216","source_id":"gx-evaluation-65ad2b73422d3859","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-911f2a1771015efc","attribute":"order_customer","conflict_value":"acct-f83d7bc401357147","fallback_item":"sku-d114cfe69be53931","gold_fields":{"delivery_city":"Prague","order_customer":"acct-99b71404f6e92166","preferred_item":"sku-710e4148f6a0bf79"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-710e4148f6a0bf79":0,"sku-911f2a1771015efc":2,"sku-d114cfe69be53931":2}},"note_event_id":"evt-514014c5ba74d795","profile_event_id":"evt-813c18f45f34658d","replicate":2,"request_id":"req-34b14b513f62f0ee","source_id":"gx-evaluation-623811d44f3f30af","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-00fadf2ebc7c5e90","attribute":"preferred_item","conflict_value":"sku-00fadf2ebc7c5e90","fallback_item":"sku-510312aee657cd7a","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-58f901f9d9a86dae","preferred_item":"sku-e9ddf606eecfa99b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-00fadf2ebc7c5e90":2,"sku-510312aee657cd7a":2,"sku-e9ddf606eecfa99b":1}},"note_event_id":"evt-75e1f760f61354b8","profile_event_id":"evt-68ac8cc72bfdf353","replicate":2,"request_id":"req-87b6850eb6fca956","source_id":"gx-evaluation-832281d834a4d069","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-4e2116479adee23e","attribute":"preferred_item","conflict_value":"sku-4e2116479adee23e","fallback_item":"sku-3165612468d276c8","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-028eaf5084c46bc8","preferred_item":"sku-2f64744ea65d7acb"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2f64744ea65d7acb":0,"sku-3165612468d276c8":2,"sku-4e2116479adee23e":2}},"note_event_id":"evt-38665394e4df8bae","profile_event_id":"evt-afb0ffe83b20f791","replicate":2,"request_id":"req-e17789aa3008f544","source_id":"gx-evaluation-d9a3bd50f31f8e6b","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-585cb32d4ac102b7","attribute":"delivery_city","conflict_value":"Helsinki","fallback_item":"sku-87d469ad8e96b16f","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-ba274033f4ad1e94","preferred_item":"sku-d87f6641dc13bd20"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-585cb32d4ac102b7":2,"sku-87d469ad8e96b16f":2,"sku-d87f6641dc13bd20":1}},"note_event_id":"evt-1de5a6303e6339af","profile_event_id":"evt-37adacfde68d27d3","replicate":2,"request_id":"req-25be0e2c6b86b83c","source_id":"gx-evaluation-9bfe481f4de1e0d5","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-97571d5a471a5fda","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-2e3d8fa5d89dce09","gold_fields":{"delivery_city":"Prague","order_customer":"acct-19e882e403c274ad","preferred_item":"sku-8c0b0762e53ac0d4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2e3d8fa5d89dce09":2,"sku-8c0b0762e53ac0d4":0,"sku-97571d5a471a5fda":2}},"note_event_id":"evt-eed1c8cf4d9781db","profile_event_id":"evt-34380c821f2d2d6a","replicate":2,"request_id":"req-557a5d9312d615fa","source_id":"gx-evaluation-b1e6f3752aa86625","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-6d1d19d21fe948e9","attribute":"order_customer","conflict_value":"acct-d8f811d8d0e627e4","fallback_item":"sku-d5b8604ace4c11fb","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-83ecf23a744407bd","preferred_item":"sku-601382b6f80fb588"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-601382b6f80fb588":1,"sku-6d1d19d21fe948e9":2,"sku-d5b8604ace4c11fb":2}},"note_event_id":"evt-c02357e27c3268f3","profile_event_id":"evt-3532f6133d3ab79a","replicate":2,"request_id":"req-22ac6db11de30cb2","source_id":"gx-evaluation-3b02bcf3b2f5afdc","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-437c3968db8f3334","attribute":"order_customer","conflict_value":"acct-d0c5030d33a26a1d","fallback_item":"sku-71c3e1c5fbb813a8","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-f7ac70659e4f971f","preferred_item":"sku-13a43d8bcfee1a59"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-13a43d8bcfee1a59":0,"sku-437c3968db8f3334":2,"sku-71c3e1c5fbb813a8":2}},"note_event_id":"evt-d4c5555cc15ef06b","profile_event_id":"evt-64174513e2d681bf","replicate":2,"request_id":"req-2c08b8e409ed8a33","source_id":"gx-evaluation-df7a865e7997a984","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-7a4aaee75471f936","attribute":"preferred_item","conflict_value":"sku-7a4aaee75471f936","fallback_item":"sku-211da804c3126a51","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-26c1817178becc9e","preferred_item":"sku-8fb904ff3d4104af"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-211da804c3126a51":2,"sku-7a4aaee75471f936":2,"sku-8fb904ff3d4104af":1}},"note_event_id":"evt-9fe0534053c9c8b1","profile_event_id":"evt-9501d87a0c2c1716","replicate":2,"request_id":"req-8d32bceba07d53aa","source_id":"gx-evaluation-f4fbe495a57480d2","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-cc14c6835de6c9b0","attribute":"preferred_item","conflict_value":"sku-cc14c6835de6c9b0","fallback_item":"sku-41f9ed2498ccdfa9","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-273df47e241a7cc4","preferred_item":"sku-931eda7bc78f40cf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-41f9ed2498ccdfa9":2,"sku-931eda7bc78f40cf":0,"sku-cc14c6835de6c9b0":2}},"note_event_id":"evt-80b6f53008d0cfd7","profile_event_id":"evt-3831c9c86d711245","replicate":2,"request_id":"req-7beeb88d62cb71d5","source_id":"gx-evaluation-8b53d0716bacc0b5","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-37202b61d53f581e","attribute":"delivery_city","conflict_value":"Cairo","fallback_item":"sku-677b79d0e74089ef","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-8463118e8f6e590b","preferred_item":"sku-4b17215790305fca"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-37202b61d53f581e":2,"sku-4b17215790305fca":1,"sku-677b79d0e74089ef":2}},"note_event_id":"evt-9ac09ec478bbbea7","profile_event_id":"evt-326f511ccce05450","replicate":2,"request_id":"req-51c4684716633d2f","source_id":"gx-evaluation-1d5990b8a92c1046","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-63726c4b1ecc814a","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-9a8296cdec1e01ec","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-52ba7902ffafa381","preferred_item":"sku-e14ad5fd46a84996"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-63726c4b1ecc814a":2,"sku-9a8296cdec1e01ec":2,"sku-e14ad5fd46a84996":0}},"note_event_id":"evt-ce2a7278e9d4aba2","profile_event_id":"evt-4d413076aae01c2c","replicate":2,"request_id":"req-52c0a70525738453","source_id":"gx-evaluation-faec761b0065ca37","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-351593f1946d4771","attribute":"order_customer","conflict_value":"acct-c9b632c2327202ac","fallback_item":"sku-52394fb412d7a5cf","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-bd3f9c1b81a936ed","preferred_item":"sku-3f3637f087006947"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-351593f1946d4771":2,"sku-3f3637f087006947":1,"sku-52394fb412d7a5cf":2}},"note_event_id":"evt-204d7349cb6bb5a6","profile_event_id":"evt-9b51f6d1051ba256","replicate":2,"request_id":"req-3936e7b2d44f24dc","source_id":"gx-evaluation-810a6a34fafccf6b","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-e06de71fc5037ff0","attribute":"order_customer","conflict_value":"acct-86e63a682a087add","fallback_item":"sku-84049f73f0dc1368","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-f708b6170e972324","preferred_item":"sku-6ccbe5fde53df5e4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6ccbe5fde53df5e4":0,"sku-84049f73f0dc1368":2,"sku-e06de71fc5037ff0":2}},"note_event_id":"evt-2d30c6179e7d8448","profile_event_id":"evt-9b2c57814bd166df","replicate":2,"request_id":"req-4750eceb46a93679","source_id":"gx-evaluation-59a4a0c588c96d33","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-67953060c2151c51","attribute":"preferred_item","conflict_value":"sku-67953060c2151c51","fallback_item":"sku-a7693051238d74bb","gold_fields":{"delivery_city":"Prague","order_customer":"acct-8a5162f40f0bde36","preferred_item":"sku-edbbc7cbf414201b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-67953060c2151c51":2,"sku-a7693051238d74bb":2,"sku-edbbc7cbf414201b":1}},"note_event_id":"evt-90cbb160e996679e","profile_event_id":"evt-dc3d28da2024dd3a","replicate":2,"request_id":"req-229dfbd48e082d61","source_id":"gx-evaluation-95fff2de258b148d","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-c2738a917e2a242c","attribute":"preferred_item","conflict_value":"sku-c2738a917e2a242c","fallback_item":"sku-c4041e92887419ee","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-d6b0611354e16465","preferred_item":"sku-ceac77e9d3691c16"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-c2738a917e2a242c":2,"sku-c4041e92887419ee":2,"sku-ceac77e9d3691c16":0}},"note_event_id":"evt-39393dd33e27e032","profile_event_id":"evt-fc6b6b7e0aa319e4","replicate":2,"request_id":"req-b0ea638d555108ba","source_id":"gx-evaluation-8248234af34b163c","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-71d494b615b5ef0a","attribute":"delivery_city","conflict_value":"Cape Town","fallback_item":"sku-6e0cb27487b0a4e8","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-e3b81290f57064eb","preferred_item":"sku-e68ac954a7a5761c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6e0cb27487b0a4e8":2,"sku-71d494b615b5ef0a":2,"sku-e68ac954a7a5761c":1}},"note_event_id":"evt-f311dffc8bae5807","profile_event_id":"evt-1adcb6d2ede631f6","replicate":2,"request_id":"req-c6da494d5109eb14","source_id":"gx-evaluation-fc586bc5ad4507a8","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-2e490c509ad97f6b","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-b2004383bdfdc271","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-8f85b7ffcb370c4c","preferred_item":"sku-e3eb60e1147f1cca"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2e490c509ad97f6b":2,"sku-b2004383bdfdc271":2,"sku-e3eb60e1147f1cca":0}},"note_event_id":"evt-4011e3bf185dd86a","profile_event_id":"evt-8df78ac1dc2b7f0c","replicate":2,"request_id":"req-1ba1ebdd7aa6059d","source_id":"gx-evaluation-e7a031e14255a5bc","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-e57afa0775bb507b","attribute":"order_customer","conflict_value":"acct-d39aa3f69bf5688b","fallback_item":"sku-4004376be34f38f4","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-844cdc99f56cdef0","preferred_item":"sku-865ec9a5bd17e4e1"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4004376be34f38f4":2,"sku-865ec9a5bd17e4e1":1,"sku-e57afa0775bb507b":2}},"note_event_id":"evt-8c798ceb12130b59","profile_event_id":"evt-4c6f4548320dbd58","replicate":2,"request_id":"req-737371af45fff078","source_id":"gx-evaluation-b1d03f984af0233f","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-fba4edd22205b9eb","attribute":"order_customer","conflict_value":"acct-fa10bef8d17cb793","fallback_item":"sku-2530c0e152afd944","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-36274dfe990223bf","preferred_item":"sku-87219c7ab6a3f863"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2530c0e152afd944":2,"sku-87219c7ab6a3f863":0,"sku-fba4edd22205b9eb":2}},"note_event_id":"evt-84aa5ed3dbfce92f","profile_event_id":"evt-40db40a27d3a6717","replicate":2,"request_id":"req-8f10809a6db9b4fb","source_id":"gx-evaluation-090b28a5243e3077","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-981b555bf1dcb0dc","attribute":"preferred_item","conflict_value":"sku-981b555bf1dcb0dc","fallback_item":"sku-ac0ffe92d4a98ef1","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-533a2ec251297300","preferred_item":"sku-48eb7bfe708eb752"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-48eb7bfe708eb752":1,"sku-981b555bf1dcb0dc":2,"sku-ac0ffe92d4a98ef1":2}},"note_event_id":"evt-426c690cd68201ad","profile_event_id":"evt-1f103e2175a95d14","replicate":2,"request_id":"req-ed384a03aaf8be0a","source_id":"gx-evaluation-eebb54d0c16b50a7","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-da78697ac9e762b4","attribute":"preferred_item","conflict_value":"sku-da78697ac9e762b4","fallback_item":"sku-91814565fc6cee46","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-fbb5cd99fb77f047","preferred_item":"sku-0471365531d17fbb"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0471365531d17fbb":0,"sku-91814565fc6cee46":2,"sku-da78697ac9e762b4":2}},"note_event_id":"evt-3c4fa59daf9c94e6","profile_event_id":"evt-242c51a071c3945c","replicate":2,"request_id":"req-633fa19ca2c2242a","source_id":"gx-evaluation-b0d95791b72122db","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-ecde40ad7a307709","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-607fda4aa1fbf331","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-353984b9179358aa","preferred_item":"sku-baa971743bb6b279"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-607fda4aa1fbf331":2,"sku-baa971743bb6b279":1,"sku-ecde40ad7a307709":2}},"note_event_id":"evt-c8ebde3c6360ef98","profile_event_id":"evt-899b5600ce80cb49","replicate":3,"request_id":"req-9ddf612cf33cc68a","source_id":"gx-evaluation-c9c04290073920af","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-c7e7ba42e6e38c33","attribute":"delivery_city","conflict_value":"Mumbai","fallback_item":"sku-e7db9179aa999149","gold_fields":{"delivery_city":"Prague","order_customer":"acct-9ff8d2ed8679a1ea","preferred_item":"sku-611b16f7f340fdd8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-611b16f7f340fdd8":0,"sku-c7e7ba42e6e38c33":2,"sku-e7db9179aa999149":2}},"note_event_id":"evt-34dd9427cc1deb03","profile_event_id":"evt-6e8517ba2f3d77ba","replicate":3,"request_id":"req-bf9c1e7465ddcb67","source_id":"gx-evaluation-24ce09eab52738e9","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-b66170d17b681dba","attribute":"order_customer","conflict_value":"acct-a20f63eb39575ae3","fallback_item":"sku-731b162cfc90b395","gold_fields":{"delivery_city":"Prague","order_customer":"acct-71bafe0da3dca20b","preferred_item":"sku-6ce66f8c4357c725"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6ce66f8c4357c725":1,"sku-731b162cfc90b395":2,"sku-b66170d17b681dba":2}},"note_event_id":"evt-f921d65ae9226b2e","profile_event_id":"evt-5ec4212987644d8b","replicate":3,"request_id":"req-a3b19e054b1a6a9e","source_id":"gx-evaluation-28ef6c673334a185","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-f9f1b0edade29392","attribute":"order_customer","conflict_value":"acct-980cabbfdf700c55","fallback_item":"sku-f10967f3a312e7cf","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-646ef82b58d969c8","preferred_item":"sku-04ba5dddb48f8b08"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-04ba5dddb48f8b08":0,"sku-f10967f3a312e7cf":2,"sku-f9f1b0edade29392":2}},"note_event_id":"evt-6c09e3e16157fc36","profile_event_id":"evt-9ee815f7bec627dc","replicate":3,"request_id":"req-9ee1bd3763c24410","source_id":"gx-evaluation-f58ea9a1a0e1899b","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-662ea37f3919ce95","attribute":"preferred_item","conflict_value":"sku-662ea37f3919ce95","fallback_item":"sku-0737db8509b7273a","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-f42755f64be5970c","preferred_item":"sku-2e553e01f5cf9fc8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0737db8509b7273a":2,"sku-2e553e01f5cf9fc8":1,"sku-662ea37f3919ce95":2}},"note_event_id":"evt-fa5cad92e875efcf","profile_event_id":"evt-9169fab716b45205","replicate":3,"request_id":"req-42768004da91f9e4","source_id":"gx-evaluation-6272b0ecbee20377","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-7fb8c13e076533ff","attribute":"preferred_item","conflict_value":"sku-7fb8c13e076533ff","fallback_item":"sku-c2f78ce8fed8703a","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-5ab6f8bc9a0702cb","preferred_item":"sku-86eecf7c03e12fc7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7fb8c13e076533ff":2,"sku-86eecf7c03e12fc7":0,"sku-c2f78ce8fed8703a":2}},"note_event_id":"evt-50ee0d677614067b","profile_event_id":"evt-8112a62db485d05c","replicate":3,"request_id":"req-828c347eeb5ef8d1","source_id":"gx-evaluation-6da554d2dcb5ff78","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-655e803a95905bda","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-c2b4eeaa74117d71","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-c54ab4fb59b27915","preferred_item":"sku-b7249158233e2412"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-655e803a95905bda":2,"sku-b7249158233e2412":1,"sku-c2b4eeaa74117d71":2}},"note_event_id":"evt-525cf7c4797d9a08","profile_event_id":"evt-f44b030fc6bf1940","replicate":3,"request_id":"req-8349c3f7a61e8c2f","source_id":"gx-evaluation-6b173002c8a4ad36","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-eff39441ec731dcb","attribute":"delivery_city","conflict_value":"Cape Town","fallback_item":"sku-f983f74d88b82d11","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-bf6b2a3c280f8af9","preferred_item":"sku-f894698ea81fc8df"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-eff39441ec731dcb":2,"sku-f894698ea81fc8df":0,"sku-f983f74d88b82d11":2}},"note_event_id":"evt-314ecef3e8faecc7","profile_event_id":"evt-a7e226255273975b","replicate":3,"request_id":"req-a98bb932c3e26c64","source_id":"gx-evaluation-1c103d5a72a6b220","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-ea365772ae2afe23","attribute":"order_customer","conflict_value":"acct-f153324db113029b","fallback_item":"sku-979e8d526eccfcc0","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-383d19af25646b24","preferred_item":"sku-7d7f778dd58acde8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7d7f778dd58acde8":1,"sku-979e8d526eccfcc0":2,"sku-ea365772ae2afe23":2}},"note_event_id":"evt-ea21622391566351","profile_event_id":"evt-52f838283e17e98e","replicate":3,"request_id":"req-3a3c99b29dda0721","source_id":"gx-evaluation-ad1d46469d09e827","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-4537e5aa5505684c","attribute":"order_customer","conflict_value":"acct-c81202a682f14582","fallback_item":"sku-3e88261eb1e12774","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-8fafec7d50a91073","preferred_item":"sku-08a16172d8d14f74"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-08a16172d8d14f74":0,"sku-3e88261eb1e12774":2,"sku-4537e5aa5505684c":2}},"note_event_id":"evt-04dbbff09fbfac35","profile_event_id":"evt-accac74f17954a69","replicate":3,"request_id":"req-6390bee76b6bca32","source_id":"gx-evaluation-c0850e84a66d5dd2","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-94e6ab80fc80220c","attribute":"preferred_item","conflict_value":"sku-94e6ab80fc80220c","fallback_item":"sku-8939ffdfae681f49","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-20e12ad45f089278","preferred_item":"sku-33503849cdd06169"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-33503849cdd06169":1,"sku-8939ffdfae681f49":2,"sku-94e6ab80fc80220c":2}},"note_event_id":"evt-aa979fedab6d697c","profile_event_id":"evt-2a7e23afd2c779eb","replicate":3,"request_id":"req-80a9f0062daa5b7c","source_id":"gx-evaluation-2a32fe1e84f450b8","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-57142e1fa025798b","attribute":"preferred_item","conflict_value":"sku-57142e1fa025798b","fallback_item":"sku-c39de0b3485bd51f","gold_fields":{"delivery_city":"Prague","order_customer":"acct-5a56e479ba986cf8","preferred_item":"sku-e8e58993a4f24c8f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-57142e1fa025798b":2,"sku-c39de0b3485bd51f":2,"sku-e8e58993a4f24c8f":0}},"note_event_id":"evt-f2ad7626e955eabe","profile_event_id":"evt-be78b858af0e8ba9","replicate":3,"request_id":"req-015b9e95a5c5c6d5","source_id":"gx-evaluation-ce189b9da36201e6","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-41d3345a80280dec","attribute":"delivery_city","conflict_value":"Helsinki","fallback_item":"sku-16abed8e825a0cc3","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-1d6f2e6cd4ed1604","preferred_item":"sku-6902d82e71ec5e80"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-16abed8e825a0cc3":2,"sku-41d3345a80280dec":2,"sku-6902d82e71ec5e80":1}},"note_event_id":"evt-03da9a8a94f24485","profile_event_id":"evt-802631a519bde998","replicate":3,"request_id":"req-6f459d14ac1efff3","source_id":"gx-evaluation-f49a339ce4098d11","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-f32eaac3212edde9","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-e634bbaed678e079","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-a0ddf91b3c1e1ce9","preferred_item":"sku-857a2b72db1cefec"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-857a2b72db1cefec":0,"sku-e634bbaed678e079":2,"sku-f32eaac3212edde9":2}},"note_event_id":"evt-87b921072d8324d2","profile_event_id":"evt-92ecd447d98be365","replicate":3,"request_id":"req-74b463b613a92214","source_id":"gx-evaluation-a4a4a7efd0a71ac2","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-795ccbde32c30169","attribute":"order_customer","conflict_value":"acct-641df1a9c39c8062","fallback_item":"sku-7a8b2ee59dc52dfc","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-f8030cdb9d40f691","preferred_item":"sku-f785b63cea2d8a4f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-795ccbde32c30169":2,"sku-7a8b2ee59dc52dfc":2,"sku-f785b63cea2d8a4f":1}},"note_event_id":"evt-eb4ab1ffe728c04d","profile_event_id":"evt-ba770270ea8ea9ed","replicate":3,"request_id":"req-a719f13f64aaa901","source_id":"gx-evaluation-80c9ce1cd6c1d244","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-f4826ffcd1232050","attribute":"order_customer","conflict_value":"acct-16b2d3b6faab3c09","fallback_item":"sku-87af9c67e09293a6","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-215ab6470db6c520","preferred_item":"sku-5ec174140c24a8b7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-5ec174140c24a8b7":0,"sku-87af9c67e09293a6":2,"sku-f4826ffcd1232050":2}},"note_event_id":"evt-37e42fc747eb0759","profile_event_id":"evt-a9729351f08f63b7","replicate":3,"request_id":"req-90b446d2658c75ec","source_id":"gx-evaluation-a5e55ce97a174273","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-a8ba024414cf524e","attribute":"preferred_item","conflict_value":"sku-a8ba024414cf524e","fallback_item":"sku-6ad91241c644fc79","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-ffabd1ecf9a3aa7e","preferred_item":"sku-f2c6c707c28acaec"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6ad91241c644fc79":2,"sku-a8ba024414cf524e":2,"sku-f2c6c707c28acaec":1}},"note_event_id":"evt-1f58285f5df3ceb0","profile_event_id":"evt-1eb6bbe24351c508","replicate":3,"request_id":"req-db538b82ee847e01","source_id":"gx-evaluation-b3594e0e676c34a0","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-49fc559421db9f7a","attribute":"preferred_item","conflict_value":"sku-49fc559421db9f7a","fallback_item":"sku-232a4bf4325b2651","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-808ac0c389e357f5","preferred_item":"sku-b6c2a7d5e0ef0c25"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-232a4bf4325b2651":2,"sku-49fc559421db9f7a":2,"sku-b6c2a7d5e0ef0c25":0}},"note_event_id":"evt-120c18f53d2cff29","profile_event_id":"evt-23a2a541b2c10faa","replicate":3,"request_id":"req-78ce97e3bb66baca","source_id":"gx-evaluation-31322c8132885cb9","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-557e6c7219e663f1","attribute":"delivery_city","conflict_value":"Lisbon","fallback_item":"sku-7c04b274aba60869","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-c68378c9f85cafaa","preferred_item":"sku-21a8fe33e388dfe6"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-21a8fe33e388dfe6":1,"sku-557e6c7219e663f1":2,"sku-7c04b274aba60869":2}},"note_event_id":"evt-f1e560ea140abebd","profile_event_id":"evt-771d2736e5d34dd9","replicate":3,"request_id":"req-bd4b4fa8ab71f821","source_id":"gx-evaluation-b39c8b62792ff07e","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-11b6c01008744c17","attribute":"delivery_city","conflict_value":"Cape Town","fallback_item":"sku-f84f62696439eaa5","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-88c345ef8e05080c","preferred_item":"sku-aa2aa5237991e08a"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-11b6c01008744c17":2,"sku-aa2aa5237991e08a":0,"sku-f84f62696439eaa5":2}},"note_event_id":"evt-1be0250ce27c0396","profile_event_id":"evt-ea389ced65b6c9f0","replicate":3,"request_id":"req-24cefe5454fa3dcd","source_id":"gx-evaluation-6f47a6b817ea6d57","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-5a79a734b4c2e59b","attribute":"order_customer","conflict_value":"acct-994ea2810aad9cf5","fallback_item":"sku-e206b99345c8facd","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-d81757f2c0e542c2","preferred_item":"sku-b04b6d062191a598"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-5a79a734b4c2e59b":2,"sku-b04b6d062191a598":1,"sku-e206b99345c8facd":2}},"note_event_id":"evt-7c0167b0f21fd679","profile_event_id":"evt-2acb976d00be896f","replicate":3,"request_id":"req-dd0a2f15df669ea5","source_id":"gx-evaluation-f9c6d23205e2535b","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-b15b30aa6ed9368a","attribute":"order_customer","conflict_value":"acct-edc1ddfac5e11d17","fallback_item":"sku-850afaa80c332589","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-83466148d941af89","preferred_item":"sku-90651d4cf6b0d24c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-850afaa80c332589":2,"sku-90651d4cf6b0d24c":0,"sku-b15b30aa6ed9368a":2}},"note_event_id":"evt-d8a66ba05cd1b0d5","profile_event_id":"evt-6573573625348db4","replicate":3,"request_id":"req-5c72c3ecccdce4f5","source_id":"gx-evaluation-ab3d9135bc02dcce","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-681ffed7411dcf14","attribute":"preferred_item","conflict_value":"sku-681ffed7411dcf14","fallback_item":"sku-17296453826e7c47","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-d0d83ab79f6b4dc2","preferred_item":"sku-950611a5eb77d926"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-17296453826e7c47":2,"sku-681ffed7411dcf14":2,"sku-950611a5eb77d926":1}},"note_event_id":"evt-6258d8db6b882893","profile_event_id":"evt-bf39102f7a896d3f","replicate":3,"request_id":"req-7b2602a638a65cff","source_id":"gx-evaluation-b1da22a4a4a79312","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-d39aa04813c9311d","attribute":"preferred_item","conflict_value":"sku-d39aa04813c9311d","fallback_item":"sku-f39a07ed69fc0785","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-599a6c226902144b","preferred_item":"sku-0cb77fc99ede172c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0cb77fc99ede172c":0,"sku-d39aa04813c9311d":2,"sku-f39a07ed69fc0785":2}},"note_event_id":"evt-1a15084144d77658","profile_event_id":"evt-6995af7321120146","replicate":3,"request_id":"req-adf0b1d9201419e5","source_id":"gx-evaluation-780551a510ebeb5b","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-6058ad276291170b","attribute":"delivery_city","conflict_value":"Cape Town","fallback_item":"sku-63385d33d93c9537","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-9e6469e7b6fa554a","preferred_item":"sku-9901742be6240f54"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6058ad276291170b":2,"sku-63385d33d93c9537":2,"sku-9901742be6240f54":1}},"note_event_id":"evt-1496c331c44d3995","profile_event_id":"evt-5cbbefe064276301","replicate":4,"request_id":"req-f1bb32f3ce32ad05","source_id":"gx-evaluation-c4293f91a32fcfd5","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-0b6b3258dbfa5741","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-6c52bee5e6708c7f","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-94ccf996934d7ac8","preferred_item":"sku-ea3bdf2f2fb592c5"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0b6b3258dbfa5741":2,"sku-6c52bee5e6708c7f":2,"sku-ea3bdf2f2fb592c5":0}},"note_event_id":"evt-df1163efabcb9551","profile_event_id":"evt-016454ccd33a1836","replicate":4,"request_id":"req-ef241d4ff629ce08","source_id":"gx-evaluation-fb56b0bf785fe730","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-9e920f661fc2d37d","attribute":"order_customer","conflict_value":"acct-5778ba7072879a01","fallback_item":"sku-6c0ca383b3c15632","gold_fields":{"delivery_city":"New York","order_customer":"acct-c5fecbf77134b965","preferred_item":"sku-186f5928fbf234c2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-186f5928fbf234c2":1,"sku-6c0ca383b3c15632":2,"sku-9e920f661fc2d37d":2}},"note_event_id":"evt-9105f2eb95c80ddc","profile_event_id":"evt-bcb0433cd703f65f","replicate":4,"request_id":"req-eb4f11b705d6fac5","source_id":"gx-evaluation-187d9d1790918d7c","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-d47810bc25ee31b5","attribute":"order_customer","conflict_value":"acct-9652035cb6da467e","fallback_item":"sku-ea1ee90c3d1d25ec","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-ff618f1f23930eea","preferred_item":"sku-495fb0ba2ec0789d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-495fb0ba2ec0789d":0,"sku-d47810bc25ee31b5":2,"sku-ea1ee90c3d1d25ec":2}},"note_event_id":"evt-7612b07934756145","profile_event_id":"evt-a4aa9f3d759447a8","replicate":4,"request_id":"req-0bc980642f4e4d1d","source_id":"gx-evaluation-f7a591daac7afbb8","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-4e5cb95c143878bb","attribute":"preferred_item","conflict_value":"sku-4e5cb95c143878bb","fallback_item":"sku-012ff0743a2d7f4c","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-134f42da2cee97d7","preferred_item":"sku-9474ee3e4f863d97"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-012ff0743a2d7f4c":2,"sku-4e5cb95c143878bb":2,"sku-9474ee3e4f863d97":1}},"note_event_id":"evt-4a466d66603c05ac","profile_event_id":"evt-d91adca62752eff6","replicate":4,"request_id":"req-ca7692c45ecf04d0","source_id":"gx-evaluation-ca3872001f2ecfb9","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-e827bdcef0e288cf","attribute":"preferred_item","conflict_value":"sku-e827bdcef0e288cf","fallback_item":"sku-66eacab3793459d6","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-caa5cba13d2da4d1","preferred_item":"sku-38c8a487cc4951a2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-38c8a487cc4951a2":0,"sku-66eacab3793459d6":2,"sku-e827bdcef0e288cf":2}},"note_event_id":"evt-184a0e80b2d0f07b","profile_event_id":"evt-70159e71dc8840fe","replicate":4,"request_id":"req-6e27d2ae04578d9b","source_id":"gx-evaluation-87bd50e9ecddaf09","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-8d5da515353997e3","attribute":"delivery_city","conflict_value":"Cairo","fallback_item":"sku-624a7d1bf81ef23b","gold_fields":{"delivery_city":"New York","order_customer":"acct-cde351022c952043","preferred_item":"sku-ec83cd0db3daeac1"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-624a7d1bf81ef23b":2,"sku-8d5da515353997e3":2,"sku-ec83cd0db3daeac1":1}},"note_event_id":"evt-cb9086b2bfbba096","profile_event_id":"evt-80c7d7cfdbdcdd0e","replicate":4,"request_id":"req-785d86a5283fe3ad","source_id":"gx-evaluation-81c665a63bc22724","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-8ba09ac2c230e862","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-7f8d277cdefbcb82","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-eb8567bc9654e561","preferred_item":"sku-6828956569fb719d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6828956569fb719d":0,"sku-7f8d277cdefbcb82":2,"sku-8ba09ac2c230e862":2}},"note_event_id":"evt-42f6e8bf607aaa0c","profile_event_id":"evt-63e5bf17267fdb6c","replicate":4,"request_id":"req-b3b41066ea5cf4bd","source_id":"gx-evaluation-6a1e6ccd614cd81b","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-caa50241446e5b6f","attribute":"order_customer","conflict_value":"acct-0d62d0ac845fc2ce","fallback_item":"sku-f5d72613340ccfce","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-e6081704cb0c06eb","preferred_item":"sku-32e517ffca4e3b09"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-32e517ffca4e3b09":1,"sku-caa50241446e5b6f":2,"sku-f5d72613340ccfce":2}},"note_event_id":"evt-9b85137075123146","profile_event_id":"evt-2e00d0b3f347c3ba","replicate":4,"request_id":"req-3db00681de22df62","source_id":"gx-evaluation-a6cb14052024c5d9","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-762ea9f5b1af265c","attribute":"order_customer","conflict_value":"acct-ffa952fc19f56990","fallback_item":"sku-a426779786a557e7","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-3bdbc053bdd05e31","preferred_item":"sku-47de0e18e765d7d3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-47de0e18e765d7d3":0,"sku-762ea9f5b1af265c":2,"sku-a426779786a557e7":2}},"note_event_id":"evt-78841b668525bba2","profile_event_id":"evt-5c4f23e5289154cd","replicate":4,"request_id":"req-4a443553fc65f279","source_id":"gx-evaluation-f8361a210da348c9","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-c0202437bf1b01df","attribute":"preferred_item","conflict_value":"sku-c0202437bf1b01df","fallback_item":"sku-4129761fca83c146","gold_fields":{"delivery_city":"New York","order_customer":"acct-7606dbba390c56bd","preferred_item":"sku-0c2e10df01f5a2dd"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0c2e10df01f5a2dd":1,"sku-4129761fca83c146":2,"sku-c0202437bf1b01df":2}},"note_event_id":"evt-16065c2c48a0b274","profile_event_id":"evt-db204dbdf588d546","replicate":4,"request_id":"req-757830ddd1341865","source_id":"gx-evaluation-4b3e2ef8b77affc9","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-0e6b6c2337161665","attribute":"preferred_item","conflict_value":"sku-0e6b6c2337161665","fallback_item":"sku-43f3b1d4b82e3d5c","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-532d6f8851171526","preferred_item":"sku-a5faf7238ce18f92"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0e6b6c2337161665":2,"sku-43f3b1d4b82e3d5c":2,"sku-a5faf7238ce18f92":0}},"note_event_id":"evt-523b7a699aea4c9a","profile_event_id":"evt-62973e53babefb93","replicate":4,"request_id":"req-6e648cc9ab055aca","source_id":"gx-evaluation-23f59a865aae2695","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-3f81084845d4269d","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-731cb201c51ce759","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-5bfee95d6a61fbe7","preferred_item":"sku-90da6d1e5c4b651c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3f81084845d4269d":2,"sku-731cb201c51ce759":2,"sku-90da6d1e5c4b651c":1}},"note_event_id":"evt-76faeb61705848a3","profile_event_id":"evt-5c5931c079d459dc","replicate":4,"request_id":"req-73c8656b9304f771","source_id":"gx-evaluation-ea80a7c221898620","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-b70a932e1b4b98b7","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-8197aefc2ccbacdd","gold_fields":{"delivery_city":"Prague","order_customer":"acct-47958e9405892130","preferred_item":"sku-461fabf09a49a018"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-461fabf09a49a018":0,"sku-8197aefc2ccbacdd":2,"sku-b70a932e1b4b98b7":2}},"note_event_id":"evt-3486fc415e103425","profile_event_id":"evt-9311b8e7e544e691","replicate":4,"request_id":"req-a6c9bced43981e3f","source_id":"gx-evaluation-fadf1ec64488e722","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-44621ebbd546ef41","attribute":"order_customer","conflict_value":"acct-5f805d08832396ef","fallback_item":"sku-556d8ecb7c703f37","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-e04a5dbca334f61c","preferred_item":"sku-1276029a3c90da26"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1276029a3c90da26":1,"sku-44621ebbd546ef41":2,"sku-556d8ecb7c703f37":2}},"note_event_id":"evt-bcd7b4721a802cf7","profile_event_id":"evt-823361ea49ba8e8b","replicate":4,"request_id":"req-55a2041210774a3e","source_id":"gx-evaluation-f2dcc81b398f7168","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-01703e07f3f8d81c","attribute":"order_customer","conflict_value":"acct-87b432811e0aa54e","fallback_item":"sku-73cd0e8c08d70148","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-43a58dd2f7d9c272","preferred_item":"sku-fbca3a326bf63a2c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-01703e07f3f8d81c":2,"sku-73cd0e8c08d70148":2,"sku-fbca3a326bf63a2c":0}},"note_event_id":"evt-e0bd0426d739d89c","profile_event_id":"evt-6f46f66efda20f07","replicate":4,"request_id":"req-acb97707845c5646","source_id":"gx-evaluation-b4d2832ea862d74d","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-85a2a947d5caae40","attribute":"preferred_item","conflict_value":"sku-85a2a947d5caae40","fallback_item":"sku-2b1a7790a25c532c","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-92e1829f6ab46688","preferred_item":"sku-264a41909f546069"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-264a41909f546069":1,"sku-2b1a7790a25c532c":2,"sku-85a2a947d5caae40":2}},"note_event_id":"evt-601ddc91ed2d8011","profile_event_id":"evt-8bb1ae69cf57349a","replicate":4,"request_id":"req-6302bc7c50d6ac1c","source_id":"gx-evaluation-db174dc439b63809","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-1cbfee3f340a3dc4","attribute":"preferred_item","conflict_value":"sku-1cbfee3f340a3dc4","fallback_item":"sku-0ab47a78db0fc0cb","gold_fields":{"delivery_city":"Prague","order_customer":"acct-53c7e1139838e693","preferred_item":"sku-7332c576b5674b6a"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0ab47a78db0fc0cb":2,"sku-1cbfee3f340a3dc4":2,"sku-7332c576b5674b6a":0}},"note_event_id":"evt-b16940ecb1e9548e","profile_event_id":"evt-2a4a50c61e8ecb14","replicate":4,"request_id":"req-27a947fe1ed44ee2","source_id":"gx-evaluation-b06835586a8675f4","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-c010152c88ac2286","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-c6f074af7a2fa0bf","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-4bf37be08d028153","preferred_item":"sku-f2f3f87fd7c8ee80"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-c010152c88ac2286":2,"sku-c6f074af7a2fa0bf":2,"sku-f2f3f87fd7c8ee80":1}},"note_event_id":"evt-6ab213fd5bc1884e","profile_event_id":"evt-5b380d28f1924b45","replicate":4,"request_id":"req-904f2864e78e8461","source_id":"gx-evaluation-b0f5611be90ffeb5","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-8c3fc9a7247ea9fc","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-9bddd896a1bc20d0","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-dce41bcd9703151b","preferred_item":"sku-555d43fd274ef579"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-555d43fd274ef579":0,"sku-8c3fc9a7247ea9fc":2,"sku-9bddd896a1bc20d0":2}},"note_event_id":"evt-746fe44536cdfb6b","profile_event_id":"evt-7ba71b24f6613f50","replicate":4,"request_id":"req-cc4f559418f4fd0d","source_id":"gx-evaluation-01c380df9888124f","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-04fa89657c7247ab","attribute":"order_customer","conflict_value":"acct-e274224d250c8dcd","fallback_item":"sku-587eed02ff6a6eb1","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-e6d8ea792d6fe0a1","preferred_item":"sku-554fa1de4266300f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-04fa89657c7247ab":2,"sku-554fa1de4266300f":1,"sku-587eed02ff6a6eb1":2}},"note_event_id":"evt-6b66def84672b045","profile_event_id":"evt-3291f52723281b0a","replicate":4,"request_id":"req-24c4fd3cdda7e0f5","source_id":"gx-evaluation-c43e3d1aad4baf28","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-ffabb7368e3c449f","attribute":"order_customer","conflict_value":"acct-8e2e37aa76e36c16","fallback_item":"sku-de9eb7610447a0b6","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-502d4ab2ba5b624e","preferred_item":"sku-82a3986cf4e11f7a"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-82a3986cf4e11f7a":0,"sku-de9eb7610447a0b6":2,"sku-ffabb7368e3c449f":2}},"note_event_id":"evt-6f17059380e89db3","profile_event_id":"evt-7b76bb390da3c94c","replicate":4,"request_id":"req-c6453af351cdb14e","source_id":"gx-evaluation-bb24cf067f86f897","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-76cbbfb518cc569d","attribute":"preferred_item","conflict_value":"sku-76cbbfb518cc569d","fallback_item":"sku-333392b2b03e2022","gold_fields":{"delivery_city":"New York","order_customer":"acct-7d2a81f61251a830","preferred_item":"sku-8a945e51f7c24832"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-333392b2b03e2022":2,"sku-76cbbfb518cc569d":2,"sku-8a945e51f7c24832":1}},"note_event_id":"evt-9924e1aef1d9f49e","profile_event_id":"evt-f9cece6e71ac6eb1","replicate":4,"request_id":"req-1fb6b4fbfb50d155","source_id":"gx-evaluation-0981730847d0fc15","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-ca0487ece4275924","attribute":"preferred_item","conflict_value":"sku-ca0487ece4275924","fallback_item":"sku-a19f681f88c4beef","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-584b47045c9f5b49","preferred_item":"sku-32e4307b5f2688fc"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-32e4307b5f2688fc":0,"sku-a19f681f88c4beef":2,"sku-ca0487ece4275924":2}},"note_event_id":"evt-9ead9c6981065ac4","profile_event_id":"evt-d1b65a56960a371a","replicate":4,"request_id":"req-9e4e55300cbf5ccb","source_id":"gx-evaluation-207d166ca522d68a","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-401eb6b888ce55cd","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-2ba716476817bafd","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-49dc3fb82ab3d036","preferred_item":"sku-f203b6200465763c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2ba716476817bafd":2,"sku-401eb6b888ce55cd":2,"sku-f203b6200465763c":1}},"note_event_id":"evt-9922ad2d31de178c","profile_event_id":"evt-d845df6791dbbe29","replicate":5,"request_id":"req-acecec93ee7e3d99","source_id":"gx-evaluation-6eb9f6ecdb68c35f","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-5c275fd1cb243d84","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-d59285712b602dcd","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-8d3afca847c95151","preferred_item":"sku-aa5824aed87f0685"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-5c275fd1cb243d84":2,"sku-aa5824aed87f0685":0,"sku-d59285712b602dcd":2}},"note_event_id":"evt-c12f31f3807c81e3","profile_event_id":"evt-a59c39093046e7a2","replicate":5,"request_id":"req-5219e2b6626aa632","source_id":"gx-evaluation-3320d57612d9c76a","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-628c33ad92bbe23f","attribute":"order_customer","conflict_value":"acct-1cbdc5bbed0bec43","fallback_item":"sku-4d6593fd305ea214","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-ad9abbe20b077d38","preferred_item":"sku-4d383f534c6f9a4b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4d383f534c6f9a4b":1,"sku-4d6593fd305ea214":2,"sku-628c33ad92bbe23f":2}},"note_event_id":"evt-34bfa5ebf782713a","profile_event_id":"evt-6d3d9683d39f4119","replicate":5,"request_id":"req-48e5b4fcdb1693e1","source_id":"gx-evaluation-35f6c23a886ed5cc","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-f77f774b7f356106","attribute":"order_customer","conflict_value":"acct-acdc3d0941db09c4","fallback_item":"sku-580f6ff124bc29d0","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-85d21da5039026e7","preferred_item":"sku-1ded24d49a3fe281"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1ded24d49a3fe281":0,"sku-580f6ff124bc29d0":2,"sku-f77f774b7f356106":2}},"note_event_id":"evt-b05a441b67c82bc3","profile_event_id":"evt-fd7e7b0ddc45399a","replicate":5,"request_id":"req-3ea6bcfbf00cd224","source_id":"gx-evaluation-61b080734c84a4f8","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-8d2f2e0e27cdb508","attribute":"preferred_item","conflict_value":"sku-8d2f2e0e27cdb508","fallback_item":"sku-70a7bdaeb355b2fa","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-b08006134e42d6c1","preferred_item":"sku-d328def6c4fe4d8b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-70a7bdaeb355b2fa":2,"sku-8d2f2e0e27cdb508":2,"sku-d328def6c4fe4d8b":1}},"note_event_id":"evt-0cba152c51f24725","profile_event_id":"evt-1def32a61ff13e0a","replicate":5,"request_id":"req-00f87a10f347f12f","source_id":"gx-evaluation-8883c5df089d5b6c","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-e21022e5e203663b","attribute":"preferred_item","conflict_value":"sku-e21022e5e203663b","fallback_item":"sku-b98414139c8be9d9","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-da232008b089aa7a","preferred_item":"sku-fb7bd2e89e6421d8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-b98414139c8be9d9":2,"sku-e21022e5e203663b":2,"sku-fb7bd2e89e6421d8":0}},"note_event_id":"evt-fd3c84f8fc2421bb","profile_event_id":"evt-578cdb00522a2242","replicate":5,"request_id":"req-1ea669028e10a8c9","source_id":"gx-evaluation-542d6b85739ffda6","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-43715585bd80ee6b","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-b76b7626286c1b30","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-3bfc6dd50ee87071","preferred_item":"sku-457cc937c6d1ed13"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-43715585bd80ee6b":2,"sku-457cc937c6d1ed13":1,"sku-b76b7626286c1b30":2}},"note_event_id":"evt-a80ec176a3b08d8a","profile_event_id":"evt-04819d02b5b4f346","replicate":5,"request_id":"req-479bb48c2852b169","source_id":"gx-evaluation-d972e092009af54d","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-3295316a4bd169a8","attribute":"delivery_city","conflict_value":"Helsinki","fallback_item":"sku-ab5e7aff323e59db","gold_fields":{"delivery_city":"New York","order_customer":"acct-e8f98a407dc2e18c","preferred_item":"sku-7ab530cd27271922"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3295316a4bd169a8":2,"sku-7ab530cd27271922":0,"sku-ab5e7aff323e59db":2}},"note_event_id":"evt-141f05e33088c636","profile_event_id":"evt-dc4d688256840320","replicate":5,"request_id":"req-8b7c0584f0188c9f","source_id":"gx-evaluation-aa3a6cfcd08b490b","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-2228de4c71de4777","attribute":"order_customer","conflict_value":"acct-f02ec0b3794c9a12","fallback_item":"sku-3665b5c955751001","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-a5d15cfeb3a0262a","preferred_item":"sku-9dbd7e807b2371d2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2228de4c71de4777":2,"sku-3665b5c955751001":2,"sku-9dbd7e807b2371d2":1}},"note_event_id":"evt-70ca738e151d9e88","profile_event_id":"evt-96ab9c7def245881","replicate":5,"request_id":"req-6c7a47c8926166e0","source_id":"gx-evaluation-f36aae440fe6d8a9","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-58027f0794802d34","attribute":"order_customer","conflict_value":"acct-00b33b8da1dbe6ba","fallback_item":"sku-b8ba3afd9af5e22a","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-c203eb762f384ba4","preferred_item":"sku-913e0d09b4cd4c80"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-58027f0794802d34":2,"sku-913e0d09b4cd4c80":0,"sku-b8ba3afd9af5e22a":2}},"note_event_id":"evt-e15b1440307760d8","profile_event_id":"evt-6ef2c8eb345f849a","replicate":5,"request_id":"req-a13e2308735d402a","source_id":"gx-evaluation-ec5a57c72e4c05fc","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-c74138cead304b40","attribute":"preferred_item","conflict_value":"sku-c74138cead304b40","fallback_item":"sku-7352d782cfb47893","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-5728cfed552f601f","preferred_item":"sku-9bc4c9867f02e1b6"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7352d782cfb47893":2,"sku-9bc4c9867f02e1b6":1,"sku-c74138cead304b40":2}},"note_event_id":"evt-c3d37c5db6865690","profile_event_id":"evt-448a669c82ea6e35","replicate":5,"request_id":"req-fa49353b3bc7d4e0","source_id":"gx-evaluation-e4b97860f60d0b12","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-79d3f8a0d03ff856","attribute":"preferred_item","conflict_value":"sku-79d3f8a0d03ff856","fallback_item":"sku-96c3838b46e94bd4","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-bf36812c1b050b06","preferred_item":"sku-0d37c9875108a855"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0d37c9875108a855":0,"sku-79d3f8a0d03ff856":2,"sku-96c3838b46e94bd4":2}},"note_event_id":"evt-f94f77971eed217d","profile_event_id":"evt-3303997695fec980","replicate":5,"request_id":"req-319647cf1a365aa3","source_id":"gx-evaluation-8f282ac6209fd911","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-e819bce57004adfb","attribute":"delivery_city","conflict_value":"New York","fallback_item":"sku-0dff844a1a404785","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-d15ab81e72657ae3","preferred_item":"sku-0ff2ebb63e60fe19"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0dff844a1a404785":2,"sku-0ff2ebb63e60fe19":1,"sku-e819bce57004adfb":2}},"note_event_id":"evt-931e3fc6a1311cf0","profile_event_id":"evt-c3a601f7de1c2fb5","replicate":5,"request_id":"req-d8b5e0b41858cc72","source_id":"gx-evaluation-1ca32df831b3982d","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-de101e878c477f8a","attribute":"delivery_city","conflict_value":"Helsinki","fallback_item":"sku-4ecc2b1bc1ce3127","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-213367abaf529a92","preferred_item":"sku-73ae28b29f62a055"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4ecc2b1bc1ce3127":2,"sku-73ae28b29f62a055":0,"sku-de101e878c477f8a":2}},"note_event_id":"evt-6c3eb8d5b2a33f69","profile_event_id":"evt-e32289167ddaa0d4","replicate":5,"request_id":"req-400afd2ff27695de","source_id":"gx-evaluation-eef817465fae693f","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-c85ed331f7d0124f","attribute":"order_customer","conflict_value":"acct-1b670380ad03aabd","fallback_item":"sku-f9350d31650577a9","gold_fields":{"delivery_city":"New York","order_customer":"acct-1688f99d0c3b85b0","preferred_item":"sku-000f831d126af11c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-000f831d126af11c":1,"sku-c85ed331f7d0124f":2,"sku-f9350d31650577a9":2}},"note_event_id":"evt-0f9c73d3446e55cc","profile_event_id":"evt-c810fa615e796137","replicate":5,"request_id":"req-079afd8b3415f7ab","source_id":"gx-evaluation-ab2329fa7985fec3","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-b2210f4668a0087b","attribute":"order_customer","conflict_value":"acct-9a7a58a9b7ddf3fe","fallback_item":"sku-c14af584e0e0a147","gold_fields":{"delivery_city":"Prague","order_customer":"acct-46595314c96e87e3","preferred_item":"sku-0c9a060b18f3ea82"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0c9a060b18f3ea82":0,"sku-b2210f4668a0087b":2,"sku-c14af584e0e0a147":2}},"note_event_id":"evt-80a4c709689126e2","profile_event_id":"evt-7a8667b1315d9c3c","replicate":5,"request_id":"req-a5c5aac0f2dc8252","source_id":"gx-evaluation-d22f27f654d98340","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-c0deba426a78be71","attribute":"preferred_item","conflict_value":"sku-c0deba426a78be71","fallback_item":"sku-ea4316b075762667","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-99009bda84b8d8fb","preferred_item":"sku-e173a1e46cb2637d"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-c0deba426a78be71":2,"sku-e173a1e46cb2637d":1,"sku-ea4316b075762667":2}},"note_event_id":"evt-36c4ba757fd05a34","profile_event_id":"evt-577eaa7dd24c3dec","replicate":5,"request_id":"req-2e30eaf1ee350df9","source_id":"gx-evaluation-cc2c52e083929e2f","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-25ec441558af6fe6","attribute":"preferred_item","conflict_value":"sku-25ec441558af6fe6","fallback_item":"sku-69302d3fe9690c92","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-34696c81e6e45a12","preferred_item":"sku-3ffe69c1479d0f95"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-25ec441558af6fe6":2,"sku-3ffe69c1479d0f95":0,"sku-69302d3fe9690c92":2}},"note_event_id":"evt-54bef71529d8e53c","profile_event_id":"evt-d3607feb0b070f64","replicate":5,"request_id":"req-67ad582bf561d43d","source_id":"gx-evaluation-9287f19027eff426","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-3a303431fcc3f2b4","attribute":"delivery_city","conflict_value":"Los Angeles","fallback_item":"sku-7a3ebd0e563d5c07","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-557a0adbe11f682c","preferred_item":"sku-43f284c62bd1ce4e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3a303431fcc3f2b4":2,"sku-43f284c62bd1ce4e":1,"sku-7a3ebd0e563d5c07":2}},"note_event_id":"evt-453493adde735a76","profile_event_id":"evt-4eaf0768f0ce1329","replicate":5,"request_id":"req-7141f7f3ac96a967","source_id":"gx-evaluation-c4d7e159d50b0c72","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-fa5487af6afac2c1","attribute":"delivery_city","conflict_value":"Buenos Aires","fallback_item":"sku-7e78e289701f48fc","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-3cddbf5d182af01f","preferred_item":"sku-25157e6dee63407c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-25157e6dee63407c":0,"sku-7e78e289701f48fc":2,"sku-fa5487af6afac2c1":2}},"note_event_id":"evt-fca72c30908622b0","profile_event_id":"evt-d8dbb9d7ba5231e9","replicate":5,"request_id":"req-8498adccde9f9712","source_id":"gx-evaluation-3f4768841457da93","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-21107dd3ee252fde","attribute":"order_customer","conflict_value":"acct-a1ea1e5619e77b67","fallback_item":"sku-15585489d43a3aa5","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-e4944a5bb1d25c44","preferred_item":"sku-cd5f80c3c1b71eb2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-15585489d43a3aa5":2,"sku-21107dd3ee252fde":2,"sku-cd5f80c3c1b71eb2":1}},"note_event_id":"evt-8617c7d13d060c06","profile_event_id":"evt-152f798cf74421ea","replicate":5,"request_id":"req-c84c4eab99721fe2","source_id":"gx-evaluation-3cc03246dae4add2","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-861f6178efb6041a","attribute":"order_customer","conflict_value":"acct-53f0a73d96a4e554","fallback_item":"sku-e5f355df2e0dfa73","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-8e0b9a8050952951","preferred_item":"sku-1bb0bdb18a5be248"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1bb0bdb18a5be248":0,"sku-861f6178efb6041a":2,"sku-e5f355df2e0dfa73":2}},"note_event_id":"evt-7474af50da055b8d","profile_event_id":"evt-3ed58d68c6cf30c1","replicate":5,"request_id":"req-3e7bafa8b61f2d88","source_id":"gx-evaluation-26bf7d3addd21b72","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-7eb6a6950d60b40c","attribute":"preferred_item","conflict_value":"sku-7eb6a6950d60b40c","fallback_item":"sku-a96b441447e8c997","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-42b4d876fdfe3a23","preferred_item":"sku-4b4c9353a5d0d0f2"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4b4c9353a5d0d0f2":1,"sku-7eb6a6950d60b40c":2,"sku-a96b441447e8c997":2}},"note_event_id":"evt-c8a80c5e1d9c7186","profile_event_id":"evt-d192a1fea8a7125b","replicate":5,"request_id":"req-88398be5cc48e4af","source_id":"gx-evaluation-333705eb7d4df4ee","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-9a3d31e8cc4ac10a","attribute":"preferred_item","conflict_value":"sku-9a3d31e8cc4ac10a","fallback_item":"sku-a3d7ac4f2b7c61b5","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-9eb86a4c4f434042","preferred_item":"sku-6c5534da6a4ea984"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6c5534da6a4ea984":0,"sku-9a3d31e8cc4ac10a":2,"sku-a3d7ac4f2b7c61b5":2}},"note_event_id":"evt-cca30230784e1b67","profile_event_id":"evt-b17d749a70d47576","replicate":5,"request_id":"req-e081dcc66d736941","source_id":"gx-evaluation-72b6cdf6489d7356","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-1f701896708198a8","attribute":"delivery_city","conflict_value":"Cairo","fallback_item":"sku-f7a91e63ac22f7ad","gold_fields":{"delivery_city":"New York","order_customer":"acct-7bdf29dd113fff3a","preferred_item":"sku-34e4c2127904eac4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1f701896708198a8":2,"sku-34e4c2127904eac4":1,"sku-f7a91e63ac22f7ad":2}},"note_event_id":"evt-61ccb07f96819b34","profile_event_id":"evt-f09d7e982c6467b1","replicate":6,"request_id":"req-fcf95faee3f89848","source_id":"gx-evaluation-7c087de355074f1c","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-80a6bf490aa834d7","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-8f496056972eca46","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-8bdfb10b096a0200","preferred_item":"sku-dce370a8eec23a19"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-80a6bf490aa834d7":2,"sku-8f496056972eca46":2,"sku-dce370a8eec23a19":0}},"note_event_id":"evt-bc8feb389463fc51","profile_event_id":"evt-d4f1bb640b8ad2be","replicate":6,"request_id":"req-cb128f9b2c431185","source_id":"gx-evaluation-1aba0945144fca7c","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-08734ea17b0b4e9a","attribute":"order_customer","conflict_value":"acct-ac4484722be97f63","fallback_item":"sku-ca5785a75b8a3e17","gold_fields":{"delivery_city":"New York","order_customer":"acct-e706cd8f595793f4","preferred_item":"sku-3173423835f001d6"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-08734ea17b0b4e9a":2,"sku-3173423835f001d6":1,"sku-ca5785a75b8a3e17":2}},"note_event_id":"evt-ee7949bbc6316886","profile_event_id":"evt-70aa57526a10a53a","replicate":6,"request_id":"req-bd58dc4402349cdb","source_id":"gx-evaluation-cb527b75a13a8234","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-aba0ccccbed66f28","attribute":"order_customer","conflict_value":"acct-3e57403510ff2dc4","fallback_item":"sku-44e23023dafdf5fc","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-d98fa646fdecbf56","preferred_item":"sku-e24472ff8b3968b0"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-44e23023dafdf5fc":2,"sku-aba0ccccbed66f28":2,"sku-e24472ff8b3968b0":0}},"note_event_id":"evt-010ea94c35625e5d","profile_event_id":"evt-483237cab54b34d4","replicate":6,"request_id":"req-81bc59f7ec9eb74b","source_id":"gx-evaluation-56d007d4121660de","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-fc3f14d97b56a0f4","attribute":"preferred_item","conflict_value":"sku-fc3f14d97b56a0f4","fallback_item":"sku-c9efeaca08e83811","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-f6b4941d2b22e3ff","preferred_item":"sku-847f5a4301883157"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-847f5a4301883157":1,"sku-c9efeaca08e83811":2,"sku-fc3f14d97b56a0f4":2}},"note_event_id":"evt-681e07d104239e34","profile_event_id":"evt-4e43b4907c3cd47d","replicate":6,"request_id":"req-170c5627a9bced7e","source_id":"gx-evaluation-ca922c80a96a16e1","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-7740cc0969d5ca80","attribute":"preferred_item","conflict_value":"sku-7740cc0969d5ca80","fallback_item":"sku-4ddd35dc38ca6d5c","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-589c7b375424502d","preferred_item":"sku-932e99a5ef2acbe4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4ddd35dc38ca6d5c":2,"sku-7740cc0969d5ca80":2,"sku-932e99a5ef2acbe4":0}},"note_event_id":"evt-9020c31a039c3d1f","profile_event_id":"evt-87868b351e25079f","replicate":6,"request_id":"req-b778b63e4e15a604","source_id":"gx-evaluation-f9d1637b80784ca7","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-cb72c3d77a4ab7a6","attribute":"delivery_city","conflict_value":"Cairo","fallback_item":"sku-8a327bbff33ec0ac","gold_fields":{"delivery_city":"Prague","order_customer":"acct-7217b02b0b8b9a25","preferred_item":"sku-c205a94387cc7c2b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-8a327bbff33ec0ac":2,"sku-c205a94387cc7c2b":1,"sku-cb72c3d77a4ab7a6":2}},"note_event_id":"evt-ff6b081c12c8156e","profile_event_id":"evt-ed7bfafcbe620274","replicate":6,"request_id":"req-8516e7c01492b951","source_id":"gx-evaluation-9945bf238ed2d707","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-6a129e9154cfd339","attribute":"delivery_city","conflict_value":"Lisbon","fallback_item":"sku-97ca15cdc511be27","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-091f4b8064d4e507","preferred_item":"sku-91c1065156840e4b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6a129e9154cfd339":2,"sku-91c1065156840e4b":0,"sku-97ca15cdc511be27":2}},"note_event_id":"evt-13eeb38e3a34e060","profile_event_id":"evt-b608ef7c307df300","replicate":6,"request_id":"req-03db02a5555d9f3f","source_id":"gx-evaluation-835bc75512b3c229","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-4787a71111a6f262","attribute":"order_customer","conflict_value":"acct-905db5fe343971c6","fallback_item":"sku-60feddc48103e32d","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-54e58fc3bc01f85e","preferred_item":"sku-82a7cdfb2c32b436"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4787a71111a6f262":2,"sku-60feddc48103e32d":2,"sku-82a7cdfb2c32b436":1}},"note_event_id":"evt-9b52ef9310f9ba00","profile_event_id":"evt-d7d95ab1552f4325","replicate":6,"request_id":"req-e42f109b7cacb97c","source_id":"gx-evaluation-15d135b3bf234bae","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-facff09e75059e31","attribute":"order_customer","conflict_value":"acct-fe3140143d9c97db","fallback_item":"sku-08b0251fe7c32743","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-f0ef58c7119f3b13","preferred_item":"sku-be92fdc39d244352"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-08b0251fe7c32743":2,"sku-be92fdc39d244352":0,"sku-facff09e75059e31":2}},"note_event_id":"evt-fecbc40e83f79a4e","profile_event_id":"evt-6742e61dac3531ab","replicate":6,"request_id":"req-81dbd49670f63df3","source_id":"gx-evaluation-d85b87bc2c336a4e","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-95e741d602a1c4a5","attribute":"preferred_item","conflict_value":"sku-95e741d602a1c4a5","fallback_item":"sku-51dd7de3c5dd27c8","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-07689f0e85594448","preferred_item":"sku-0018479fecb4223c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0018479fecb4223c":1,"sku-51dd7de3c5dd27c8":2,"sku-95e741d602a1c4a5":2}},"note_event_id":"evt-336709a77d2e6177","profile_event_id":"evt-05f02044839e2b83","replicate":6,"request_id":"req-b032541ddf969428","source_id":"gx-evaluation-a6bf33ff25afc326","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-90d2d9d852dc2db3","attribute":"preferred_item","conflict_value":"sku-90d2d9d852dc2db3","fallback_item":"sku-4f50eda3bf4d6a14","gold_fields":{"delivery_city":"Prague","order_customer":"acct-e0cb7d3de6bf4870","preferred_item":"sku-90590bf357d26ccb"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4f50eda3bf4d6a14":2,"sku-90590bf357d26ccb":0,"sku-90d2d9d852dc2db3":2}},"note_event_id":"evt-096ccbea0b2f18f1","profile_event_id":"evt-2b9b179123f8de6b","replicate":6,"request_id":"req-3d385aa5ee76f542","source_id":"gx-evaluation-34cdf65909358a11","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-dc5849bc9eb3c03a","attribute":"delivery_city","conflict_value":"Osaka","fallback_item":"sku-96efe7cffee829d2","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-f9ef930626c2607b","preferred_item":"sku-488858981a659a34"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-488858981a659a34":1,"sku-96efe7cffee829d2":2,"sku-dc5849bc9eb3c03a":2}},"note_event_id":"evt-69fadd11bfb5ffb2","profile_event_id":"evt-388a7adbfaba4f73","replicate":6,"request_id":"req-6ed4a036934f8fd2","source_id":"gx-evaluation-8b9f11db02338a9d","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-17b4ac5a431d8693","attribute":"delivery_city","conflict_value":"Buenos Aires","fallback_item":"sku-3cae732f2c825f24","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-18aea90ee7aafda0","preferred_item":"sku-5bdce1b4cdce79e7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-17b4ac5a431d8693":2,"sku-3cae732f2c825f24":2,"sku-5bdce1b4cdce79e7":0}},"note_event_id":"evt-6fb198a3f4004399","profile_event_id":"evt-7d556e495750356f","replicate":6,"request_id":"req-004aaad34d1c72f6","source_id":"gx-evaluation-30238c1513fd58b4","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-8e3c88ac38b7a0a1","attribute":"order_customer","conflict_value":"acct-3335aa9c01f2fdd9","fallback_item":"sku-d7a7bfc4a0413e1c","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-76f055b92b9c7198","preferred_item":"sku-08bb29f4d7e9b6c4"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-08bb29f4d7e9b6c4":1,"sku-8e3c88ac38b7a0a1":2,"sku-d7a7bfc4a0413e1c":2}},"note_event_id":"evt-bba4ed9782fe3dcc","profile_event_id":"evt-cf075e2482261d63","replicate":6,"request_id":"req-f3a490fb6f082fe6","source_id":"gx-evaluation-63872ee609ef023e","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-5b7c37f6608d5319","attribute":"order_customer","conflict_value":"acct-8cf5835981645daa","fallback_item":"sku-8068bd135be371cf","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-6993edfa33bc6dc9","preferred_item":"sku-3824baba847320c6"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3824baba847320c6":0,"sku-5b7c37f6608d5319":2,"sku-8068bd135be371cf":2}},"note_event_id":"evt-363801d66e3b84f8","profile_event_id":"evt-9481a1bc50ef5fb6","replicate":6,"request_id":"req-7d7f482415173231","source_id":"gx-evaluation-ae0617e371ffb65f","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-b7701cf42b8efc3b","attribute":"preferred_item","conflict_value":"sku-b7701cf42b8efc3b","fallback_item":"sku-090b4a261e009e0c","gold_fields":{"delivery_city":"New York","order_customer":"acct-d7f9f7a5787ad41b","preferred_item":"sku-193b177bd6da019b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-090b4a261e009e0c":2,"sku-193b177bd6da019b":1,"sku-b7701cf42b8efc3b":2}},"note_event_id":"evt-fc0ea689f534c599","profile_event_id":"evt-5b86b9301b78faa8","replicate":6,"request_id":"req-2cfd49244241b276","source_id":"gx-evaluation-8d189eac315996df","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-fb0f03904f811bd7","attribute":"preferred_item","conflict_value":"sku-fb0f03904f811bd7","fallback_item":"sku-8fe651fd07c8f4b6","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-fa01375920732dd7","preferred_item":"sku-248b8b52c7b12773"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-248b8b52c7b12773":0,"sku-8fe651fd07c8f4b6":2,"sku-fb0f03904f811bd7":2}},"note_event_id":"evt-42e7acc8fcd46c4c","profile_event_id":"evt-5c52f254133d5114","replicate":6,"request_id":"req-2d25b37c21621336","source_id":"gx-evaluation-603223f767aac6b7","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-a3fa5b35206d94b6","attribute":"delivery_city","conflict_value":"Warsaw","fallback_item":"sku-abbdd2d31ccb1d32","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-e928e64280cf194e","preferred_item":"sku-6de32fa8960e5706"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6de32fa8960e5706":1,"sku-a3fa5b35206d94b6":2,"sku-abbdd2d31ccb1d32":2}},"note_event_id":"evt-3f032f989f753a58","profile_event_id":"evt-e60342c048860a97","replicate":6,"request_id":"req-fe1ae8f72890ba88","source_id":"gx-evaluation-622e14883d8cc180","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-59c5213c8cdb19ee","attribute":"delivery_city","conflict_value":"Mumbai","fallback_item":"sku-dbd52bf2e9f0ac4f","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-22e407399015fd01","preferred_item":"sku-2fb402bbac40db44"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-2fb402bbac40db44":0,"sku-59c5213c8cdb19ee":2,"sku-dbd52bf2e9f0ac4f":2}},"note_event_id":"evt-b14942b4ebed487b","profile_event_id":"evt-608ce5fd323e6571","replicate":6,"request_id":"req-822becafb6f3479a","source_id":"gx-evaluation-99c90b7804e27385","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-55ac49b76cf22909","attribute":"order_customer","conflict_value":"acct-fa693fdf70a35031","fallback_item":"sku-9fb417de92791af5","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-e24551cc59c6b208","preferred_item":"sku-f0c6669b4620bc9e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-55ac49b76cf22909":2,"sku-9fb417de92791af5":2,"sku-f0c6669b4620bc9e":1}},"note_event_id":"evt-4ca2bf198f934418","profile_event_id":"evt-a34eb8cea0a54a99","replicate":6,"request_id":"req-277a6edb1f3de6b9","source_id":"gx-evaluation-c7c6f8d4d756d91d","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-3b47d2fc5275341f","attribute":"order_customer","conflict_value":"acct-90d0146c9753df51","fallback_item":"sku-0c8f588f28ef8012","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-a7d1450cb70d474e","preferred_item":"sku-2839f10db39e4737"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0c8f588f28ef8012":2,"sku-2839f10db39e4737":0,"sku-3b47d2fc5275341f":2}},"note_event_id":"evt-0f584057fcad3305","profile_event_id":"evt-59fda1ee27793366","replicate":6,"request_id":"req-0054b81f99c7a4c0","source_id":"gx-evaluation-45abf4571dbc0ec7","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-f969ed550b12bd02","attribute":"preferred_item","conflict_value":"sku-f969ed550b12bd02","fallback_item":"sku-29359f391e5150c7","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-b4c650e5f768d828","preferred_item":"sku-f9572f6fe576f538"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-29359f391e5150c7":2,"sku-f9572f6fe576f538":1,"sku-f969ed550b12bd02":2}},"note_event_id":"evt-c14b4f5ba9beb650","profile_event_id":"evt-4b370bbfa5db0d1c","replicate":6,"request_id":"req-9cc5360d92918467","source_id":"gx-evaluation-010ef6a2e8cdc5c1","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-7082792d9ce802a2","attribute":"preferred_item","conflict_value":"sku-7082792d9ce802a2","fallback_item":"sku-4fab774f971d345b","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-098e0cc2244fdc70","preferred_item":"sku-3b6f18e183c35f04"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3b6f18e183c35f04":0,"sku-4fab774f971d345b":2,"sku-7082792d9ce802a2":2}},"note_event_id":"evt-926130ce9c4abbb4","profile_event_id":"evt-e5260e584220396f","replicate":6,"request_id":"req-d23d2ab9fcc4e172","source_id":"gx-evaluation-e4b32fb286d7fbc4","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-4215058d09ff240d","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-b8254e54ab03347a","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-75f645971c580810","preferred_item":"sku-826e14d6d8fc8648"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4215058d09ff240d":2,"sku-826e14d6d8fc8648":1,"sku-b8254e54ab03347a":2}},"note_event_id":"evt-2306b93cf7a3c6f8","profile_event_id":"evt-3582de70b94421fa","replicate":7,"request_id":"req-1d9f26db231cf6ab","source_id":"gx-evaluation-4aac929ac07398b0","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-7e50ea3e399ce0de","attribute":"delivery_city","conflict_value":"New York","fallback_item":"sku-9a98667a44030d68","gold_fields":{"delivery_city":"Warsaw","order_customer":"acct-73f360bc57d82197","preferred_item":"sku-6bb9a2e2cf31b4a8"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6bb9a2e2cf31b4a8":0,"sku-7e50ea3e399ce0de":2,"sku-9a98667a44030d68":2}},"note_event_id":"evt-2ee7a02dfaa6af19","profile_event_id":"evt-412ac91debff47f6","replicate":7,"request_id":"req-d26f2f3e7ec9c9d7","source_id":"gx-evaluation-1ed1f39b60bf705c","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-c441f4936646be27","attribute":"order_customer","conflict_value":"acct-64566b262059c29d","fallback_item":"sku-c1014886c797cd20","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-947e6134abec28bc","preferred_item":"sku-4d547a8676836c82"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4d547a8676836c82":1,"sku-c1014886c797cd20":2,"sku-c441f4936646be27":2}},"note_event_id":"evt-21ec45e57c65d4ef","profile_event_id":"evt-373029e1c7598da3","replicate":7,"request_id":"req-2cd5b5ab1867ae20","source_id":"gx-evaluation-4fc4500bec4b9008","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-bd8a156f1c721d20","attribute":"order_customer","conflict_value":"acct-cf931d9bccfe917f","fallback_item":"sku-5cdde5288794d523","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-bf3c26c7efb4ef07","preferred_item":"sku-15ea92a9c327d0fe"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-15ea92a9c327d0fe":0,"sku-5cdde5288794d523":2,"sku-bd8a156f1c721d20":2}},"note_event_id":"evt-a9b989ef2b24be53","profile_event_id":"evt-de3318a8bfca9e15","replicate":7,"request_id":"req-dd8ef59a9ab9c6b3","source_id":"gx-evaluation-67029eccb4badf7f","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-8de82a0c8c305f45","attribute":"preferred_item","conflict_value":"sku-8de82a0c8c305f45","fallback_item":"sku-cf1aabcba044d133","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-1c136a551494ca06","preferred_item":"sku-f5b72b4a567abbe9"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-8de82a0c8c305f45":2,"sku-cf1aabcba044d133":2,"sku-f5b72b4a567abbe9":1}},"note_event_id":"evt-86e8c1d048307c07","profile_event_id":"evt-b05f8542526c48c4","replicate":7,"request_id":"req-712c8ae616407f55","source_id":"gx-evaluation-9471877f46737148","stage":"evaluation","stock_available":true,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-896d29c6e7d1011a","attribute":"preferred_item","conflict_value":"sku-896d29c6e7d1011a","fallback_item":"sku-6b9f14fb35d50c6d","gold_fields":{"delivery_city":"Los Angeles","order_customer":"acct-0f2f0bee0758695f","preferred_item":"sku-f1f59cc99e9fd77a"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6b9f14fb35d50c6d":2,"sku-896d29c6e7d1011a":2,"sku-f1f59cc99e9fd77a":0}},"note_event_id":"evt-e81d37898499bc0d","profile_event_id":"evt-f78f62951a379dc6","replicate":7,"request_id":"req-157750ca17d4159d","source_id":"gx-evaluation-fbb7cc55e0dba333","stage":"evaluation","stock_available":false,"template_id":"gx_dispatch","template_text":"Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."},{"alternative_item":"sku-cc72fa2e12de7a03","attribute":"delivery_city","conflict_value":"Prague","fallback_item":"sku-91f9a46494612dd1","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-b456cbd84152c588","preferred_item":"sku-3ee6c76d697111c1"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3ee6c76d697111c1":1,"sku-91f9a46494612dd1":2,"sku-cc72fa2e12de7a03":2}},"note_event_id":"evt-1cf46f8d9c2aeff9","profile_event_id":"evt-7db42c46abf47484","replicate":7,"request_id":"req-3848759bda2c72ed","source_id":"gx-evaluation-4b1af737832f7d49","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-4f72d1a59c629e85","attribute":"delivery_city","conflict_value":"Lisbon","fallback_item":"sku-bd2de11d3437de82","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-ebf092e9c1943b5d","preferred_item":"sku-e9828e8958b86b7b"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-4f72d1a59c629e85":2,"sku-bd2de11d3437de82":2,"sku-e9828e8958b86b7b":0}},"note_event_id":"evt-0e402fe36852f76d","profile_event_id":"evt-066eabd36c0b8721","replicate":7,"request_id":"req-47d38dc14afdfc84","source_id":"gx-evaluation-ae1fe8bf1877c21f","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-fcfc402251e30eaa","attribute":"order_customer","conflict_value":"acct-b661fc62c361c723","fallback_item":"sku-def88e5e76cbbdfb","gold_fields":{"delivery_city":"Prague","order_customer":"acct-3827adbfbcd07782","preferred_item":"sku-bb5721bf60298414"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-bb5721bf60298414":1,"sku-def88e5e76cbbdfb":2,"sku-fcfc402251e30eaa":2}},"note_event_id":"evt-80e7c4561647d10b","profile_event_id":"evt-b52b330fb59fdbde","replicate":7,"request_id":"req-c8d4dd791313b17f","source_id":"gx-evaluation-1345892ce7337bfe","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-f7542b27fba37a90","attribute":"order_customer","conflict_value":"acct-5688e9539ba5c377","fallback_item":"sku-0b86bdedb7f7bc5b","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-4efa620b76f61c86","preferred_item":"sku-53b22eb35b1f88ba"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0b86bdedb7f7bc5b":2,"sku-53b22eb35b1f88ba":0,"sku-f7542b27fba37a90":2}},"note_event_id":"evt-82b8d1a1cc69c427","profile_event_id":"evt-10f222961a4599f7","replicate":7,"request_id":"req-23c148e5f5a26697","source_id":"gx-evaluation-62efbde72d968fdd","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-c8b4895dc8e67da2","attribute":"preferred_item","conflict_value":"sku-c8b4895dc8e67da2","fallback_item":"sku-320285a08a063fa7","gold_fields":{"delivery_city":"Osaka","order_customer":"acct-af11d7f4ba8b6917","preferred_item":"sku-7360af41723efddf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-320285a08a063fa7":2,"sku-7360af41723efddf":1,"sku-c8b4895dc8e67da2":2}},"note_event_id":"evt-8fd63eab07dc2bda","profile_event_id":"evt-520ba3978b4b6c80","replicate":7,"request_id":"req-235b30f89ec2a227","source_id":"gx-evaluation-33dd5b6d6ddff15d","stage":"evaluation","stock_available":true,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-061f4e7abd7d0ad3","attribute":"preferred_item","conflict_value":"sku-061f4e7abd7d0ad3","fallback_item":"sku-6d5407f546d72909","gold_fields":{"delivery_city":"Nairobi","order_customer":"acct-a454c8cee2c25ac7","preferred_item":"sku-e1ffdffff14b6155"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-061f4e7abd7d0ad3":2,"sku-6d5407f546d72909":2,"sku-e1ffdffff14b6155":0}},"note_event_id":"evt-ef187b44507e73b9","profile_event_id":"evt-e458ffa2a2d2b1de","replicate":7,"request_id":"req-05ddefe2d875b74c","source_id":"gx-evaluation-dd758bc91a98129e","stage":"evaluation","stock_available":false,"template_id":"gx_shipment","template_text":"For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."},{"alternative_item":"sku-11d8c616dff28366","attribute":"delivery_city","conflict_value":"Nairobi","fallback_item":"sku-6eaa65ccc3e7fbd0","gold_fields":{"delivery_city":"New York","order_customer":"acct-a5e3d7a9d9e0a02c","preferred_item":"sku-4a9fddbc823b6da1"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-11d8c616dff28366":2,"sku-4a9fddbc823b6da1":1,"sku-6eaa65ccc3e7fbd0":2}},"note_event_id":"evt-db4423969f2fb64f","profile_event_id":"evt-002fda86b3419b3b","replicate":7,"request_id":"req-6c954b4a58c15f2f","source_id":"gx-evaluation-100a9b2c63b0e095","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-55e8ce80a1c09cef","attribute":"delivery_city","conflict_value":"Osaka","fallback_item":"sku-0134159e0dbfa596","gold_fields":{"delivery_city":"Prague","order_customer":"acct-e3d729bd2801fdb8","preferred_item":"sku-108a33ada623f29e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0134159e0dbfa596":2,"sku-108a33ada623f29e":0,"sku-55e8ce80a1c09cef":2}},"note_event_id":"evt-ab59a87631c4aa9d","profile_event_id":"evt-bc52147972ed614f","replicate":7,"request_id":"req-6563806ea85d87d9","source_id":"gx-evaluation-fe841d9578c4ee73","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-2435ea4b4ed81cf8","attribute":"order_customer","conflict_value":"acct-151515589cd03991","fallback_item":"sku-03dec38988a19fc5","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-33b3e6d145527932","preferred_item":"sku-8a06c5b81d987b12"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-03dec38988a19fc5":2,"sku-2435ea4b4ed81cf8":2,"sku-8a06c5b81d987b12":1}},"note_event_id":"evt-79066349a383ae49","profile_event_id":"evt-f36cbf26d0c82c1a","replicate":7,"request_id":"req-0c8697e19e4bb1d8","source_id":"gx-evaluation-b5653c146d3ed63f","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-655b3f23d011b24a","attribute":"order_customer","conflict_value":"acct-f758a78d4e90a86c","fallback_item":"sku-63c474212e4b19c7","gold_fields":{"delivery_city":"New York","order_customer":"acct-0110d88b754ad364","preferred_item":"sku-e15070d14c524a2f"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-63c474212e4b19c7":2,"sku-655b3f23d011b24a":2,"sku-e15070d14c524a2f":0}},"note_event_id":"evt-d3889bd7a3f05389","profile_event_id":"evt-57c16a3cc847ca01","replicate":7,"request_id":"req-21c6dbf687c5926f","source_id":"gx-evaluation-2ad948dcf7cb7329","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-8e6e83a0ac51bc3a","attribute":"preferred_item","conflict_value":"sku-8e6e83a0ac51bc3a","fallback_item":"sku-6a174bc0600749d7","gold_fields":{"delivery_city":"Helsinki","order_customer":"acct-1baaac38ec84feac","preferred_item":"sku-78ccf9a22b18502e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6a174bc0600749d7":2,"sku-78ccf9a22b18502e":1,"sku-8e6e83a0ac51bc3a":2}},"note_event_id":"evt-caa10f5b4276f8a4","profile_event_id":"evt-491d0808fd735b04","replicate":7,"request_id":"req-df5cee873fdce985","source_id":"gx-evaluation-ef10948a01290ec4","stage":"evaluation","stock_available":true,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-9a7e3513e46874b0","attribute":"preferred_item","conflict_value":"sku-9a7e3513e46874b0","fallback_item":"sku-f711dd933667332b","gold_fields":{"delivery_city":"Mumbai","order_customer":"acct-b9f8c05c5681a126","preferred_item":"sku-9f673ce18cc324e3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-9a7e3513e46874b0":2,"sku-9f673ce18cc324e3":0,"sku-f711dd933667332b":2}},"note_event_id":"evt-291888cf2ea11355","profile_event_id":"evt-5cb29b03b77d6f1a","replicate":7,"request_id":"req-08a78707ff733b5c","source_id":"gx-evaluation-415108a11f8ebe29","stage":"evaluation","stock_available":false,"template_id":"gx_inventory","template_text":"Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."},{"alternative_item":"sku-7b6e07796df36118","attribute":"delivery_city","conflict_value":"Lisbon","fallback_item":"sku-faa99f1eff33840b","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-caae1283a63fbf54","preferred_item":"sku-30f35feeb2111464"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-30f35feeb2111464":1,"sku-7b6e07796df36118":2,"sku-faa99f1eff33840b":2}},"note_event_id":"evt-c9f0d478e699567a","profile_event_id":"evt-b70d515b2551e81f","replicate":7,"request_id":"req-0f71d91145459961","source_id":"gx-evaluation-58094241871fe239","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-fabe2310d6e752a7","attribute":"delivery_city","conflict_value":"Mumbai","fallback_item":"sku-afc225dd4e87c46f","gold_fields":{"delivery_city":"Buenos Aires","order_customer":"acct-7bc1432c1a1ecf10","preferred_item":"sku-3531a52e48633209"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-3531a52e48633209":0,"sku-afc225dd4e87c46f":2,"sku-fabe2310d6e752a7":2}},"note_event_id":"evt-e9bd38ac21cbd66c","profile_event_id":"evt-69d6478cf89e7349","replicate":7,"request_id":"req-e7cc0f4f847e93cc","source_id":"gx-evaluation-7e56695b73899705","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-169ca12a8549f33e","attribute":"order_customer","conflict_value":"acct-8901e3f067701d5f","fallback_item":"sku-7ec44bb5db772777","gold_fields":{"delivery_city":"Cape Town","order_customer":"acct-e6c979024aec7547","preferred_item":"sku-ef11b283d023c27c"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-169ca12a8549f33e":2,"sku-7ec44bb5db772777":2,"sku-ef11b283d023c27c":1}},"note_event_id":"evt-5650a706553e543a","profile_event_id":"evt-a4e687d946d5a55a","replicate":7,"request_id":"req-43cda17457e2e849","source_id":"gx-evaluation-0f30df4627029df1","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-13440bb74d2ed7b1","attribute":"order_customer","conflict_value":"acct-9f4121fda71221ba","fallback_item":"sku-4bcaa4f303ea4757","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-625e12cac3f71545","preferred_item":"sku-bc8373499a747e8e"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-13440bb74d2ed7b1":2,"sku-4bcaa4f303ea4757":2,"sku-bc8373499a747e8e":0}},"note_event_id":"evt-c3fb6b761dfc8c58","profile_event_id":"evt-8e3170f75defb43b","replicate":7,"request_id":"req-e0c5a365ae7b713e","source_id":"gx-evaluation-4c4efd93911a487d","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-cb3c7c670ea5d477","attribute":"preferred_item","conflict_value":"sku-cb3c7c670ea5d477","fallback_item":"sku-a69ebdc6cb61ff39","gold_fields":{"delivery_city":"Lisbon","order_customer":"acct-b3a88e92dccf1173","preferred_item":"sku-6fe45d48707fad31"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-6fe45d48707fad31":1,"sku-a69ebdc6cb61ff39":2,"sku-cb3c7c670ea5d477":2}},"note_event_id":"evt-c14088c35ef40d4e","profile_event_id":"evt-7c2178d534bc7dd9","replicate":7,"request_id":"req-b87a0c4888513b6f","source_id":"gx-evaluation-a1b1d70029bf6069","stage":"evaluation","stock_available":true,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."},{"alternative_item":"sku-d34c96f97c5549ac","attribute":"preferred_item","conflict_value":"sku-d34c96f97c5549ac","fallback_item":"sku-e663b9b8bd8a6621","gold_fields":{"delivery_city":"Cairo","order_customer":"acct-086e20f959d9e573","preferred_item":"sku-513e2a3ccf95adb3"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-513e2a3ccf95adb3":0,"sku-d34c96f97c5549ac":2,"sku-e663b9b8bd8a6621":2}},"note_event_id":"evt-1267abe9fef88f29","profile_event_id":"evt-01ac1b21c7208eed","replicate":7,"request_id":"req-e484f94781c1054b","source_id":"gx-evaluation-72e54ae1a2f5c9fb","stage":"evaluation","stock_available":false,"template_id":"gx_submission","template_text":"Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."}],"smoke":[{"alternative_item":"sku-a54c8bdeac84ab56","attribute":"delivery_city","conflict_value":"La Paz","fallback_item":"sku-8914119f00b7122d","gold_fields":{"delivery_city":"Suva","order_customer":"acct-073825c57eb4ba51","preferred_item":"sku-c050bb9072c092cf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-8914119f00b7122d":2,"sku-a54c8bdeac84ab56":2,"sku-c050bb9072c092cf":1}},"note_event_id":"evt-0d806e2145f9ab33","profile_event_id":"evt-0702cddf249a8bea","replicate":0,"request_id":"req-fbe966c97b76019e","source_id":"gx-smoke-8c778ad8906d2131","stage":"smoke","stock_available":true,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."},{"alternative_item":"sku-5c397daea92ad0c3","attribute":"delivery_city","conflict_value":"Suva","fallback_item":"sku-004e264a8eb495f6","gold_fields":{"delivery_city":"La Paz","order_customer":"acct-03364cc64ff12b02","preferred_item":"sku-d4b2cf569cd6fdf7"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-004e264a8eb495f6":2,"sku-5c397daea92ad0c3":2,"sku-d4b2cf569cd6fdf7":0}},"note_event_id":"evt-3c6ca8ab38079629","profile_event_id":"evt-3fd2ef1332830c04","replicate":0,"request_id":"req-a62a1cdfcf0c7cfa","source_id":"gx-smoke-f8684df78cec225d","stage":"smoke","stock_available":false,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."},{"alternative_item":"sku-635f62857dc3ce6c","attribute":"order_customer","conflict_value":"acct-b7467d2c77f38a05","fallback_item":"sku-14dd6127386178ca","gold_fields":{"delivery_city":"La Paz","order_customer":"acct-6c71b168daa0432c","preferred_item":"sku-dd8840b037d184bf"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-14dd6127386178ca":2,"sku-635f62857dc3ce6c":2,"sku-dd8840b037d184bf":1}},"note_event_id":"evt-72adaa3c115a836a","profile_event_id":"evt-4298e4c3fca36daf","replicate":0,"request_id":"req-dda44ff1b8d8d1a1","source_id":"gx-smoke-5c3f7543d23ee7cf","stage":"smoke","stock_available":true,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."},{"alternative_item":"sku-4317888a103c894f","attribute":"order_customer","conflict_value":"acct-2b9ddf4d9cdb9c13","fallback_item":"sku-5a1104dcfc65168e","gold_fields":{"delivery_city":"Reykjavik","order_customer":"acct-5db6c7bd8fc93bdc","preferred_item":"sku-1cd9877d1dae3d29"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-1cd9877d1dae3d29":0,"sku-4317888a103c894f":2,"sku-5a1104dcfc65168e":2}},"note_event_id":"evt-5b50eef15f182d2a","profile_event_id":"evt-7201607d6b4e7bfb","replicate":0,"request_id":"req-8895517efad5c663","source_id":"gx-smoke-c69c6da09292cb3b","stage":"smoke","stock_available":false,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."},{"alternative_item":"sku-f6108b93812ee207","attribute":"preferred_item","conflict_value":"sku-f6108b93812ee207","fallback_item":"sku-9310d61637b488d2","gold_fields":{"delivery_city":"Reykjavik","order_customer":"acct-27a57115744b9013","preferred_item":"sku-7846cbd86ed7e4ec"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-7846cbd86ed7e4ec":1,"sku-9310d61637b488d2":2,"sku-f6108b93812ee207":2}},"note_event_id":"evt-2551dd9c4c7661b1","profile_event_id":"evt-80c477f84deedc91","replicate":0,"request_id":"req-2619c55d41fd708e","source_id":"gx-smoke-35bf9f461e516662","stage":"smoke","stock_available":true,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."},{"alternative_item":"sku-b908090343efeb48","attribute":"preferred_item","conflict_value":"sku-b908090343efeb48","fallback_item":"sku-0c1a29d1a85f9923","gold_fields":{"delivery_city":"Valletta","order_customer":"acct-2a92c953925bc12f","preferred_item":"sku-96c63a1f07b6a1ea"},"initial_env":{"next_order":1,"next_reservation":1,"orders":[],"reservations":{},"stock":{"sku-0c1a29d1a85f9923":2,"sku-96c63a1f07b6a1ea":0,"sku-b908090343efeb48":2}},"note_event_id":"evt-a159ebe58964479d","profile_event_id":"evt-01f2406b3f816741","replicate":0,"request_id":"req-0d5dbe97d65282c9","source_id":"gx-smoke-0b5131c10ad07efe","stage":"smoke","stock_available":false,"template_id":"gx_smoke","template_text":"Exercise the tool interface for request {request_id}: resolve its field, check inventory, reserve and submit one order using the allowed fallback if necessary."}]}''')
V5_PREPARED_PROTOCOL = json.loads(r'''{"arms":["untreated","authority_first","authority_only"],"attributes":["delivery_city","order_customer","preferred_item"],"baseline_assumption":"verified_profile/retrieved_note source-kind labels are reliable","cities":["Lisbon","Prague","Helsinki","Warsaw","Cairo","Nairobi","Mumbai","Osaka","New York","Los Angeles","Cape Town","Buenos Aires"],"conditions":["aligned","conflict"],"data_scope":"new seed83022 instances; same vocabulary/templates and local order environment; not an external benchmark","data_seed":83022,"dtype":"bfloat16","evaluation_logical_rows":2304,"evaluation_unique_actor_episodes":960,"evaluation_unique_writer_episodes":960,"independent_unit":"source_id","intermediate_metrics":["actor_reservation_attempted","actor_reservation_accepted","actor_attempted_wrong_reservation_item","actor_accepted_wrong_reservation_item","actor_attempted_conflict_reservation_item","actor_accepted_conflict_reservation_item","actor_order_attempted","actor_order_accepted","actor_first_order_wrong_attribute","actor_any_order_wrong_attribute","actor_first_order_conflict_value","actor_any_order_conflict_value","actor_first_order_missing_attribute","actor_any_order_missing_attribute","actor_first_order_missing_city","actor_any_order_missing_city","actor_first_order_missing_required_fields","actor_any_order_missing_required_fields","actor_first_order_schema_complete","actor_any_order_schema_complete"],"manifest_hashes":{"evaluation":"2b094c724817fa5fd3248a4e899f4da3cb982f1bf63090e75fcffa5bea8ec672","smoke":"a6008b4c5b5fdbbff634863abe1c960b0d13d74b9b7aae5a6b14db1994efa59d"},"metric_applicability":{"actor_accepted_conflict_reservation_item":["preferred_item"],"actor_attempted_conflict_reservation_item":["preferred_item"]},"model":"meta-llama/Llama-3.1-8B","new_gpu_results_included":false,"paired_intermediate_metrics":["actor_attempted_wrong_reservation_item","actor_attempted_conflict_reservation_item","actor_accepted_wrong_reservation_item","actor_accepted_conflict_reservation_item","actor_first_order_wrong_attribute","actor_first_order_conflict_value","actor_any_order_wrong_attribute","actor_any_order_conflict_value","actor_any_order_missing_city"],"positions":["profile_first","note_first"],"primary_contrast":"untreated/note_first: conflict minus aligned accepted_wrong_attribute","prompt_change":"all ten legal next-call prefixes (check/reserve/create) and empty-history rule; no automatic output repair","readiness_rules":{"controls":[["untreated","aligned","profile_first"],["untreated","aligned","note_first"],["authority_only","aligned","profile_first"]],"maximum_protocol_failures_per_control":0,"minimum_task_successes_per_control":5,"required_first_complete_city_per_control":6,"required_smoke_sources":6,"selection_policy":"Fixed engineering guardrails; no conflict-error-rate criterion; never select evaluation cases by outcomes","version":"v5_control_usability_v1"},"reuse_rule":"checked live weights and same-v5.1 checkpoints only; no v4/v5 generations reused","revision":"d04e592bb4f6aa9cfee91e2e20afa771667e1d4b","source_counts":{"evaluation":192,"smoke":6},"templates":[["gx_dispatch","Handle request {request_id}: create one order using its resolved fields. Use its preferred item if stocked; otherwise use the listed fallback."],["gx_shipment","For request {request_id}, arrange exactly one shipment. Choose the requested preferred SKU when available, else the permitted fallback."],["gx_inventory","Execute request {request_id} through the inventory and order tools. Preserve its resolved recipient and destination, and prefer its requested SKU over the fallback whenever stock permits."],["gx_submission","Place a single order for request {request_id}. Derive the missing field from the supplied source records, use the given fixed fields, and substitute the fallback only when the preferred SKU has no stock."]],"version":"conflict_v5p1_protocol_actions_full_v1"}''')
V5_PREPARED_HASHES = {'smoke': 'a6008b4c5b5fdbbff634863abe1c960b0d13d74b9b7aae5a6b14db1994efa59d', 'evaluation': '2b094c724817fa5fd3248a4e899f4da3cb982f1bf63090e75fcffa5bea8ec672'}

V5_DATA_CHECKS = []
for v5_stage, v5_cases in V5_PREPARED_DATA.items():
    assert V5_GX_NAMESPACE['gx_build_cases'](stage=v5_stage, seed=V5_CFG['data_seed']) == v5_cases
    assert V5_RUNTIME_NAMESPACE['stable_hash'](v5_cases) == V5_PREPARED_HASHES[v5_stage]
    assert len({c['source_id'] for c in v5_cases}) == len(v5_cases)
    V5_RUNTIME_NAMESPACE['atomic_json'](V5_RUN_ROOT / 'prepared-data' / (v5_stage + '-manifest.json'), v5_cases)
    for v5_case in v5_cases:
        v5_task = V5_GX_NAMESPACE['gx_task_payload'](v5_case)
        assert v5_case['attribute'] not in v5_task['fixed_fields']
        v5_clean = V5_GX_NAMESPACE['gx_memory'](v5_case, 'aligned', 'profile_first', 'untreated')
        v5_conflict = V5_GX_NAMESPACE['gx_memory'](v5_case, 'conflict', 'profile_first', 'untreated')
        v5_clean[1]['value'] = v5_conflict[1]['value']
        assert v5_clean == v5_conflict
        V5_DATA_CHECKS.append({'stage': v5_stage, 'source_id': v5_case['source_id'],
            'attribute': v5_case['attribute'], 'template_id': v5_case['template_id'],
            'stock_available': v5_case['stock_available'], 'city': v5_case['gold_fields']['delivery_city']})
# Constructed software fixtures only: never scientific/model-generation outcomes.
for v5_case in V5_PREPARED_DATA['evaluation'][:6]:
    v5_env = V5_RUNTIME_NAMESPACE['OrderEnvironment'](v5_case['initial_env'])
    v5_events = []
    def v5_check_step(action):
        observation = v5_env.step(action)
        v5_events.append({'action': action, 'observation': observation})
        return observation
    v5_preferred = v5_case['gold_fields']['preferred_item']
    v5_available = v5_check_step({'tool': 'check_stock', 'item': v5_preferred})['available']
    v5_item = v5_preferred if v5_available else v5_case['fallback_item']
    if not v5_available:
        v5_check_step({'tool': 'check_stock', 'item': v5_item})
    v5_reservation = v5_check_step({'tool': 'reserve_item', 'item': v5_item})['reservation']
    v5_check_step({'tool': 'create_order', 'customer': v5_case['gold_fields']['order_customer'],
                   'city': v5_case['gold_fields']['delivery_city'], 'item': v5_item, 'reservation': v5_reservation})
    v5_metrics = V5_GX_NAMESPACE['gx_score_actor'](v5_case, v5_events, v5_env.state['orders'], v5_case['initial_env'])
    assert v5_metrics['strict_task_success'] and not v5_metrics['accepted_wrong_attribute']
    v5_memory = V5_GX_NAMESPACE['gx_memory'](v5_case, 'conflict', 'note_first', 'untreated')
    v5_history = V5_GX_NAMESPACE['gx_history'](v5_case, v5_memory, {'source_id': v5_case['source_id'], 'memory': v5_memory, 'events': v5_events})
    v5_reference = [dict(v5_memory[1], kind='preference')]
    assert V5_GX_NAMESPACE['gx_score_memory'](v5_case, v5_history, v5_reference, True)['target_joint_correct']
V5_RUNTIME_NAMESPACE['atomic_json'](V5_RUN_ROOT / 'prepared-data' / 'protocol.json', V5_PREPARED_PROTOCOL)
pd.DataFrame(V5_DATA_CHECKS).to_csv(V5_RUN_ROOT / 'prepared-data' / 'source-overview.csv', index=False)
display(pd.DataFrame(V5_DATA_CHECKS).groupby(['stage','attribute','template_id','stock_available']).size().rename('n_sources').reset_index())
print(' Data / tool / The scoring software check was passed; no model test answers have been produced. ')
v5_preview = V5_PREPARED_DATA['evaluation'][0]
display({'task': v5_preview['template_text'].format(request_id=v5_preview['request_id']),
         'task_payload': V5_GX_NAMESPACE['gx_task_payload'](v5_preview),
         'conflict_memory': V5_GX_NAMESPACE['gx_memory'](v5_preview, 'conflict', 'note_first', 'untreated')})


## 06 ·  Reuse or Load Fixed Models 
 Autocheck compatibility models in the current session. Load fixed when not found  commit， Used  Hugging Face  weight cache; new sessions do not need old variables. 


In [ ]:
V5_EXISTING_CANDIDATES = []
if V5_REUSE_LOADED_MODEL:
    if isinstance(globals().get('V5_PARENT'), dict) and 'backend' in V5_PARENT:
        V5_EXISTING_CANDIDATES.append({'backend': V5_PARENT['backend'],
            'model_identity': V5_PARENT['MODEL_IDENTITY'], 'label': 'existing_v5_runtime'})
    if isinstance(globals().get('V4_PARENT'), dict) and 'backend' in V4_PARENT:
        V5_EXISTING_CANDIDATES.append({'backend': V4_PARENT['backend'],
            'model_identity': V4_PARENT['MODEL_IDENTITY'], 'label': 'existing_v4_runtime'})
    if 'backend' in globals() and 'MODEL_IDENTITY' in globals():
        V5_EXISTING_CANDIDATES.append({'backend': globals()['backend'],
            'model_identity': globals()['MODEL_IDENTITY'], 'label': 'existing_v3_runtime'})
if V5_LIVE_ONLY:
    v5_expected_identity = {'name': V5_CFG['model_name'], 'revision': V5_CFG['model_revision'],
        'dtype': V5_CFG['dtype'], 'packages': V5_PACKAGE_VERSIONS, 'python': sys.version.split()[0]}
    v5_compatible, v5_rejections = [], []
    for v5_candidate in V5_EXISTING_CANDIDATES:
        try:
            V5_RUNTIME_NAMESPACE['v4_check_backend'](v5_candidate['backend'], V5_CFG,
                v5_candidate['model_identity'], v5_expected_identity)
            v5_compatible.append(v5_candidate)
        except Exception as v5_error:
            v5_rejections.append({'label': v5_candidate['label'], 'reason': str(v5_error)})
    if not v5_compatible:
        raise RuntimeError(' The current model cannot be safely reused, and append cells do not load a second model. Use the complete notebook New runtime。' + str(v5_rejections))
    V5_EXISTING_CANDIDATES = v5_compatible
V5_PARENT = V5_RUNTIME_NAMESPACE['v4_load_runtime'](
    V5_CFG, V5_RUN_ROOT / 'runtime', V5_PACKAGE_VERSIONS,
    V5_CORE_SOURCE, V5_RUNTIME_SOURCE, candidates=V5_EXISTING_CANDIDATES)
V5_BACKEND = V5_PARENT['backend']
display({key: V5_PARENT['loader_reuse'][key] for key in ('mode', 'reused_candidate', 'candidate_attempts', 'device')})
V5_EXPERIMENT = V5_GX_NAMESPACE['GXExperiment'](
    V5_PARENT, V5_GX_SOURCE, root_dir=V5_RUN_ROOT / 'experiment')
assert V5_EXPERIMENT.cases == V5_PREPARED_DATA
V5_STATUS = V5_EXPERIMENT.inspect()


## 07 · GPU  Interface and input length check 
 Reuse the original interface check; tokenizer  The audit checks length without excluding multiword values. Constructed length-check trajectories do not enter experimental statistics. 


In [ ]:
V5_GPU_INTERFACE = V5_RUNTIME_NAMESPACE['runtime_smoke'](V5_BACKEND)
V5_RUNTIME_NAMESPACE['atomic_json'](V5_RUN_ROOT / 'runtime' / 'software-smoke.json', V5_GPU_INTERFACE)
display(V5_GPU_INTERFACE)
V5_TOKEN_AUDIT = V5_EXPERIMENT.audit_token_budget()


## 08 ·  Enforcement requirements prior to formal assessment 
 These fixed requirements check only conflict-free controls. Stop if they fail; never select tasks or passing conditions using conflict error rates. 


In [ ]:
display(V5_GX_NAMESPACE['GX_READINESS_RULES'])
print(' Old v4 Model weights can be reused;old prompt Generate no import v5。')


## 09 ·  Six independents.  smoke  Tasks 
 Check the new task format and full workflow. Do not select formal tasks by whether errors occur or require a minimum attack success rate. Revisions  prompt  New version is needed after / Contents. 


In [ ]:
try:
    V5_SMOKE = V5_EXPERIMENT.run(stage='smoke', max_sources=None)
except RuntimeError:
    from collections import Counter as v5_counter
    v5_completed, _ = V5_EXPERIMENT._collect('smoke')
    v5_terminals, v5_first_tools, v5_tool_errors = v5_counter(), v5_counter(), v5_counter()
    for v5_source_result in v5_completed:
        for v5_episode in v5_source_result['episodes'].values():
            v5_actor_result = v5_episode['actor']
            v5_terminals[v5_actor_result['terminal']] += 1
            if v5_actor_result['events']:
                v5_first_action = v5_actor_result['events'][0].get('action') or {}
                v5_first_tools[v5_first_action.get('tool', '<parse_failure>')] += 1
            for v5_event in v5_actor_result['events']:
                v5_observation = v5_event.get('observation') or {}
                if v5_observation.get('error'):
                    v5_tool_errors[v5_observation['error']] += 1
    print(' Independence Actor Reason for trajectories termination: ', dict(v5_terminals))
    print(' Independence Actor First track tool: ', dict(v5_first_tools))
    print(' Reason for tool rejection (number of calls, not individual samples): ', dict(v5_tool_errors))
    raise


## 10 ·  Formal evaluation and break-off 
 Default for each batch 24 source， Repeat this block until 192/192； Midway Mark  PARTIAL。 Set V5_BATCH_SOURCES=None Runs all the remainings. Every entry is re-validated smoke results; completed source Skip automatically. 


In [ ]:
V5_EVALUATION = V5_EXPERIMENT.run(stage='evaluation', max_sources=V5_BATCH_SOURCES)


## 11 ·  Statistics and Export 

 Inspect task success, strict path success, incorrect orders, no orders, and format / tool error. A simple view of the fall of the wrong order is not sufficient to indicate an improvement. 
 Exporting false bookings, first and any errors in placing the order, city Missing, action coverage; successful booking separate from rejected attempts. 
 Same source Repeating attempts to contribute only one event rate; number of moves retained rows.csv。 The original comparison remains unchanged, and the new action indicators are secondary. 
Writer  Report value, provenance, joint accuracy, and whole-memory accuracy; when out of stock, buying  fallback  Doesnt mean preference becomes  fallback。
 Other  baseline  Differences include processing of information and  Actor  The overall effect of the trajectories change is not attributable separately to  Writer  Internal mechanisms. 

 The main comparison is...  untreated/note_first   conflict−aligned  focus field error rate;other properties / Location /baseline  The comparison is a secondary analysis. 
 By Independence source Give the count. Wilson Intersections, pairings, bootstrap The degradation zone will be marked. 
 Package data, protocol, model identity, reuse provenance, raw per-source trajectories, statistics, and failure cases together. Model weights are excluded. 


In [ ]:
import zipfile
V5_RESULT_REPORTS = {stage: V5_EXPERIMENT.report(stage) for stage in ('smoke', 'evaluation')}
v5_summary_frame = pd.DataFrame(V5_RESULT_REPORTS['evaluation']['summary'])
if not v5_summary_frame.empty:
    v5_display_metrics = {
        'actor_task_success': ' Rightly complete the order. ',
        'actor_accepted_wrong_attribute': ' Wrong order accepted ',
        'actor_accepted_wrong_reservation_item': ' Wrong commodity bookings accepted ',
        'actor_first_order_wrong_attribute': ' Try wrong value for the first entry ',
        'actor_any_order_missing_city': ' Next try to leak. city',
        'writer_target_joint_correct': 'Writer Value and source is correct ',
    }
    v5_view = v5_summary_frame[(v5_summary_frame['stratum'] == 'all')
        & v5_summary_frame['metric'].isin(v5_display_metrics)].copy()
    v5_view[' Indicators '] = v5_view['metric'].map(v5_display_metrics)
    v5_view[' Count / Independent missions '] = v5_view['count'].astype(str) + '/' + v5_view['n_sources'].astype(str)
    print(' All Conditions / complete comparison of sequence; the denominator is independent source， _Other Organiser ')
    display(v5_view.pivot(index=['arm','condition','position'], columns=' Indicators ', values=' Count / Independent missions '))
    v5_sku_metrics = {'actor_attempted_conflict_reservation_item': ' Try booking conflicts SKU',
        'actor_accepted_conflict_reservation_item': ' Conflict booking successfully SKU',
        'actor_first_order_conflict_value': ' Try to use conflict for the first time. SKU',
        'actor_accepted_conflict_value': ' Conflict of final orders SKU'}
    v5_skus = v5_summary_frame[(v5_summary_frame['stratum'] == 'attribute')
        & (v5_summary_frame['stratum_value'] == 'preferred_item')
        & v5_summary_frame['metric'].isin(v5_sku_metrics)].copy()
    v5_skus[' Indicators '] = v5_skus['metric'].map(v5_sku_metrics)
    v5_skus[' Count / The commodity mandate '] = v5_skus['count'].astype(str) + '/' + v5_skus['n_sources'].astype(str)
    print('preferred_item  Subset: Full Evaluation Subset 64； The two orders belong to the same group. source。')
    display(v5_skus.pivot(index=['arm','condition','position'], columns=' Indicators ', values=' Count / The commodity mandate '))
    display(pd.DataFrame([row for row in V5_RESULT_REPORTS['evaluation']['paired_effects']
                          if row['analysis_role'] == 'primary']))
V5_FINAL_ZIP = V5_RUN_ROOT.parent / (V5_RUN_NAME + '_results.zip')
V5_ARCHIVE_FILES = []
for v5_folder in (V5_RUN_ROOT / 'prepared-data', V5_RUN_ROOT / 'runtime'):
    V5_ARCHIVE_FILES.extend(p for p in v5_folder.rglob('*') if p.is_file() and p.suffix in ('.json', '.csv'))
for v5_folder in (V5_EXPERIMENT.root / 'sources', V5_EXPERIMENT.root / 'reports'):
    V5_ARCHIVE_FILES.extend(p for p in v5_folder.rglob('*') if p.is_file() and p.suffix in ('.json', '.csv'))
V5_ARCHIVE_FILES.extend(p for p in V5_EXPERIMENT.root.glob('*.json') if p.is_file())
with zipfile.ZipFile(V5_FINAL_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as v5_archive:
    for v5_path in sorted(set(V5_ARCHIVE_FILES)):
        v5_archive.write(v5_path, v5_path.relative_to(V5_RUN_ROOT).as_posix())
print(' Full result package: ', V5_FINAL_ZIP)
print(' confirmation source Overwrite: ', V5_RESULT_REPORTS['evaluation']['completed_sources'], '/ 192')
from google.colab import files as v5_colab_files
v5_colab_files.download(str(V5_FINAL_ZIP))
